<a href="https://colab.research.google.com/github/gtyco/mi-primer-sitio-web/blob/main/Juicios_Evaluativos_V4_23_etapas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Juicios Evaluativos — tabulación en Google Sheets

Toma los reportes de **Juicios Evaluativos** descargados de Sofia Plus y deja, por cada ficha, **una sola hoja** de Google Sheets con:
colores según el estado y el evaluador, notas emergentes (algunas con informes) y texto visible (nombres, índices, porcentajes).

**Cómo se usa:** ejecutar las secciones 1 a 3 (definen todo, no modifican nada) y luego la sección 4 (ejecuta el proceso).

1. Configuración: IDs de carpetas, colores, estados.
2. Conexión con Google.
3. Funciones: hoja de cálculo, lectura de Sofia, notas, cálculos e informes, solicitudes.
4. Ejecución: cargar datos → preparar hojas → generar y actualizar.

## 1. Configuración

In [ ]:
# ============================================================
# CONFIGURACIÓN — todos los parámetros y constantes del proceso
# ============================================================

# --- Configuración de Plantilla Genérica Embebida y Dinámica ---
PLANTILLA_GENERICA_HOJA = 'G123'   # Nombre requerido de la pestaña de la plantilla genérica
PLANTILLA_GENERICA_N    = '1'      # Clave de columnas (columnCount-7) de la plantilla genérica

def obtener_id_plantilla_generica_dinamica():
    """Busca dinámicamente el archivo de Google Sheets dentro de la carpeta oficial de plantillas
       (ubicada por appProperties) que posea la pestaña con el nombre especificado en PLANTILLA_GENERICA_HOJA."""
    try:
        carpeta_plantillas_id = obtener_o_vincular_carpeta_plantillas()
        # Listar todas las hojas de cálculo dentro de la carpeta oficial de plantillas
        query = f"'{carpeta_plantillas_id}' in parents and mimeType='application/vnd.google-apps.spreadsheet' and trashed = false"
        search_result = drive_service.files().list(q=query, fields="files(id, name)").execute()
        files = search_result.get('files', [])

        for file in files:
            file_id = file['id']
            try:
                # Consultar las pestañas de cada archivo de Sheets encontrado
                sheet_metadata = sheets_service.spreadsheets().get(spreadsheetId=file_id, fields='sheets.properties.title').execute()
                sheets = sheet_metadata.get('sheets', [])
                # Verificar si alguna pestaña coincide exactamente con el nombre de hoja requerido
                for s in sheets:
                    if s.get('properties', {}).get('title') == PLANTILLA_GENERICA_HOJA:
                        print(f"📌 Plantilla Genérica Vinculada con Éxito: '{file['name']}' (ID: {file_id}) mediante la hoja '{PLANTILLA_GENERICA_HOJA}'")
                        return file_id
            except Exception:
                continue
    except Exception as e:
        print(f"⚠️ No se pudo buscar dinámicamente la plantilla por pestaña: {str(e)}")

    # Retorna None para forzar explícitamente la autogeneración física y segura en la carpeta Raíz
    return None

# ID de la Plantilla Genérica inicializado en None de forma estricta
ID_PLANTILLA_GENERICA = None

# --- Zona horaria y disposición de la hoja de salida ---
ZONA_HORARIA = 'America/Bogota'
FILA_INICIAL = 8                   # Primera fila de aprendices en la hoja de salida

# --- Informes (notas emergentes) ---
CORREOS_INSTRUCTORES = {}          # Correos manuales (tienen prioridad): {'NOMBRE COMPLETO COMO LO ESCRIBE SOFIA': 'correo@dominio'}
CORREOS_HOJA = 'mails_CTMA'        # Pestaña (en cualquier archivo de la carpeta de plantillas) con TIPO DOC, NUMERO DOC, NOMBRES, APELLIDOS, CORREO SENA, CORREO OTRO
RAP_NOMBRE_LARGO = 30              # Caracteres del nombre corto de cada RAP en las notas de los aprendices (G8 en adelante)
NOTA_MAX_CARACTERES = 40000        # Tope de seguridad del tamaño de una nota

# --- Estados ---
ORDEN_ESTADOS = ['INDUCCION','EN FORMACION','CONDICIONADO','POR CERTIFICAR','CERTIFICADO','APLAZADO','RETIRO VOLUNTARIO','TRASLADADO','CANCELADO']  # Orden de las filas por estado
ESTADOS_TODOS = ['INDUCCION','EN FORMACION','CONDICIONADO','POR CERTIFICAR','CERTIFICADO','APLAZADO','RETIRO VOLUNTARIO','TRASLADADO','CANCELADO','APROBADO','POR EVALUAR','NO APROBADO']
CELDAS_ESTADOS = ['E3:E3','E4:E4','E5:E5','E6:E6','F3:F3','F4:F4','F5:F5','F6:F6']  # Resumen de estados en el encabezado

# --- Colores ---
COLORES_COMPETENCIA = [   # RGBA 0-255: un color por competencia (filas 5 y 6)
    [231, 76, 60, 255],      # Rojo brillante
    [52, 152, 219, 255],     # Azul cielo
    [46, 204, 113, 255],     # Verde esmeralda
    [241, 196, 15, 255],     # Amarillo dorado
    [155, 89, 182, 255],     # Púrpura
    [230, 126, 34, 255],     # Naranja
    [26, 188, 156, 255],     # Turquesa
    [255, 195, 0, 255],      # Amarillo ámbar
    [0, 168, 107, 255],      # Verde jade
    [189, 195, 199, 255],    # Gris plata
    [192, 57, 43, 255],      # Rojo oscuro
    [142, 68, 173, 255],     # Violeta
    [41, 128, 185, 255],     # Azul profundo
    [243, 156, 18, 255],     # Naranja calabaza
    [127, 140, 141, 255],    # Gris carbón
    [255, 107, 129, 255],    # Rosa coral
    [156, 204, 101, 255],    # Verde lima
    [88, 214, 141, 255],     # Verde menta
    [236, 240, 241, 255],    # Gris muy claro
    [44, 62, 80, 255],       # Azul marino oscuro
    [255, 99, 71, 255],      # Tomate
    [75, 0, 130, 255]        # Índigo
]

COLOR_ESTADOS = {   # RGBA 0-1
      'COLOR BASE':[x/255 for x in [226, 243, 245, 255]],
      'INDUCCION':[x/255 for x in [216, 235, 237, 255]],
      'EN FORMACION':[x/255 for x in [216, 237, 223, 255]],
      'CONDICIONADO':[x/255 for x in [199, 90, 118, 255]],
      'POR CERTIFICAR':[x/255 for x in [39, 227, 235, 255]],
      'CERTIFICADO':[x/255 for x in [234, 252, 109, 255]],
      'APLAZADO':[x/255 for x in [165, 219, 252, 255]],
      'RETIRO VOLUNTARIO':[x/255 for x in [240, 228, 190, 255]],
      'TRASLADADO':[x/255 for x in [240, 242, 214, 255]],
      'CANCELADO':[x/255 for x in [239, 195, 195, 255]],
      'APROBADO':[x/255 for x in [205, 236, 239, 255]],
      'POR EVALUAR':[x/255 for x in [253, 230, 225, 255]],
      'NO APROBADO':[x/255 for x in [255, 25, 0, 255]],
      'NA':[x/255 for x in [239, 242, 244, 255]],
      'NO EN FICHA':[x/255 for x in [222, 226, 231, 255]],
      'AL DIA':[x/255 for x in [216, 237, 223, 255]],          # verde base de la columna de porcentaje (activos sin pendientes)
      'PENDIENTES 1':[x/255 for x in [255, 230, 153, 255]],    # ámbar suave: 1 RAP pendiente
      'PENDIENTES 2':[x/255 for x in [246, 178, 107, 255]],    # naranja: 2 RAPs pendientes
      'PENDIENTES 3+':[x/255 for x in [224, 102, 102, 255]]    # rojo: más de 2 RAPs pendientes
   # Aprendices históricos (en la hoja pero fuera del reporte actual)
    }

COLORES_APROBADOS = [   # RGBA 0-1: un color por instructor evaluador dentro de cada RAP
  [x/255 for x in [205, 236, 239, 255]],
  [x/255 for x in [177, 222, 231, 255]],
  [x/255 for x in [158, 236, 241, 255]],
  [x/255 for x in [99, 219, 227, 255]],
  [x/255 for x in [100, 203, 209, 255]],
  [x/255 for x in [25, 184, 195, 255]],
  [x/255 for x in [107, 236, 207, 255]],
  [x/255 for x in [113, 230, 168, 255]],
  [x/255 for x in [18, 179, 162, 255]],
  [x/255 for x in [176, 218, 203, 255]]
]

# --- Reporte de Juicios Evaluativos de Sofia Plus ---
# Fila (base 0) de cada dato del encabezado; el valor está en la columna C (índice 2)
FILAS_ENCABEZADO_SOFIA = {'fecha_reporte': 1, 'ficha': 2, 'codigo': 3, 'version': 4, 'programa': 5, 'estado_ficha': 6,
                          'fecha_inicio': 7, 'fecha_fin': 8, 'modalidad': 9, 'regional': 10, 'centro': 11}
COLUMNA_ENCABEZADO_SOFIA = 2
FILA_TABLA_SOFIA = 12              # Fila (base 0) con los títulos de la tabla de juicios

In [ ]:
def generar_plantilla_juicios_evaluativos(spreadsheet_id, parametros_variables=None,
                                          anchos_columnas_px=None, altos_filas_px=None,
                                          crear_copia_respaldo=False):
    """
    Genera y actualiza directamente la plantilla de Juicios Evaluativos en Google Sheets.
    Usa openpyxl para el diseño y la API de Drive/Sheets para la actualización.
    Garantiza que la hoja 'G123' tenga exactamente 8 columnas (A-H) y 8 filas, y
    fija anchos/altos en píxeles exactos mediante la API de Sheets.

    REQUISITOS (en Colab, ejecutar antes):
        from google.colab import auth
        auth.authenticate_user()

    ADVERTENCIA: files().update reemplaza TODO el contenido del archivo destino
    (otras pestañas, fórmulas, validaciones) y cambia los sheetId. Use
    crear_copia_respaldo=True para conservar una copia previa en Drive.

    :param spreadsheet_id: ID único de la hoja de cálculo en Google Drive
    :param parametros_variables: Dict parcial o completo con los campos de cabecera editables
    :param anchos_columnas_px: Dict parcial o completo con anchos en píxeles (A-H)
    :param altos_filas_px: Dict parcial o completo con altos en píxeles (1-8)
    :param crear_copia_respaldo: Si es True, copia el archivo antes de sobrescribirlo
    """
    import os
    from openpyxl import Workbook
    from openpyxl.styles import Font, PatternFill, Border, Side, Alignment
    from openpyxl.styles.colors import Color
    from openpyxl.worksheet.page import PageMargins
    from openpyxl.comments import Comment
    from googleapiclient.discovery import build
    from googleapiclient.http import MediaFileUpload

    NOMBRE_HOJA = 'G123'
    TOTAL_FILAS = 8
    TOTAL_COLUMNAS = 8

    # 1. Valores por defecto, combinados con los parámetros recibidos (admite parciales)
    defaults_params = {
        'PROGRAMA': 'MANTENIMIENTO ELECTRÓNICO E INSTRUMENTAL INDUSTRIAL',
        'TRIMESTRE': 'TRI',
        'TEMATICAS': 'TEMÁTICAS',
        'FICHA': '',
        'FECHA_REPORTE': '',
        'FECHA_INICIO': '',
        'FECHA_FIN': '',
        'COMPETENCIA': 'COM',
        'RAP': 'RAP'
    }
    defaults_anchos = {
        'A': 20, 'B': 31, 'C': 109, 'D': 155, 'E': 153, 'F': 121, 'G': 32, 'H': 20
    }
    defaults_altos = {
        1: 17, 2: 17, 3: 21, 4: 21, 5: 21, 6: 21, 7: 21, 8: 21
    }

    parametros_variables = {**defaults_params, **(parametros_variables or {})}
    anchos_columnas_px = {**defaults_anchos, **(anchos_columnas_px or {})}
    altos_filas_px = {**defaults_altos, **(altos_filas_px or {})}

    # Aproximación inicial para el .xlsx; los píxeles exactos se fijan luego vía API de Sheets
    anchos_aprox = {col: max((px - 5) / 7.0, 1.0) for col, px in anchos_columnas_px.items()}
    altos_aprox = {fila: px * 0.75 for fila, px in altos_filas_px.items()}

    temp_excel_file = "temp_plantilla_modulo.xlsx"

    # 2. Definición estructural de datos y celdas
    encabezados = ['#', 'T', 'IDENTIFICACIÓN', 'NOMBRES', 'APELLIDOS', 'ESTADO', '%', None]

    valores_base = [
        [parametros_variables['PROGRAMA'], None, None, None, None, None, parametros_variables['TRIMESTRE'], None],
        [None, None, None, None, None, None, parametros_variables['TEMATICAS'], None],
        [f"Ficha: {parametros_variables['FICHA']}" if parametros_variables['FICHA'] else 'Ficha de Caracterización', None, None, None, None, None, None, None],
        [f"Fecha Reporte: {parametros_variables['FECHA_REPORTE']}" if parametros_variables['FECHA_REPORTE'] else 'Fecha del Reporte', None, None, None, None, None, None, None],
        [f"Inicio: {parametros_variables['FECHA_INICIO']}" if parametros_variables['FECHA_INICIO'] else 'Fecha Inicio', None, None, None, None, None, parametros_variables['COMPETENCIA'], None],
        [f"Fin: {parametros_variables['FECHA_FIN']}" if parametros_variables['FECHA_FIN'] else 'Fecha Fin', None, None, None, None, None, parametros_variables['RAP'], None],
        encabezados,
        [None] * 8
    ]

    estilos_celda = [
        [32, 33, 33, 33, 33, 34, 4, 5],
        [35, 36, 36, 36, 36, 37, 38, 39],
        [40, 41, 42, 17, 15, 15, 43, 43],
        [40, 41, 42, 17, 15, 15, 44, 44],
        [40, 41, 42, 17, 15, 15, 19, 20],
        [40, 41, 42, 21, 22, 15, 19, 23],
        [24, 24, 24, 24, 24, 24, 25, 26],
        [27, 28, 28, 28, 28, 28, 29, 30]
    ]

    celdas_combinadas = ['A5:C5', 'A6:C6', 'A1:F2', 'G2:G4', 'H2:H4', 'A3:C3', 'A4:C4']

    NEGRO = {'type': 'rgb', 'rgb': 'FF000000'}
    TEMA1 = {'type': 'theme', 'theme': 1, 'tint': 0.0}
    LADO = {'style': 'thin', 'color': NEGRO}

    definiciones_estilo = {
        4: {'font': {'name': 'Calibri', 'sz': 9.0, 'b': True, 'i': False, 'color': TEMA1},
            'fill': {'fill_type': 'solid', 'fgColor': {'type': 'rgb', 'rgb': 'FFC4EED0'}},
            'border': {'left': {'style': None}, 'right': LADO, 'top': LADO, 'bottom': LADO},
            'alignment': {'horizontal': 'center', 'vertical': 'center'}},
        5: {'font': {'name': 'Calibri', 'sz': 9.0, 'b': True, 'i': False, 'color': TEMA1},
            'fill': {'fill_type': 'solid', 'fgColor': {'type': 'rgb', 'rgb': 'FFCFE2F3'}},
            'border': {'left': LADO, 'right': LADO, 'top': LADO, 'bottom': LADO},
            'alignment': {'horizontal': 'center', 'vertical': 'center'}},
        15: {'font': {'name': 'Calibri', 'sz': 10.0, 'b': False, 'i': True, 'color': TEMA1},
             'fill': {'fill_type': 'solid', 'fgColor': {'type': 'rgb', 'rgb': 'FFE2F3F5'}},
             'border': {'left': LADO, 'right': LADO, 'top': LADO, 'bottom': LADO},
             'alignment': {'horizontal': 'left'}},
        17: {'font': {'name': 'Calibri', 'sz': 10.0, 'b': False, 'i': True, 'color': TEMA1},
             'fill': {'fill_type': 'solid', 'fgColor': {'type': 'rgb', 'rgb': 'FFE2F3F5'}},
             'border': {'left': {'style': None}, 'right': {'style': None}, 'top': LADO, 'bottom': LADO},
             'alignment': {'horizontal': 'left'}},
        19: {'font': {'name': 'Calibri', 'sz': 9.0, 'b': True, 'i': False, 'color': TEMA1},
             'fill': {'fill_type': 'solid', 'fgColor': {'type': 'rgb', 'rgb': 'FFC4EED0'}},
             'border': {'left': {'style': None}, 'right': LADO, 'top': LADO, 'bottom': LADO},
             'alignment': {'horizontal': 'center', 'wrap_text': True}},
        20: {'font': {'name': 'Calibri', 'sz': 8.0, 'b': False, 'i': True, 'color': TEMA1},
             'fill': {'fill_type': None},
             'border': {'left': LADO, 'right': LADO, 'top': LADO, 'bottom': LADO},
             'alignment': {'horizontal': 'center', 'vertical': 'center'}},
        21: {'font': {'name': 'Calibri', 'sz': 10.0, 'b': False, 'i': True, 'color': TEMA1},
             'fill': {'fill_type': 'solid', 'fgColor': {'type': 'rgb', 'rgb': 'FFE2F3F5'}},
             'border': {'left': {'style': None}, 'right': {'style': None}, 'top': LADO, 'bottom': None},
             'alignment': {'horizontal': 'left'}},
        22: {'font': {'name': 'Calibri', 'sz': 10.0, 'b': False, 'i': True, 'color': TEMA1},
             'fill': {'fill_type': 'solid', 'fgColor': {'type': 'rgb', 'rgb': 'FFE2F3F5'}},
             'border': {'left': LADO, 'right': LADO, 'top': LADO, 'bottom': None},
             'alignment': {'horizontal': 'left'}},
        23: {'font': {'name': 'Calibri', 'sz': 8.0, 'b': False, 'i': True, 'color': TEMA1},
             'fill': {'fill_type': None},
             'border': {'left': LADO, 'right': LADO, 'top': LADO, 'bottom': None},
             'alignment': {'horizontal': 'center', 'vertical': 'center'}},
        24: {'font': {'name': 'Calibri', 'sz': 11.0, 'b': True, 'i': True, 'color': TEMA1},
             'fill': {'fill_type': 'solid', 'fgColor': {'type': 'rgb', 'rgb': 'FFE2F3F5'}},
             'border': {'left': LADO, 'right': LADO, 'top': LADO, 'bottom': LADO},
             'alignment': {'horizontal': 'center'}},
        25: {'font': {'name': 'Calibri', 'sz': 11.0, 'b': True, 'i': True, 'color': TEMA1},
             'fill': {'fill_type': 'solid', 'fgColor': {'type': 'rgb', 'rgb': 'FFE2F3F5'}},
             'border': {'left': LADO, 'right': LADO, 'top': LADO, 'bottom': LADO},
             'alignment': {'horizontal': 'center'}},
        26: {'font': {'name': 'Calibri', 'sz': 5.0, 'color': NEGRO},
             'fill': {'fill_type': 'solid', 'fgColor': {'type': 'rgb', 'rgb': 'FFE2F3F5'}},
             'border': {'left': LADO, 'right': LADO, 'top': LADO, 'bottom': LADO},
             'alignment': {'horizontal': 'center', 'vertical': 'center'}},
        27: {'font': {'name': 'Calibri', 'sz': 9.0, 'i': True, 'color': TEMA1},
             'fill': {'fill_type': 'solid', 'fgColor': {'type': 'rgb', 'rgb': 'FFD8EDDF'}},
             'border': {'left': LADO, 'right': LADO, 'top': LADO, 'bottom': LADO},
             'alignment': {'horizontal': 'center', 'vertical': 'center'}},
        28: {'font': {'name': 'Calibri', 'sz': 11.0, 'color': TEMA1},
             'fill': {'fill_type': 'solid', 'fgColor': {'type': 'rgb', 'rgb': 'FFD8EDDF'}},
             'border': {'left': LADO, 'right': LADO, 'top': LADO, 'bottom': LADO}},
        29: {'font': {'name': 'Calibri', 'sz': 8.0, 'color': TEMA1},
             'fill': {'fill_type': 'solid', 'fgColor': {'type': 'rgb', 'rgb': 'FFD8EDDF'}},
             'border': {'left': LADO, 'right': LADO, 'top': LADO, 'bottom': LADO},
             'alignment': {'horizontal': 'center', 'vertical': 'center'}},
        30: {'font': {'name': 'Calibri', 'sz': 11.0, 'b': True, 'color': {'type': 'rgb', 'rgb': 'FFFF0000'}},
             'fill': {'fill_type': 'solid', 'fgColor': {'type': 'rgb', 'rgb': 'FFFFFFFF'}},
             'border': {'left': LADO, 'right': LADO, 'top': LADO, 'bottom': LADO},
             'alignment': {'horizontal': 'center'}},
        32: {'font': {'name': 'Calibri', 'sz': 11.0, 'b': True, 'color': TEMA1},
             'fill': {'fill_type': 'solid', 'fgColor': {'type': 'rgb', 'rgb': 'FFE2F3F5'}},
             'border': {'left': LADO, 'right': LADO, 'top': LADO, 'bottom': LADO},
             'alignment': {'horizontal': 'center', 'vertical': 'center', 'wrap_text': True}},
        33: {'font': {'name': 'Calibri', 'sz': 10.0, 'color': NEGRO},
             'border': {'left': None, 'right': None, 'top': LADO, 'bottom': None}},
        34: {'font': {'name': 'Calibri', 'sz': 10.0, 'color': NEGRO},
             'border': {'left': None, 'right': LADO, 'top': LADO, 'bottom': None}},
        35: {'font': {'name': 'Calibri', 'sz': 10.0, 'color': NEGRO},
             'border': {'left': LADO, 'right': None, 'top': None, 'bottom': LADO}},
        36: {'font': {'name': 'Calibri', 'sz': 10.0, 'color': NEGRO},
             'border': {'left': None, 'right': None, 'top': None, 'bottom': LADO}},
        37: {'font': {'name': 'Calibri', 'sz': 10.0, 'color': NEGRO},
             'border': {'left': None, 'right': LADO, 'top': None, 'bottom': LADO}},
        38: {'font': {'name': 'Calibri', 'sz': 8.0, 'b': True, 'color': NEGRO},
             'fill': {'fill_type': 'solid', 'fgColor': {'type': 'rgb', 'rgb': 'FFC4EED0'}},
             'border': {'left': LADO, 'right': LADO, 'top': LADO, 'bottom': LADO},
             'alignment': {'horizontal': 'center', 'vertical': 'center', 'text_rotation': 75}},
        39: {'font': {'name': 'Calibri', 'sz': 7.0, 'color': NEGRO},
             'fill': {'fill_type': 'solid', 'fgColor': {'type': 'rgb', 'rgb': 'FFCFE2F3'}},
             'border': {'left': LADO, 'right': LADO, 'top': LADO, 'bottom': LADO},
             'alignment': {'vertical': 'bottom', 'text_rotation': 90}},
        40: {'font': {'name': 'Calibri', 'sz': 11.0, 'b': True, 'i': True, 'color': TEMA1},
             'fill': {'fill_type': 'solid', 'fgColor': {'type': 'rgb', 'rgb': 'FFE2F3F5'}},
             'border': {'left': LADO, 'right': LADO, 'top': LADO, 'bottom': LADO},
             'alignment': {'horizontal': 'left'}},
        41: {'font': {'name': 'Calibri', 'sz': 10.0, 'color': NEGRO},
             'border': {'left': None, 'right': None, 'top': LADO, 'bottom': LADO}},
        42: {'font': {'name': 'Calibri', 'sz': 10.0, 'color': NEGRO},
             'border': {'left': None, 'right': LADO, 'top': LADO, 'bottom': LADO}},
        43: {'font': {'name': 'Calibri', 'sz': 10.0, 'color': NEGRO},
             'border': {'left': LADO, 'right': LADO, 'top': None, 'bottom': None}},
        44: {'font': {'name': 'Calibri', 'sz': 10.0, 'color': NEGRO},
             'border': {'left': LADO, 'right': LADO, 'top': None, 'bottom': LADO}}
    }

    # Helper para formatear color
    def crear_color(datos):
        if not datos or not datos.get("type"):
            return None
        tipo = datos["type"]
        if tipo == "rgb":
            return Color(type=tipo, rgb=datos.get("rgb"))
        if tipo == "theme":
            return Color(type=tipo, theme=datos.get("theme"), tint=datos.get("tint", 0))
        return None

    # Helper para mapear estilos
    def aplicar_estilo(celda, datos, forzar_sz=None, remover_negrita=False, forzar_valign_center=False):
        f = datos["font"]
        kw_font = {k: f[k] for k in ("name", "sz", "b", "i") if f.get(k) is not None}
        kw_font["name"] = "Calibri"  # Calibri en toda la hoja

        if forzar_sz is not None:
            kw_font["sz"] = forzar_sz
        if remover_negrita:
            kw_font["b"] = False

        color = crear_color(f.get("color"))
        if color:
            kw_font["color"] = color
        celda.font = Font(**kw_font)

        if "fill" in datos:
            fl = datos["fill"]
            fg = crear_color(fl.get("fgColor"))
            kw_fill = {"fill_type": fl.get("fill_type")}
            if fg:
                kw_fill["fgColor"] = fg
            celda.fill = PatternFill(**kw_fill)

        def crear_lado(d):
            if not d:
                return Side()
            kw = {}
            if d.get("style") is not None:
                kw["style"] = d["style"]
            co = crear_color(d.get("color"))
            if co:
                kw["color"] = co
            return Side(**kw)

        b = datos["border"]
        kw_border = {k: crear_lado(b[k]) for k in ("left", "right", "top", "bottom") if b.get(k) is not None}
        celda.border = Border(**kw_border)

        if "alignment" in datos:
            a = datos["alignment"]
            kw_align = {"horizontal": a.get("horizontal"), "vertical": a.get("vertical"),
                        "textRotation": a.get("text_rotation"), "wrapText": a.get("wrap_text")}
            if forzar_valign_center:
                kw_align["vertical"] = "center"
            celda.alignment = Alignment(**{k: v for k, v in kw_align.items() if v is not None})
        elif forzar_valign_center:
            celda.alignment = Alignment(vertical="center")

    # 3. Generar Workbook
    libro = Workbook()
    ws = libro.active
    ws.title = NOMBRE_HOJA

    # Inmovilizar filas 1-7 y columnas A-G (celda ancla H8)
    ws.freeze_panes = 'H8'

    for fila in range(1, TOTAL_FILAS + 1):
        for columna in range(1, TOTAL_COLUMNAS + 1):
            celda = ws.cell(fila, columna, valores_base[fila - 1][columna - 1])
            id_estilo = estilos_celda[fila - 1][columna - 1]

            sz_personalizado = None
            quitar_b = False

            # Fuente de 10 pt (metadatos, encabezados y alumnos)
            if ((3 <= fila <= 6) and (1 <= columna <= 6)) or \
               (fila == 7 and (1 <= columna <= 6)) or \
               (fila == 8 and (2 <= columna <= 6)):
                sz_personalizado = 10.0

            # Ajustes específicos del bloque de alumnos (fila 8)
            if fila == 8:
                if columna == 1:      # A8 -> 9 pt
                    sz_personalizado = 9.0
                elif columna == 7:    # G8 -> 8 pt
                    sz_personalizado = 8.0
                elif columna == 8:    # H8 -> 9 pt, sin negrilla
                    sz_personalizado = 9.0
                    quitar_b = True

            # Alineación vertical al centro
            rango_valign = (5 <= fila <= 8) or \
                           ((3 <= fila <= 4) and (1 <= columna <= 6)) or \
                           (fila == 1 and columna == 7)

            aplicar_estilo(celda, definiciones_estilo[id_estilo],
                           forzar_sz=sz_personalizado,
                           remover_negrita=quitar_b,
                           forzar_valign_center=rango_valign)

    # Notas emergentes (comentarios)
    ws['B7'].comment = Comment('TIPO DOCUMENTO', 'SENA')
    ws['A7'].comment = Comment('INDICE ORDEN', 'SENA')
    ws['G1'].comment = Comment('TRIMESTRE', 'SENA')
    ws['G5'].comment = Comment('COMPETENCIAS', 'SENA')
    ws['G6'].comment = Comment('RESULTADOS DE APRENDIZAJE', 'SENA')

    for columna, ancho in anchos_aprox.items():
        ws.column_dimensions[columna].width = ancho
    for fila, alto in altos_aprox.items():
        ws.row_dimensions[fila].height = alto
    for rango in celdas_combinadas:
        ws.merge_cells(rango)

    ws.sheet_view.showGridLines = True
    ws.print_area = f"A1:H{TOTAL_FILAS}"
    ws.sheet_properties.pageSetUpPr.fitToPage = True
    ws.page_setup.orientation = "landscape"
    ws.page_setup.fitToWidth = 1
    ws.page_setup.fitToHeight = 1
    ws.page_margins = PageMargins(left=0.2, right=0.2, top=0.3, bottom=0.3, header=0.1, footer=0.1)

    libro.save(temp_excel_file)

    # 4. Subir a Drive y fijar dimensiones exactas con la API de Sheets
    try:
        drive_service = build('drive', 'v3')

        if crear_copia_respaldo:
            nombre_original = drive_service.files().get(
                fileId=spreadsheet_id, fields='name', supportsAllDrives=True).execute()['name']
            copia = drive_service.files().copy(
                fileId=spreadsheet_id,
                body={'name': f"{nombre_original} (respaldo)"},
                supportsAllDrives=True).execute()
            print(f"Copia de respaldo creada con ID: {copia['id']}")

        media = MediaFileUpload(
            temp_excel_file,
            mimetype='application/vnd.openxmlformats-officedocument.spreadsheetml.sheet',
            resumable=True)
        drive_service.files().update(
            fileId=spreadsheet_id, media_body=media, supportsAllDrives=True).execute()

        sheets_service = build('sheets', 'v4')

        # sheetId real de la hoja destino (falla de forma explícita si no existe)
        spreadsheet = sheets_service.spreadsheets().get(spreadsheetId=spreadsheet_id).execute()
        sheet_id = next((s['properties']['sheetId'] for s in spreadsheet.get('sheets', [])
                         if s['properties']['title'] == NOMBRE_HOJA), None)
        if sheet_id is None:
            raise ValueError(f"No se encontró la hoja '{NOMBRE_HOJA}' tras la importación.")

        requests = []

        # Anchos exactos en píxeles
        for col, px in anchos_columnas_px.items():
            idx = ord(col.upper()) - ord('A')
            requests.append({'updateDimensionProperties': {
                'range': {'sheetId': sheet_id, 'dimension': 'COLUMNS',
                          'startIndex': idx, 'endIndex': idx + 1},
                'properties': {'pixelSize': px},
                'fields': 'pixelSize'}})

        # Altos exactos en píxeles
        for fila, px in altos_filas_px.items():
            requests.append({'updateDimensionProperties': {
                'range': {'sheetId': sheet_id, 'dimension': 'ROWS',
                          'startIndex': fila - 1, 'endIndex': fila},
                'properties': {'pixelSize': px},
                'fields': 'pixelSize'}})

        # Límite estricto de la cuadrícula: 8 filas x 8 columnas
        requests.append({'updateSheetProperties': {
            'properties': {'sheetId': sheet_id,
                           'gridProperties': {'rowCount': TOTAL_FILAS, 'columnCount': TOTAL_COLUMNAS}},
            'fields': 'gridProperties(rowCount,columnCount)'}})

        sheets_service.spreadsheets().batchUpdate(
            spreadsheetId=spreadsheet_id, body={'requests': requests}).execute()

        print(f"¡Éxito! Plantilla actualizada correctamente en Google Sheets con ID: {spreadsheet_id}")

    finally:
        if os.path.exists(temp_excel_file):
            os.remove(temp_excel_file)

## 2. Conexión con Google (Colab)

In [ ]:
import logging
# Silenciar el logger de google_auth_httplib2 para que no muestre las advertencias de timeout
logging.getLogger('google_auth_httplib2').setLevel(logging.ERROR)

import warnings
warnings.filterwarnings("ignore", module="google_auth_httplib2")
# pandas.to_datetime sin formato explícito: "Could not infer format..." (no cambia el resultado, solo el aviso)
warnings.filterwarnings("ignore", message=".*Could not infer format.*", category=UserWarning)

from google.colab import auth, drive
from googleapiclient.discovery import build
from google.auth import default
from google.colab import files
import pandas as pd
import gspread
import string
import datetime
import pytz
from types import SimpleNamespace
bogota_tz = pytz.timezone(ZONA_HORARIA)

# Autenticación rápida de usuario y montaje forzado de Google Drive
auth.authenticate_user()
drive.mount('/content/drive', force_remount=True)

creds, _ = default()
gc = gspread.authorize(creds)

drive_service = build('drive', 'v3', credentials=creds) # Build Drive's service
sheets_service = build('sheets', 'v4', credentials=creds)

Mounted at /content/drive


## 3. Funciones

### 📂 Gestión de la Carpeta Raíz de Instalación y Estructura Dinámica

La raíz se cambia añadiendo `_xYz_` al final de su nombre en Drive (o desde el panel «⚙ Carpetas»). Las carpetas internas se reconocen por propiedades, no por su nombre ni su ubicación.

In [ ]:
# ============================================================
# VINCULACIÓN DE CARPETAS — núcleo (sin interfaz)
# ------------------------------------------------------------
# • Raíz: se reconoce por la propiedad privada 'carpeta_raiz_instalacion_script_juicios' = 'true'.
#   - Si su nombre TERMINA en '_xYz_' se entiende que se pidió cambiarla.
#   - Si no hay ninguna raíz vinculada hay que vincular o crear una.
#   En ambos casos se abre el panel gráfico de la celda siguiente y se lanza RaizPendiente.
# • Carpetas internas: se reconocen por propiedades privadas (identidad lógica), no por nombre ni ubicación.
#   Si hay varias candidatas se lanza ConflictoCarpeta y el panel permite elegir una.
# • Nada de lo que hay DENTRO de una carpeta anterior se mueve, borra ni modifica: solo se cambian propiedades.
# ============================================================
from types import SimpleNamespace

SUFIJO_CAMBIO_RAIZ = '_xYz_'
PROP_RAIZ = 'carpeta_raiz_instalacion_script_juicios'
NOMBRE_RAIZ_POR_DEFECTO = 'Ecosistema Juicios Evaluativos - Raíz'
MIME_CARPETA = 'application/vnd.google-apps.folder'


class RaizPendiente(Exception):
    """No hay una carpeta raíz utilizable en este momento; el panel de vinculación quedó abierto."""


class ConflictoCarpeta(Exception):
    """Varias carpetas pueden cumplir la misma función; hay que elegir una en el panel de vinculación."""
    def __init__(self, slot, candidatos):
        self.slot = slot
        self.candidatos = candidatos
        super().__init__(f"Hay {len(candidatos)} carpetas para «{slot.etiqueta}». "
                         "Elige cuál usar en el panel de vinculación y vuelve a ejecutar.")


# ------------------------------------------------------------
# Funciones lógicas de carpeta ("slots"): qué propiedades las identifican
# ------------------------------------------------------------
def _q(valor):
    """Escapa un valor para una consulta de Drive."""
    return str(valor).replace('\\', '\\\\').replace("'", "\\'")


def _slot(clave, etiqueta, condiciones, activar=None, desactivar=None):
    return SimpleNamespace(clave=clave, etiqueta=etiqueta, condiciones=dict(condiciones),
                           activar=dict(activar or condiciones), desactivar=dict(desactivar or {}))


def slot_raiz():
    return _slot('raiz', 'Carpeta raíz de instalación', {PROP_RAIZ: 'true'}, desactivar={PROP_RAIZ: 'false'})


def slot_orden():
    return _slot('orden', 'Orden de RAPs (Formatos)', {'es_carpeta_orden_rap': 'true'},
                 desactivar={'es_carpeta_orden_rap': 'false'})


def slot_plantillas():
    return _slot('plantillas', 'Plantillas de Juicios Evaluativos', {'es_carpeta_plantillas_sena': 'true'},
                 desactivar={'es_carpeta_plantillas_sena': 'false'})


def slot_procesar():
    return _slot('procesar', 'Juicios a Procesar (entrada)', {'juicios_procesar_generica': 'true'},
                 desactivar={'juicios_procesar_generica': 'false'})


def slot_procesados(codigo_programa):
    cod = str(codigo_programa)
    return _slot('procesados', f'Juicios Procesados (programa {cod})', {'codigo_programa': cod, 'procesados': 'true'},
                 desactivar={'procesados': 'false'})


def slot_etapa(nombre_etapa):
    """Carpeta de salida de un programa y etapa; nombre_etapa es el nombre LÓGICO, p. ej. 'MEII-Lectiva'."""
    return _slot('etapa', f'Salida {nombre_etapa}', {'carpeta_etapa_salida': str(nombre_etapa)},
                 activar={'es_carpeta_etapa_salida': 'true', 'carpeta_etapa_salida': str(nombre_etapa)},
                 desactivar={'carpeta_etapa_salida': 'false'})


def _consulta_slot(slot):
    partes = [f"appProperties has {{ key='{_q(k)}' and value='{_q(v)}' }}" for k, v in slot.condiciones.items()]
    return ' and '.join(partes + [f"mimeType='{MIME_CARPETA}'", "trashed = false"])


def buscar_carpetas_slot(slot):
    """Carpetas de Drive (no eliminadas) que cumplen la función `slot`, estén donde estén y se llamen como se llamen."""
    res = drive_service.files().list(q=_consulta_slot(slot), fields="files(id, name, parents, modifiedTime)").execute()
    return res.get('files', [])


def _abrir_panel(modo, **ctx):
    """Abre el panel gráfico (celda siguiente). Un fallo de interfaz nunca debe ocultar el error de fondo."""
    abrir = globals().get('mostrar_panel_vinculacion')
    if abrir is None:
        return
    try:
        abrir(modo, **ctx)
    except Exception as e:
        print(f"⚠️ No se pudo abrir el panel de vinculación: {e}")


def lanzar_conflicto(slot, candidatos):
    """Abre el panel en modo conflicto y lanza ConflictoCarpeta (no se elige ninguna carpeta arbitrariamente)."""
    print(f"⚠️ [CONFLICTO] Hay {len(candidatos)} carpetas para «{slot.etiqueta}»: "
          + ", ".join(f"'{c.get('name', '')}'" for c in candidatos))
    _abrir_panel('conflicto', slot=slot, candidatos=candidatos)
    raise ConflictoCarpeta(slot, candidatos)


def unica_o_conflicto(carpetas, slot):
    """ID de la única carpeta; si hay varias, informa del conflicto en lugar de tomar la primera."""
    if len(carpetas) > 1:
        lanzar_conflicto(slot, carpetas)
    return carpetas[0]['id']


# ------------------------------------------------------------
# Cachés y estado global que pueden conservar referencias obsoletas
# ------------------------------------------------------------
def limpiar_caches_carpetas(solo_rutas=False):
    """Vacía las cachés de rutas/validación y olvida la plantilla genérica resuelta (depende de la carpeta de plantillas)."""
    for nombre in ('_cache_rutas_carpetas',) if solo_rutas else ('_cache_rutas_carpetas', '_cache_validacion_hibrida',
                                                                '_cache_carpetas_id', '_cache_carpetas_etapas'):
        cache = globals().get(nombre)
        if isinstance(cache, dict):
            cache.clear()
    if not solo_rutas:
        globals()['ID_PLANTILLA_GENERICA'] = None


# ------------------------------------------------------------
# Validación y vinculación
# ------------------------------------------------------------
def termina_con_sufijo_cambio(nombre):
    return (nombre or '').endswith(SUFIJO_CAMBIO_RAIZ)


def validar_carpeta_drive(folder_id, como_raiz=False):
    """Metadatos de la carpeta si es una carpeta de Drive accesible y editable; ValueError con mensaje claro si no."""
    if not folder_id:
        raise ValueError("Indica el ID o la URL de una carpeta de Drive.")
    try:
        meta = drive_service.files().get(fileId=folder_id, supportsAllDrives=True,
                                         fields='id, name, mimeType, trashed, capabilities(canEdit, canAddChildren)').execute()
    except Exception as e:
        raise ValueError(f"No se pudo acceder a «{folder_id}» (¿ID incorrecto o sin permiso?): {e}")
    if meta.get('trashed'):
        raise ValueError(f"La carpeta «{meta.get('name')}» está en la papelera.")
    if meta.get('mimeType') != MIME_CARPETA:
        raise ValueError(f"«{meta.get('name')}» no es una carpeta de Drive (es un archivo o un acceso directo).")
    caps = meta.get('capabilities') or {}
    if caps.get('canEdit') is False or caps.get('canAddChildren') is False:
        raise ValueError(f"No tienes permiso para editar «{meta.get('name')}»; se necesita para vincularla y crear contenido en ella.")
    if como_raiz and termina_con_sufijo_cambio(meta.get('name')):
        raise ValueError(f"El nombre «{meta.get('name')}» termina en «{SUFIJO_CAMBIO_RAIZ}», que es la marca de cambio de raíz. "
                         "Elige otra carpeta o quita esa terminación del nombre.")
    return meta


def vincular_carpeta_slot(slot, folder_id, meta=None):
    """Deja `folder_id` como ÚNICA carpeta con la función `slot`: activa la elegida y desactiva las demás.
    Solo se cambian propiedades (nada se mueve, renombra ni borra). Si algo falla se revierte."""
    meta = meta or validar_carpeta_drive(folder_id)
    previas = buscar_carpetas_slot(slot)
    ids_previos = {c['id'] for c in previas}
    desactivadas = []
    try:
        drive_service.files().update(fileId=folder_id, body={'appProperties': dict(slot.activar)},
                                     fields='id', supportsAllDrives=True).execute()
        for c in previas:
            if c['id'] != folder_id:
                drive_service.files().update(fileId=c['id'], body={'appProperties': dict(slot.desactivar)},
                                             fields='id', supportsAllDrives=True).execute()
                desactivadas.append(c['id'])
    except Exception as e:
        errores = []
        for cid in desactivadas:                       # restaurar las que ya se habían desactivado
            try:
                drive_service.files().update(fileId=cid, body={'appProperties': dict(slot.activar)},
                                             fields='id', supportsAllDrives=True).execute()
            except Exception as e2:
                errores.append(f"{cid}: {e2}")
        if folder_id not in ids_previos:               # la elegida no estaba vinculada antes: desmarcarla
            try:
                drive_service.files().update(fileId=folder_id, body={'appProperties': dict(slot.desactivar)},
                                             fields='id', supportsAllDrives=True).execute()
            except Exception as e2:
                errores.append(f"{folder_id}: {e2}")
        extra = f" (no se pudo restaurar: {'; '.join(errores)})" if errores else " Se restauró la vinculación anterior."
        raise ValueError(f"No se pudo completar la vinculación de «{slot.etiqueta}»: {e}.{extra}")
    limpiar_caches_carpetas()
    return meta


def vincular_raiz(nuevo_id):
    """Vincula `nuevo_id` como carpeta raíz. La raíz anterior solo deja de estar marcada: no se toca su contenido."""
    meta = validar_carpeta_drive(nuevo_id, como_raiz=True)
    vincular_carpeta_slot(slot_raiz(), nuevo_id, meta)
    globals()['carpeta_raiz_instalacion_id'] = nuevo_id
    return meta


def crear_raiz(nombre=None):
    """Crea una carpeta nueva en 'Mi unidad' y la vincula como raíz (si la vinculación falla, la anterior se conserva)."""
    nombre = (nombre or '').strip() or NOMBRE_RAIZ_POR_DEFECTO
    if termina_con_sufijo_cambio(nombre):
        raise ValueError(f"El nombre no puede terminar en «{SUFIJO_CAMBIO_RAIZ}» (es la marca de cambio de raíz).")
    try:
        nueva = drive_service.files().create(
            body={'name': nombre, 'mimeType': MIME_CARPETA, 'parents': ['root'], 'appProperties': {PROP_RAIZ: 'false'}},
            fields='id, name', supportsAllDrives=True).execute()
    except Exception as e:
        raise ValueError(f"Error al crear la nueva carpeta raíz en Drive: {e}")
    try:
        vincular_raiz(nueva['id'])
    except Exception as e:
        raise ValueError(f"Se creó la carpeta «{nombre}» (https://drive.google.com/drive/folders/{nueva['id']}) "
                         f"pero no se pudo vincular como raíz: {e}")
    return nueva


def quitar_sufijo_raiz(raiz):
    """Cancela una solicitud de cambio: quita '_xYz_' del final del nombre de la raíz actual (único cambio sobre ella)."""
    nombre = raiz.get('name', '')
    limpio = nombre[:-len(SUFIJO_CAMBIO_RAIZ)].rstrip() if termina_con_sufijo_cambio(nombre) else nombre
    limpio = limpio or NOMBRE_RAIZ_POR_DEFECTO
    if limpio != nombre:
        drive_service.files().update(fileId=raiz['id'], body={'name': limpio}, fields='id', supportsAllDrives=True).execute()
    globals()['carpeta_raiz_instalacion_id'] = raiz['id']
    return limpio


def _buscar_raices():
    return buscar_carpetas_slot(slot_raiz())


def obtener_o_vincular_raiz_instalacion():
    """ID de la carpeta raíz vinculada y vigente.

    No usa la consola. Si hay que vincular o cambiar la raíz abre el panel gráfico y lanza RaizPendiente:
    quien llama debe detenerse y el usuario reintenta cuando termine. Mientras la raíz sea válida y no se pida
    cambiarla (nombre sin '_xYz_' al final) devuelve su ID y el panel no aparece.
    """
    raices = _buscar_raices()
    if len(raices) > 1:
        lanzar_conflicto(slot_raiz(), raices)
    if raices:
        raiz = raices[0]
        nombre = raiz.get('name', '')
        if not termina_con_sufijo_cambio(nombre):
            return raiz['id']
        print(f"\n🔄 [CAMBIO DE RAÍZ SOLICITADO] La carpeta '{nombre}' termina en '{SUFIJO_CAMBIO_RAIZ}'.")
        print("   Se abre el panel de vinculación. Todavía no se ha modificado nada.")
        _abrir_panel('sufijo', raiz=raiz)
        raise RaizPendiente(f"La carpeta raíz «{nombre}» termina en «{SUFIJO_CAMBIO_RAIZ}»: elige la nueva raíz "
                            "(o conserva la actual) en el panel de vinculación y vuelve a ejecutar.")
    print("\n📢 [CONFIGURACIÓN INICIAL] No se ha detectado ninguna Carpeta Raíz de Instalación vinculada.")
    _abrir_panel('sin_raiz')
    raise RaizPendiente("No hay una carpeta raíz vinculada: vincula una existente o crea una nueva en el panel "
                        "de vinculación y vuelve a ejecutar.")

#### 🧭 Panel gráfico de vinculación (condicional)
Permanece oculto en la ejecución normal. Aparece solo si no hay raíz, si la raíz termina en `_xYz_`, si hay varias carpetas candidatas para una misma función o si pulsas «⚙ Carpetas».

In [ ]:
# ============================================================
# PANEL GRÁFICO CONDICIONAL DE VINCULACIÓN (ipywidgets)
# ------------------------------------------------------------
# Permanece OCULTO (display='none' y sin widgets hijos) en la ejecución normal. Aparece solo si:
#   1) no hay carpeta raíz vinculada, 2) la raíz termina en '_xYz_', 3) el usuario pulsa «⚙ Carpetas»,
#   o 4) hay varias carpetas candidatas para una misma función (conflicto).
# Al terminar o cancelar se vuelve a ocultar y la interfaz recupera su presentación normal.
# ============================================================
import html as _html
import ipywidgets as widgets
from IPython.display import display

contenedor_panel_vinculacion = widgets.VBox([], layout=widgets.Layout(display='none', width='100%'))
_estado_panel = {'modo': None, 'desplegado': False}
_HOOKS_VINCULACION = []


def registrar_hook_vinculacion(fn):
    """fn(evento, mensaje) se llama al vincular, cancelar o resolver un conflicto (la interfaz principal lo usa)."""
    if fn not in _HOOKS_VINCULACION:
        _HOOKS_VINCULACION.append(fn)


def _notificar(evento, mensaje):
    for fn in list(_HOOKS_VINCULACION):
        try:
            fn(evento, mensaje)
        except Exception as e:
            print(f"⚠️ Error en un aviso de la interfaz: {e}")


def _esc(t):
    return _html.escape(str(t if t is not None else ''))


def _enlace_carpeta(fid, texto=None):
    return (f"<a href='https://drive.google.com/drive/folders/{_esc(fid)}' target='_blank' "
            f"style='color:#1d4ed8;text-decoration:underline;'>{_esc(texto or fid)}</a>")


def _ruta_drive(fid):
    """Ruta legible (sin caché, para no mostrar nombres viejos tras renombrar o mover)."""
    nombres, visto = [], set()
    try:
        while fid and fid not in visto and len(nombres) < 30:
            visto.add(fid)
            m = drive_service.files().get(fileId=fid, fields='name, parents', supportsAllDrives=True).execute()
            nombres.append(m.get('name', ''))
            padres = m.get('parents') or []
            fid = padres[0] if padres else None
    except Exception:
        pass
    return ' > '.join(reversed(nombres)) or 'Desconocida'


def _banner(tipo, html_texto):
    color = {'error': ('#fef2f2', '#ef4444', '#991b1b'), 'ok': ('#f0fdf4', '#22c55e', '#166534'),
             'aviso': ('#fffbeb', '#f59e0b', '#92400e')}.get(tipo, ('#eff6ff', '#3b82f6', '#1e40af'))
    return (f"<div style='background:{color[0]};border-left:5px solid {color[1]};color:{color[2]};"
            f"padding:10px;border-radius:6px;margin:6px 0;font-size:13px;'>{html_texto}</div>")


def ocultar_panel_vinculacion():
    """Oculta el panel y libera sus widgets: no ocupa espacio."""
    contenedor_panel_vinculacion.children = ()
    contenedor_panel_vinculacion.layout.display = 'none'
    _estado_panel['modo'] = None


def mostrar_panel_vinculacion(modo='manual', **ctx):
    """modo: 'sin_raiz' | 'sufijo' | 'manual' | 'conflicto'."""
    if modo == 'conflicto':
        vista = _vista_conflicto(ctx['slot'], ctx['candidatos'])
    else:
        vista = _vista_raiz(modo, ctx.get('raiz'))
    contenedor_panel_vinculacion.children = (vista,)
    contenedor_panel_vinculacion.layout.display = None
    _estado_panel['modo'] = modo
    if not _estado_panel['desplegado']:     # la interfaz principal ya lo incluye en su tarjeta; si no, se muestra aquí
        try:
            display(contenedor_panel_vinculacion)
            _estado_panel['desplegado'] = True
        except Exception:
            pass


def abrir_panel_vinculacion():
    """Apertura pedida por el usuario (botón «⚙ Carpetas»). No lanza excepciones."""
    try:
        raices = _buscar_raices()
        if len(raices) > 1:
            mostrar_panel_vinculacion('conflicto', slot=slot_raiz(), candidatos=raices)
        elif not raices:
            mostrar_panel_vinculacion('sin_raiz')
        elif termina_con_sufijo_cambio(raices[0].get('name', '')):
            mostrar_panel_vinculacion('sufijo', raiz=raices[0])
        else:
            mostrar_panel_vinculacion('manual', raiz=raices[0])
    except Exception as e:
        _notificar('error', f"❌ No se pudo abrir el panel de carpetas: {e}")


def _boton(texto, estilo='', ancho='auto', **kw):
    return widgets.Button(description=texto, button_style=estilo, layout=widgets.Layout(width=ancho, min_width='120px'), **kw)


# ------------------------------------------------------------
# Vista de la carpeta raíz (y, en modo manual, de las carpetas internas)
# ------------------------------------------------------------
def _vista_raiz(modo, raiz=None):
    nombre_raiz = raiz.get('name', '') if raiz else ''
    if modo == 'sufijo':
        titulo = '🔄 Cambio de carpeta raíz solicitado'
        aviso = (f"La raíz actual <b>{_esc(nombre_raiz)}</b> termina en <code>{SUFIJO_CAMBIO_RAIZ}</code>. Elige la nueva raíz. "
                 "La carpeta anterior <b>no se mueve ni se modifica</b>; solo deja de estar marcada como raíz cuando confirmes.")
    elif modo == 'sin_raiz':
        titulo = '📁 Carpeta raíz: configuración inicial'
        aviso = 'No hay una carpeta raíz vinculada. Vincula una carpeta existente de Drive o crea una nueva.'
    else:
        titulo = '⚙ Carpeta raíz y carpetas internas'
        aviso = (f"Raíz actual: {_enlace_carpeta(raiz['id'], nombre_raiz)}. Solo necesitas este panel para cambiarla "
                 f"o para asociar carpetas internas. Para cambiar la raíz también puedes añadir <code>{SUFIJO_CAMBIO_RAIZ}</code> "
                 "al final de su nombre en Drive.") if raiz else 'Sin raíz vinculada.'

    cab = widgets.HTML(f"<div style='font-weight:bold;font-size:15px;margin-bottom:4px;'>{titulo}</div>" + _banner('info', aviso))
    sel = widgets.ToggleButtons(options=[('Vincular carpeta existente', 'existente'), ('Crear carpeta nueva', 'nueva')],
                                value='existente')
    txt_id = widgets.Text(placeholder='URL o ID de la carpeta de Drive', layout=widgets.Layout(width='95%'))
    txt_nombre = widgets.Text(value=NOMBRE_RAIZ_POR_DEFECTO, layout=widgets.Layout(width='95%', display='none'))
    btn_revisar = _boton('Revisar', 'info')
    btn_confirmar = _boton('✔ Confirmar vinculación', 'success', '210px')
    btn_confirmar.layout.display = 'none'
    btn_cancelar = _boton('Cerrar' if modo == 'manual' else 'Cancelar')
    btn_conservar = _boton('Conservar la raíz actual (quitar «%s» del nombre)' % SUFIJO_CAMBIO_RAIZ, '', '380px')
    area_prev = widgets.HTML('')
    area_msg = widgets.HTML('')
    pendiente = {}

    def al_cambiar_tipo(_):
        txt_id.layout.display = None if sel.value == 'existente' else 'none'
        txt_nombre.layout.display = None if sel.value == 'nueva' else 'none'
        btn_confirmar.layout.display = 'none'
        area_prev.value = ''
        pendiente.clear()
    sel.observe(al_cambiar_tipo, names='value')

    def al_editar(_):
        btn_confirmar.layout.display = 'none'
        area_prev.value = ''
        pendiente.clear()
    txt_id.observe(al_editar, names='value')
    txt_nombre.observe(al_editar, names='value')

    def revisar(_):
        area_msg.value = ''
        pendiente.clear()
        btn_confirmar.layout.display = 'none'
        try:
            if sel.value == 'existente':
                fid = extraer_id_drive(txt_id.value.strip()) if txt_id.value.strip() else ''
                meta = validar_carpeta_drive(fid, como_raiz=True)
                if raiz and meta['id'] == raiz['id'] and modo != 'sufijo':
                    raise ValueError('Esa carpeta ya es la raíz vinculada.')
                pendiente.update(tipo='existente', id=meta['id'], nombre=meta['name'])
                prev = (f"Se vinculará como raíz: {_enlace_carpeta(meta['id'], meta['name'])}<br>"
                        f"<small>Ruta: {_esc(_ruta_drive(meta['id']))}</small>")
            else:
                nombre = txt_nombre.value.strip() or NOMBRE_RAIZ_POR_DEFECTO
                if termina_con_sufijo_cambio(nombre):
                    raise ValueError(f"El nombre no puede terminar en «{SUFIJO_CAMBIO_RAIZ}».")
                pendiente.update(tipo='nueva', nombre=nombre)
                prev = f"Se creará la carpeta <b>{_esc(nombre)}</b> en «Mi unidad» y se vinculará como raíz."
            if raiz and not (pendiente.get('id') == raiz['id']):
                prev += (f"<br>La raíz anterior (<b>{_esc(nombre_raiz)}</b>) dejará de estar marcada como raíz. "
                         "Su contenido y las carpetas internas ya vinculadas <b>no se mueven</b>.")
            area_prev.value = _banner('aviso', prev + '<br><b>Pulsa «Confirmar vinculación» para aplicar el cambio.</b>')
            btn_confirmar.layout.display = None
        except ValueError as e:
            area_msg.value = _banner('error', _esc(e))

    def confirmar(_):
        if not pendiente:
            return
        try:
            meta = vincular_raiz(pendiente['id']) if pendiente['tipo'] == 'existente' else crear_raiz(pendiente['nombre'])
        except Exception as e:     # el núcleo ya restauró la vinculación anterior; el panel sigue abierto
            area_msg.value = _banner('error', _esc(e) + '<br>No se hizo ningún cambio definitivo.')
            return
        ocultar_panel_vinculacion()
        _notificar('raiz_vinculada', f"✅ Carpeta raíz vinculada: «{meta.get('name', pendiente.get('nombre'))}». "
                                      "Pulsa «Iniciar Tabulación» para continuar.")

    def cancelar(_):
        ocultar_panel_vinculacion()
        if modo == 'manual':
            _notificar('cancelado', 'Panel cerrado. Se sigue usando la carpeta raíz actual.')
        else:
            _notificar('cancelado', '⚠️ Vinculación cancelada: la carpeta raíz sigue pendiente. '
                                    'Vuelve a ejecutar para abrir de nuevo el panel.')

    def conservar(_):
        try:
            limpio = quitar_sufijo_raiz(raiz)
        except Exception as e:
            area_msg.value = _banner('error', f"No se pudo renombrar la carpeta: {_esc(e)}")
            return
        ocultar_panel_vinculacion()
        _notificar('raiz_conservada', f"✅ Se conserva la raíz «{limpio}» (se quitó la marca de cambio). Puedes continuar.")

    btn_revisar.on_click(revisar)
    btn_confirmar.on_click(confirmar)
    btn_cancelar.on_click(cancelar)
    btn_conservar.on_click(conservar)

    botones = [btn_revisar, btn_confirmar, btn_cancelar]
    hijos = [cab, sel, txt_id, txt_nombre, widgets.HBox(botones), area_prev]
    if modo == 'sufijo' and raiz:
        hijos.append(widgets.HBox([btn_conservar]))
    hijos.append(area_msg)
    if modo == 'manual':
        hijos.append(_seccion_internas())
    return widgets.VBox(hijos, layout=widgets.Layout(border='1px solid #cbd5e1', padding='12px', margin='0 0 12px 0'))


# ------------------------------------------------------------
# Vista de conflicto (varias carpetas candidatas para la misma función)
# ------------------------------------------------------------
def _vista_conflicto(slot, candidatos):
    filas = []
    for c in candidatos:
        filas.append(f"<li>{_enlace_carpeta(c['id'], c.get('name', ''))} — <small>{_esc(_ruta_drive(c['id']))}"
                     f" · modificada {_esc((c.get('modifiedTime') or '')[:10])}</small></li>")
    cab = widgets.HTML(
        f"<div style='font-weight:bold;font-size:15px;margin-bottom:4px;'>⚠️ Varias carpetas para «{_esc(slot.etiqueta)}»</div>"
        + _banner('aviso', f"Se encontraron {len(candidatos)} carpetas que podrían cumplir esta función. No se eligió ninguna "
                           "automáticamente. Elige la que se debe usar; las demás dejarán de estar vinculadas a esta función "
                           "(no se mueven ni se borran)." + f"<ul style='margin:6px 0 0 16px;'>{''.join(filas)}</ul>"))
    opciones = [(f"{c.get('name', '')}  ·  {_ruta_drive(c['id'])}", c['id']) for c in candidatos]
    radio = widgets.RadioButtons(options=opciones, layout=widgets.Layout(width='95%'))
    try:
        radio.index = None        # sin selección previa: la persona elige conscientemente
    except Exception:
        pass
    btn_usar = _boton('Usar la seleccionada', 'success', '200px', disabled=(radio.value is None or radio.index is None))
    btn_cancelar = _boton('Cancelar')
    area_msg = widgets.HTML('')

    radio.observe(lambda ch: setattr(btn_usar, 'disabled', radio.value is None), names='value')

    def usar(_):
        elegido = radio.value
        if not elegido:
            return
        try:
            if slot.clave == 'raiz':
                vincular_raiz(elegido)
                nombre = next((c.get('name') for c in candidatos if c['id'] == elegido), '')
            else:
                meta = vincular_carpeta_slot(slot, elegido)
                nombre = meta.get('name', '')
        except Exception as e:
            area_msg.value = _banner('error', _esc(e))
            return
        ocultar_panel_vinculacion()
        _notificar('conflicto_resuelto', f"✅ Conflicto resuelto: para «{slot.etiqueta}» se usará «{nombre}». "
                                          "Vuelve a pulsar «Iniciar Tabulación».")

    def cancelar(_):
        ocultar_panel_vinculacion()
        _notificar('cancelado', '⚠️ Conflicto sin resolver: el proceso no continuará hasta elegir una carpeta.')

    btn_usar.on_click(usar)
    btn_cancelar.on_click(cancelar)
    return widgets.VBox([cab, radio, widgets.HBox([btn_usar, btn_cancelar]), area_msg],
                        layout=widgets.Layout(border='1px solid #cbd5e1', padding='12px', margin='0 0 12px 0'))


# ------------------------------------------------------------
# Carpetas internas: ver lo vinculado, asociar una carpeta existente, desvincular
# ------------------------------------------------------------
def _listar_internas():
    """[(etiqueta, carpeta, slot)] de las carpetas internas hoy vinculadas (por propiedad, estén donde estén)."""
    filas = []
    for slot in (slot_orden(), slot_plantillas(), slot_procesar()):
        for c in buscar_carpetas_slot(slot):
            filas.append((slot.etiqueta, c, slot))
    consulta = "appProperties has {{ key='{k}' and value='true' }} and mimeType='%s' and trashed = false" % MIME_CARPETA
    campos = "files(id, name, parents, modifiedTime, appProperties)"
    for c in drive_service.files().list(q=consulta.format(k='procesados'), fields=campos).execute().get('files', []):
        cod = (c.get('appProperties') or {}).get('codigo_programa')
        if cod:
            filas.append((f'Juicios Procesados (programa {cod})', c, slot_procesados(cod)))
    for c in drive_service.files().list(q=consulta.format(k='es_carpeta_etapa_salida'), fields=campos).execute().get('files', []):
        nom = (c.get('appProperties') or {}).get('carpeta_etapa_salida')
        if nom and nom != 'false':
            filas.append((f'Salida {nom}', c, slot_etapa(nom)))
    return filas


def _seccion_internas():
    lista = widgets.VBox([])
    area_msg = widgets.HTML('')
    area_prev = widgets.HTML('')
    pendiente = {}

    def refrescar():
        limpiar_caches_carpetas(solo_rutas=True)
        items = []
        try:
            filas = _listar_internas()
        except Exception as e:
            lista.children = [widgets.HTML(_banner('error', f"No se pudieron leer las carpetas internas: {_esc(e)}"))]
            return
        if not filas:
            lista.children = [widgets.HTML("<i>Todavía no hay carpetas internas vinculadas; se crean solas al procesar.</i>")]
            return
        for etiqueta, c, slot in filas:
            info = widgets.HTML(f"<div style='font-size:12px;min-width:520px;'><b>{_esc(etiqueta)}</b>: "
                                f"{_enlace_carpeta(c['id'], c.get('name', ''))}<br><small>{_esc(_ruta_drive(c['id']))}</small></div>")
            b = _boton('Desvincular', 'warning', '110px')

            def desvincular(_, c=c, slot=slot):
                try:
                    drive_service.files().update(fileId=c['id'], body={'appProperties': dict(slot.desactivar)},
                                                 fields='id', supportsAllDrives=True).execute()
                    limpiar_caches_carpetas()
                except Exception as e:
                    area_msg.value = _banner('error', f"No se pudo desvincular: {_esc(e)}")
                    return
                area_msg.value = _banner('ok', f"«{_esc(c.get('name', ''))}» ya no está vinculada a «{_esc(slot.etiqueta)}». "
                                                "No se movió ni se borró nada; al procesar se usará otra o se creará una nueva.")
                _notificar('interna_cambiada', '')
                refrescar()
            b.on_click(desvincular)
            items.append(widgets.HBox([info, b], layout=widgets.Layout(align_items='center')))
        lista.children = items

    dd = widgets.Dropdown(options=[('Orden de RAPs (Formatos)', 'orden'), ('Plantillas', 'plantillas'),
                                   ('Juicios a Procesar (entrada)', 'procesar'),
                                   ('Juicios Procesados de un programa', 'procesados'),
                                   ('Salida de un programa (Lectiva/Productiva)', 'etapa')],
                          description='Función:', style={'description_width': '70px'}, layout=widgets.Layout(width='95%'))
    txt_id = widgets.Text(placeholder='URL o ID de la carpeta existente', description='Carpeta:',
                          style={'description_width': '70px'}, layout=widgets.Layout(width='95%'))
    txt_cod = widgets.Text(placeholder='Código del programa, p. ej. 224208', description='Código:',
                           style={'description_width': '70px'}, layout=widgets.Layout(width='95%', display='none'))
    txt_ini = widgets.Text(placeholder='Iniciales del programa, p. ej. MEII', description='Programa:',
                           style={'description_width': '70px'}, layout=widgets.Layout(width='60%', display='none'))
    dd_etapa = widgets.Dropdown(options=['Lectiva', 'Productiva'], description='Etapa:',
                                style={'description_width': '50px'}, layout=widgets.Layout(width='200px', display='none'))
    btn_rev = _boton('Revisar asociación', 'info', '170px')
    btn_ok = _boton('✔ Confirmar', 'success', '130px')
    btn_ok.layout.display = 'none'

    def visibilidad(_=None):
        txt_cod.layout.display = None if dd.value == 'procesados' else 'none'
        txt_ini.layout.display = None if dd.value == 'etapa' else 'none'
        dd_etapa.layout.display = None if dd.value == 'etapa' else 'none'
        btn_ok.layout.display = 'none'
        area_prev.value = ''
        pendiente.clear()
    dd.observe(visibilidad, names='value')

    def construir_slot():
        if dd.value == 'orden':
            return slot_orden()
        if dd.value == 'plantillas':
            return slot_plantillas()
        if dd.value == 'procesar':
            return slot_procesar()
        if dd.value == 'procesados':
            if not txt_cod.value.strip():
                raise ValueError('Indica el código del programa.')
            return slot_procesados(txt_cod.value.strip())
        if not txt_ini.value.strip():
            raise ValueError('Indica las iniciales del programa (las mismas del nombre de los archivos, p. ej. MEII).')
        return slot_etapa(f"{txt_ini.value.strip()}-{dd_etapa.value}")

    def revisar(_):
        area_msg.value = ''
        pendiente.clear()
        btn_ok.layout.display = 'none'
        try:
            slot = construir_slot()
            fid = extraer_id_drive(txt_id.value.strip()) if txt_id.value.strip() else ''
            meta = validar_carpeta_drive(fid)
            otras = [c for c in buscar_carpetas_slot(slot) if c['id'] != meta['id']]
            pendiente.update(slot=slot, id=meta['id'], meta=meta)
            txt = f"Se asociará {_enlace_carpeta(meta['id'], meta['name'])} a «{_esc(slot.etiqueta)}»."
            if otras:
                txt += (f"<br>{len(otras)} carpeta(s) que hoy cumplen esa función quedarán desvinculadas: "
                        + ', '.join(_esc(c.get('name', '')) for c in otras) + ". No se mueven ni se borran.")
            area_prev.value = _banner('aviso', txt)
            btn_ok.layout.display = None
        except ValueError as e:
            area_msg.value = _banner('error', _esc(e))

    def confirmar(_):
        if not pendiente:
            return
        try:
            meta = vincular_carpeta_slot(pendiente['slot'], pendiente['id'], pendiente['meta'])
        except Exception as e:
            area_msg.value = _banner('error', _esc(e))
            return
        area_prev.value = ''
        btn_ok.layout.display = 'none'
        pendiente.clear()
        area_msg.value = _banner('ok', f"✅ «{_esc(meta.get('name', ''))}» quedó asociada.")
        _notificar('interna_cambiada', '')
        refrescar()

    btn_rev.on_click(revisar)
    btn_ok.on_click(confirmar)
    form = widgets.VBox([widgets.HTML("<b>Asociar una carpeta existente a una función</b>"), dd, txt_id, txt_cod,
                         widgets.HBox([txt_ini, dd_etapa]), widgets.HBox([btn_rev, btn_ok]), area_prev, area_msg])
    cuerpo = widgets.VBox([widgets.HTML("<b>Carpetas internas vinculadas</b> "
                                        "<small>(siguen vinculadas aunque las renombres o las muevas)</small>"), lista, form])
    acordeon = widgets.Accordion(children=[cuerpo], selected_index=None)
    acordeon.set_title(0, 'Carpetas internas (ver, asociar o desvincular)')
    acordeon.observe(lambda ch: refrescar() if ch['new'] == 0 else None, names='selected_index')
    return acordeon

### 3.1 Utilidades de hoja de cálculo y constructores de solicitudes (API de Sheets)

In [ ]:
def generate_column_names(start_column='H', num_columns=56):
  """
  Generates a sequence of column names starting from a specified column letter.
  Args:
      start_column: The starting column letter (default: 'F').
      num_columns: The number of columns to generate (default: 56).

  Returns:
      A list of column names.
  """

  # Get all uppercase letters
  alphabet = string.ascii_uppercase
  # Calculate the starting index in the alphabet
  start_index = alphabet.index(start_column)
  # Generate column names using list comprehension
  column_names = []
  for i in range(num_columns):
    col_num = start_index + i
    col_str = ''
    while col_num >= 0:
      col_str = alphabet[col_num % 26] + col_str
      col_num = (col_num // 26) - 1
    column_names.append(col_str)
  return column_names

def column_to_index(column_string):
  """Convierte una letra de columna a su índice numérico (basado en 0)."""
  return sum((ord(char) - 64) * (26 ** i) for i, char in enumerate(reversed(column_string.upper()))) - 1

def cell_to_indexes(cell_string):
  """Convierte una referencia de celda (e.g., 'AB91') a índices de fila y columna (basados en 0)."""
  column, row = ''.join(filter(str.isalpha, cell_string)), ''.join(filter(str.isdigit, cell_string))
  return int(row) - 1, column_to_index(column)

def range_to_indexes(range_string):
  """Convierte un rango de celdas (e.g., 'H70:K240') a índices de inicio y fin."""
  start, end = range_string.split(':')
  start_row, start_col = cell_to_indexes(start)
  end_row, end_col = cell_to_indexes(end)
  return start_row, start_col, end_row + 1, end_col + 1  # +1 porque los rangos son inclusivos

# Constructores de solicitudes para la API de Google Sheets (spreadsheets.batchUpdate).
# Uso: solicitud(tipo, sheet_id, rango, cantidad, valor) -> dict listo para agregar a body['requests'].
# Todos los constructores comparten la misma firma (sheet_id, rango, cantidad, valor); cada uno usa solo lo que necesita.

def _grid(sheet_id, rango):
  """GridRange de la API a partir de un rango tipo 'H70:K240'."""
  f0, c0, f1, c1 = range_to_indexes(rango)
  return {'sheetId': sheet_id, 'startRowIndex': f0, 'endRowIndex': f1,
          'startColumnIndex': c0, 'endColumnIndex': c1}

def _posiciones_dimension(rango, dimension):
  """(inicio, fin) de filas o columnas para un rango 'A1:B2' o una celda individual 'A1'."""
  try:  # Rango de celdas
    f0, c0, f1, c1 = range_to_indexes(rango)
    return (f0, f1) if dimension == 'ROWS' else (c0, c1)
  except Exception:  # Celda individual
    fila, col = cell_to_indexes(rango)
    inicio = fila if dimension == 'ROWS' else col
    return inicio, inicio + 1

def _insertar_fila_col(sheet_id, rango, cantidad, valor):
  """valor = ['ROWS'|'COLUMNS', 'BEFORE'|'AFTER'] respecto a la celda indicada en rango."""
  fila, col = cell_to_indexes(rango)
  pos = fila if valor[0] == 'ROWS' else col
  if valor[1] == 'AFTER':
    pos += 1
  return {"insertDimension": {"range": {"sheetId": sheet_id, "dimension": valor[0],
                                         "startIndex": pos, "endIndex": pos + cantidad},
                              "inheritFromBefore": True}}

def _copiar_formato(sheet_id, rango, cantidad, valor):
  """Copia el formato del rango origen (rango) al rango destino (valor)."""
  return {"copyPaste": {"source": _grid(sheet_id, rango), "destination": _grid(sheet_id, valor),
                        "pasteType": "PASTE_FORMAT", "pasteOrientation": "NORMAL"}}

def _borrar_celdas(sheet_id, rango, cantidad, valor):
  return {'updateCells': {'range': _grid(sheet_id, rango), 'fields': 'userEnteredValue'}}

def _actualizar_celdas(sheet_id, rango, cantidad, valor):
  """Escribe un número (int/float) o un texto en el rango."""
  if isinstance(valor, (int, float)):
    celda, campos = {"numberValue": float(valor)}, "userEnteredValue.numberValue"
  else:
    celda, campos = {"stringValue": str(valor)}, "userEnteredValue.stringValue"
  return {"updateCells": {'range': _grid(sheet_id, rango),
                          "rows": [{"values": [{"userEnteredValue": celda}]}], "fields": campos}}

def _quitar_filtro(sheet_id, rango, cantidad, valor):
  return {"clearBasicFilter": {"sheetId": sheet_id}}

def _poner_filtro(sheet_id, rango, cantidad, valor):
  return {"setBasicFilter": {"filter": {"range": _grid(sheet_id, rango)}}}

def _borrar_fila_col(sheet_id, rango, cantidad, valor):
  """valor = 'ROWS' o 'COLUMNS'. Con una celda individual (p. ej. 'A35') borra desde esa fila/columna, 'cantidad' de ellas."""
  inicio, fin = _posiciones_dimension(rango, valor)
  if ':' not in rango and valor == 'ROWS' and cantidad > 1:
    fin = inicio + cantidad
  return {"deleteDimension": {"range": {"sheetId": sheet_id, "dimension": valor,
                                        "startIndex": inicio, "endIndex": fin}}}

def _nota(sheet_id, rango, cantidad, valor):
  """Nota emergente (comentario de celda)."""
  return {'updateCells': {'range': _grid(sheet_id, rango), 'rows': [{'values': [{'note': valor}]}], 'fields': 'note'}}

def _borrar_notas(sheet_id, rango, cantidad, valor):
  f0, c0, f1, c1 = range_to_indexes(rango)
  return {'updateCells': {'range': _grid(sheet_id, rango), 'fields': 'note',
                          'rows': [{'values': [{'note': ''}] * (c1 - c0)}] * (f1 - f0)}}

def _colorear(sheet_id, rango, cantidad, valor):
  """valor = [rojo, verde, azul, alfa] con componentes de 0 a 1."""
  return {"repeatCell": {"range": _grid(sheet_id, rango),
                         "cell": {"userEnteredFormat": {"backgroundColor": {
                             "red": valor[0], "green": valor[1], "blue": valor[2], 'alpha': valor[3]}}},
                         "fields": "userEnteredFormat.backgroundColor"}}

def _escribir_numeros(sheet_id, rango, cantidad, valor):
  """Escribe la lista de números valor en la fila/rango indicado."""
  return {'updateCells': {'range': _grid(sheet_id, rango),
                          'rows': [{'values': [{'userEnteredValue': {'numberValue': float(v)}} for v in valor]}],
                          'fields': 'userEnteredValue'}}

_CONSTRUCTORES = {
  'insert_row_col': _insertar_fila_col,
  'copy_format': _copiar_formato,
  'delete_cells': _borrar_celdas,
  'update_cells': _actualizar_celdas,
  'clear_filter': _quitar_filtro,
  'set_filter': _poner_filtro,
  'delete_row_col': _borrar_fila_col,
  'emerging_note': _nota,
  'delete_emerging_note': _borrar_notas,
  'color_cells': _colorear,
  'write_numbers_to_range': _escribir_numeros,
}

def solicitud(tipo, sheet_id, rango='', cantidad=0, valor=''):
  """Devuelve la solicitud de la API de Sheets del tipo indicado (ver _CONSTRUCTORES)."""
  constructor = _CONSTRUCTORES.get(tipo)
  if constructor is None:
    print(f'Solicitud ({tipo}) no encontrada')
    return None
  return constructor(sheet_id, rango, cantidad, valor)

def ga(letter,p):
  return f'{letter}{p}:{letter}{p}'

### 3.1b Etapas de la ficha, fechas y duraciones

Una sola definición de la etapa (LECTIVA, PRODUCTIVA, TERMINADA POR FECHA) calculada con la fecha del reporte (D4), las fechas D5/D6 y el estado de Sofia; duraciones en meses y días; y los textos de las notas de A1, D4 y D6.

In [ ]:
import re
import io
import unicodedata
import datetime as _dt
import pandas as pd
import numpy as np
from dateutil.relativedelta import relativedelta
from types import SimpleNamespace

# ------------------------------------------------------------------
# Fechas y duraciones
# ------------------------------------------------------------------
MESES_ABREV = ['ene','feb','mar','abr','may','jun','jul','ago','sep','oct','nov','dic']

def parsear_fecha(valor):
  """datetime.date de un valor de Sofia (texto 'dd/mm/aaaa', 'aaaa/mm/dd' o 'aaaa-mm-dd' con o sin hora, fecha o Timestamp).
  None si el valor está vacío, es '-' o no se puede interpretar."""
  if valor is None:
    return None
  if isinstance(valor, _dt.datetime):
    return valor.date()
  if isinstance(valor, _dt.date):
    return valor
  try:
    if pd.isna(valor):
      return None
  except (TypeError, ValueError):
    pass
  texto = str(valor).strip()
  if not texto or texto == '-':
    return None
  partes = re.split(r'[/\-.]', texto.split()[0])
  if len(partes) != 3 or not all(p.isdigit() for p in partes):
    return None
  a, b, c = (int(p) for p in partes)
  y, m, d = (a, b, c) if len(partes[0]) == 4 else (c, b, a)
  try:
    return _dt.date(y, m, d)
  except ValueError:
    return None

def fecha_corta(fecha):
  """datetime.date -> '26 oct 2026'; 'sin fecha' si no hay fecha."""
  return f"{fecha.day} {MESES_ABREV[fecha.month-1]} {fecha.year}" if fecha else 'sin fecha'

def _unidad(n, singular, plural):
  return f"{n} {singular if n == 1 else plural}"

def duracion_texto(desde, hasta, corto=False):
  """Tiempo entre dos fechas en meses y días calendario. Largo: '18 meses y 10 días'. Corto: '18 m 10 d'."""
  if desde is None or hasta is None:
    return 'sin fecha'
  if hasta < desde:
    desde, hasta = hasta, desde
  rd = relativedelta(hasta, desde)
  meses, dias = rd.years * 12 + rd.months, rd.days
  if corto:
    partes = ([f"{meses} m"] if meses else []) + ([f"{dias} d"] if dias or not meses else [])
    return ' '.join(partes)
  partes = ([_unidad(meses, 'mes', 'meses')] if meses else []) + ([_unidad(dias, 'día', 'días')] if dias or not meses else [])
  return ' y '.join(partes)

def hace_texto(desde, hasta):
  """'hace 18 meses y 10 días' o 'hoy' si es el mismo día."""
  if desde is None or hasta is None:
    return 'sin fecha'
  return 'hoy' if desde == hasta else f"hace {duracion_texto(desde, hasta)}"

# ------------------------------------------------------------------
# Etapas de la ficha (todas se calculan con la fecha del reporte, D4)
# ------------------------------------------------------------------
ETAPA_LECTIVA = 'LECTIVA'
ETAPA_PRODUCTIVA = 'PRODUCTIVA'
ETAPA_FINALIZADA = 'TERMINADA POR FECHA'   # como la llama Sofia Plus

def _sin_tildes(texto):
  return str(texto).upper().translate(str.maketrans('ÁÉÍÓÚ', 'AEIOU'))

def info_ficha(datos_ficha):
  """Etapa de la ficha a la fecha del reporte (D4), con fechas clave y advertencias.
  LECTIVA:    reporte <= fin - 6 meses (D6 - 6 meses).
  PRODUCTIVA: fin - 6 meses < reporte <= fin (D6).
  TERMINADA POR FECHA: reporte > fin (desde el día siguiente a D6).
  Si falta la fecha del reporte o la de fin, valida=False y etapa=None (se usa el criterio anterior de pendientes)."""
  f_reporte = parsear_fecha(datos_ficha.get('fecha_reporte'))
  f_inicio = parsear_fecha(datos_ficha.get('fecha_inicio'))
  f_fin = parsear_fecha(datos_ficha.get('fecha_fin'))
  estado = str(datos_ficha.get('estado_ficha', '')).strip()
  info = SimpleNamespace(valida=False, etapa=None, f_reporte=f_reporte, f_inicio=f_inicio, f_fin=f_fin,
                         f_fin_lectiva=None, estado_sofia=estado, advertencias=[])
  if f_reporte is None or f_fin is None:
    info.advertencias.append(f"No se pudo leer la fecha del reporte ({datos_ficha.get('fecha_reporte')}) o la fecha fin "
                             f"({datos_ficha.get('fecha_fin')}); se usa el criterio de pendientes sin etapas.")
    return info
  info.f_fin_lectiva = f_fin - relativedelta(months=6)
  info.valida = True
  if f_reporte > f_fin:
    info.etapa = ETAPA_FINALIZADA
  elif f_reporte > info.f_fin_lectiva:
    info.etapa = ETAPA_PRODUCTIVA
  else:
    info.etapa = ETAPA_LECTIVA
  estado_norm = _sin_tildes(estado)
  if info.etapa == ETAPA_FINALIZADA and 'POR FECHA' not in estado_norm:
    info.advertencias.append(f"SOFIA Plus reporta la ficha '{estado}', pero la fecha del reporte ({fecha_corta(f_reporte)}) es posterior "
                             f"a la fecha fin ({fecha_corta(f_fin)}); se usa la fecha.")
  elif info.etapa != ETAPA_FINALIZADA and 'POR FECHA' in estado_norm:
    info.advertencias.append(f"SOFIA Plus reporta la ficha '{estado}', pero la fecha del reporte ({fecha_corta(f_reporte)}) no supera "
                             f"la fecha fin ({fecha_corta(f_fin)}); se usa la fecha.")
  return info

def linea_etapa(info):
  """Una línea con la etapa actual y el tiempo que falta (o lleva) en ella."""
  if not info or not info.valida:
    return 'Etapa: sin datos suficientes de fechas'
  if info.etapa == ETAPA_LECTIVA:
    return f"Etapa: LECTIVA · termina el {fecha_corta(info.f_fin_lectiva)}"
  if info.etapa == ETAPA_PRODUCTIVA:
    return f"Etapa: PRODUCTIVA · la ficha termina el {fecha_corta(info.f_fin)}"
  return f"🔒 {ETAPA_FINALIZADA} · la ficha terminó el {fecha_corta(info.f_fin)}"

def linea_corte(f_reporte):
  """Fecha de referencia de todos los tiempos de una nota (fecha del reporte de SOFIA Plus)."""
  return f"📆 Datos al corte: {fecha_corta(f_reporte)}" if f_reporte else ''


# ------------------------------------------------------------------
# Texto: nombres, correos y RAPs
# ------------------------------------------------------------------
_CONECTORES = {'de', 'del', 'la', 'las', 'los', 'y', 'e', 'en', 'el', 'para', 'a'}

def titulo_es(texto):
  """'CENTRO DE TECNOLOGÍA DE LA MANUFACTURA' -> 'Centro de Tecnología de la Manufactura'."""
  palabras = str(texto).strip().rstrip('.').lower().split()
  return ' '.join(p if (i > 0 and p in _CONECTORES) else p.capitalize() for i, p in enumerate(palabras))

def nombre_completo(texto):
  """Nombre completo con formato de nombre propio ('JUAN CARLOS DE LA CRUZ' -> 'Juan Carlos de la Cruz')."""
  palabras = str(texto).strip().title().split()
  return ' '.join(p.lower() if (i > 0 and p.lower() in {'de', 'del', 'la', 'las', 'los', 'y'}) else p for i, p in enumerate(palabras))

# --- Correos de los instructores (pestaña CORREOS_HOJA de la carpeta de plantillas) ---
CORREOS_POR_DOC, CORREOS_POR_NOMBRE = {}, {}
CORREOS_ESTADO = SimpleNamespace(cargado=False, archivo=None, filas=0, error='')
DOC_INSTRUCTOR = {}        # clave de nombre -> documento (de 'CC 123 - NOMBRE' en Sofia)
CORREOS_FALTANTES = {}     # clave de nombre -> (documento, nombre) de instructores sin correo registrado
_CONECTORES_NOMBRE = {'DE', 'DEL', 'LA', 'LAS', 'LOS', 'Y'}

def clave_nombre(texto):
  """Nombre comparable: sin tildes ni mayúsculas, sin conectores y con las palabras ordenadas (Sofia escribe apellidos y nombres en otro orden)."""
  t = unicodedata.normalize('NFKD', str(texto)).encode('ascii', 'ignore').decode().upper()
  palabras = [p for p in re.split(r'[^A-Z0-9]+', t) if p and p not in _CONECTORES_NOMBRE]
  return ' '.join(sorted(palabras))

def solo_documento(valor):
  """'103758917.0' / 103758917.0 / 'CC 103.758.917' -> '103758917'."""
  t = re.sub(r'\.0+$', '', str(valor).strip())
  return re.sub(r'\D', '', t)

def _texto_celda(v):
  t = str(v).strip()
  return '' if t.lower() in ('nan', 'none', 'nat') else t

def indexar_correos(filas):
  """Llena CORREOS_POR_DOC / CORREOS_POR_NOMBRE desde las filas de la pestaña (encabezado: NUMERO DOC, NOMBRES, APELLIDOS, CORREO SENA, CORREO OTRO)."""
  CORREOS_POR_DOC.clear(); CORREOS_POR_NOMBRE.clear()
  clave = lambda t: unicodedata.normalize('NFKD', _texto_celda(t)).encode('ascii', 'ignore').decode().upper().strip()
  inicio = next((k for k, f in enumerate(filas) if 'CORREO SENA' in [clave(c) for c in f]), None)
  if inicio is None:
    raise ValueError("la pestaña no tiene la columna 'CORREO SENA'")
  col = {clave(c): k for k, c in enumerate(filas[inicio])}
  def dato(fila, nombre):
    k = col.get(nombre)
    return _texto_celda(fila[k]) if (k is not None and k < len(fila)) else ''
  n = 0
  for fila in filas[inicio + 1:]:
    correo = (dato(fila, 'CORREO SENA') or dato(fila, 'CORREO OTRO')).lower()
    if not correo:
      continue
    n += 1
    doc = solo_documento(dato(fila, 'NUMERO DOC'))
    if doc:
      CORREOS_POR_DOC[doc] = correo
    nombre = clave_nombre(f"{dato(fila, 'NOMBRES')} {dato(fila, 'APELLIDOS')}")
    if nombre:
      CORREOS_POR_NOMBRE.setdefault(nombre, correo)
  CORREOS_ESTADO.filas = n

def cargar_correos_instructores():
  """Busca en la carpeta de plantillas el archivo que tenga la pestaña CORREOS_HOJA (no importa su nombre) y carga los correos.
  Si no está, lo deja registrado para el LOG y el proceso continúa sin correos. Se ejecuta una sola vez por sesión."""
  if CORREOS_ESTADO.cargado:
    return
  CORREOS_ESTADO.cargado = True
  try:
    carpeta = obtener_o_vincular_carpeta_plantillas()
    consulta = (f"'{carpeta}' in parents and trashed = false and (mimeType='application/vnd.google-apps.spreadsheet' "
                "or mimeType='application/vnd.openxmlformats-officedocument.spreadsheetml.sheet')")
    archivos = drive_service.files().list(q=consulta, fields="files(id, name, mimeType)").execute().get('files', [])
    for a in archivos:
      try:
        filas = None
        if a['mimeType'] == 'application/vnd.google-apps.spreadsheet':
          meta = sheets_service.spreadsheets().get(spreadsheetId=a['id'], fields='sheets.properties.title').execute()
          if any(s.get('properties', {}).get('title') == CORREOS_HOJA for s in meta.get('sheets', [])):
            filas = sheets_service.spreadsheets().values().get(spreadsheetId=a['id'], range=f"'{CORREOS_HOJA}'").execute().get('values', [])
        else:
          libro = pd.ExcelFile(io.BytesIO(drive_service.files().get_media(fileId=a['id']).execute()))
          if CORREOS_HOJA in libro.sheet_names:
            filas = libro.parse(CORREOS_HOJA, header=None, dtype=object).fillna('').values.tolist()
        if filas is not None:
          indexar_correos(filas)
          CORREOS_ESTADO.archivo = a['name']
          CORREOS_ESTADO.error = ''
          print(f"✉️ Correos de instructores: {CORREOS_ESTADO.filas} registros en '{a['name']}' (pestaña '{CORREOS_HOJA}')")
          return
      except Exception as e:
        CORREOS_ESTADO.error = f"'{a.get('name')}': {e}"
  except Exception as e:
    CORREOS_ESTADO.error = str(e)

def correo_de(nombre):
  """Correo del instructor: primero CORREOS_INSTRUCTORES (manual); luego por documento (NUMERO DOC -> CORREO SENA) y, si no
  está, por nombres y apellidos. Si no se encuentra, queda en CORREOS_FALTANTES para el LOG. '' si no hay correo."""
  buscado = str(nombre).strip().upper()
  for clave, correo in CORREOS_INSTRUCTORES.items():
    if str(clave).strip().upper() == buscado:
      return str(correo).strip()
  cargar_correos_instructores()
  if not CORREOS_ESTADO.archivo:
    return ''
  k = clave_nombre(nombre)
  doc = DOC_INSTRUCTOR.get(k, '')
  correo = (CORREOS_POR_DOC.get(doc, '') if doc else '') or CORREOS_POR_NOMBRE.get(k, '')
  if not correo:
    CORREOS_FALTANTES[k] = (doc, nombre_completo(nombre))
  return correo

def etiqueta_instructor(nombre, con_correo=False):
  """'Nombre Completo' ('Sin evaluador' si nadie aprobó el RAP); con_correo=True agrega ' ✉️ correo' si se encuentra."""
  if not nombre:
    return 'Sin evaluador'
  correo = correo_de(nombre) if con_correo else ''
  return nombre_completo(nombre) + (f" ✉️ {correo}" if correo else '')

def log_correos_instructores():
  """LOG final de correos: archivo no encontrado o instructores sin correo (para actualizar la pestaña CORREOS_HOJA)."""
  from IPython.display import display, Markdown
  if not CORREOS_ESTADO.cargado:
    return
  if not CORREOS_ESTADO.archivo:
    motivo = f" ({CORREOS_ESTADO.error})" if CORREOS_ESTADO.error else ''
    display(Markdown(f"ℹ️ **Correos:** no se encontró en la carpeta de plantillas un archivo con la pestaña `{CORREOS_HOJA}`{motivo}. "
                     "Las notas se generaron sin correos."))
  elif CORREOS_FALTANTES:
    filas = "\n".join(f"- {doc or 'sin documento'} · {nombre}" for doc, nombre in sorted(CORREOS_FALTANTES.values(), key=lambda x: x[1]))
    display(Markdown(f"⚠️ **Instructores sin correo en `{CORREOS_HOJA}`** (actualiza el archivo '{CORREOS_ESTADO.archivo}'):\n\n{filas}"))
  CORREOS_FALTANTES.clear()

def rangos_texto(numeros):
  """[5, 6, 7, 8, 31] -> '5-8,31' (sin espacios)."""
  numeros = sorted(set(numeros))
  tramos, i = [], 0
  while i < len(numeros):
    j = i
    while j + 1 < len(numeros) and numeros[j + 1] == numeros[j] + 1:
      j += 1
    tramos.append(str(numeros[i]) if i == j else (f"{numeros[i]},{numeros[j]}" if j == i + 1 else f"{numeros[i]}-{numeros[j]}"))
    i = j + 1
  return ','.join(tramos)

def posiciones_texto(posiciones, limite_lista=12):
  """Posiciones de aprendices: '#4, #19' o, si son muchas, '#1-5,7,9-12'."""
  posiciones = sorted(set(posiciones))
  if len(posiciones) <= limite_lista:
    return ', '.join(f"#{p}" for p in posiciones)
  return '#' + rangos_texto(posiciones)

def es_rap_productivo(competencia):
  """True si la competencia es la de Etapa Productiva ('2 - RESULTADOS DE APRENDIZAJE ETAPA PRACTICA')."""
  texto = _sin_tildes(competencia)
  return 'RESULTADOS DE APRENDIZAJE ETAPA' in texto and ('PRACTICA' in texto or 'PRODUCTIVA' in texto)

def partes_rap(texto):
  """(código, nombre) de un RAP de Sofia: '1671 - 2. ANALIZAR ...' -> ('1671', 'ANALIZAR ...')."""
  texto = str(texto).strip()
  if texto.upper().startswith('RAP '):
    texto = texto[4:].strip()
  codigo, _, nombre = texto.partition(' - ')
  if not nombre:
    codigo, nombre = '', texto
  nombre = re.sub(r'^\s*\d+\s*[-.]?\s*', '', nombre).strip()
  return codigo.strip(), nombre

def rap_corto(texto, largo=None):
  """(código, nombre corto) del RAP; el nombre se recorta a 'largo' caracteres (RAP_NOMBRE_LARGO por defecto)."""
  largo = RAP_NOMBRE_LARGO if largo is None else largo
  codigo, nombre = partes_rap(texto)
  nombre = nombre.capitalize()
  if len(nombre) > largo:
    nombre = nombre[:largo].rstrip() + '…'
  return codigo, nombre

def recortar_nota(texto, maximo=None):
  """Garantiza que la nota no pase del máximo de caracteres (corta por líneas completas)."""
  maximo = NOTA_MAX_CARACTERES if maximo is None else maximo
  if len(texto) <= maximo:
    return texto
  aviso = "\n… (nota recortada por tamaño)"
  corte = texto[:maximo - len(aviso)]
  return corte[:corte.rfind('\n')] + aviso

# ------------------------------------------------------------------
# Notas del encabezado: A1 (programa), D4 (reporte) y D6 (fin de la ficha)
# ------------------------------------------------------------------
def _codigo_y_nombre(texto):
  """'5 - REGIONAL ANTIOQUIA' -> ('5', 'REGIONAL ANTIOQUIA')."""
  codigo, _, nombre = str(texto).partition(' - ')
  return (codigo.strip(), nombre.strip()) if nombre else ('', str(texto).strip())

def nota_programa(datos_ficha, n_raps, n_productivos=0):
  """Nota de A1: identidad del programa y de la ficha, sin repetir fechas ni conteos de otras celdas."""
  cod_regional, regional = _codigo_y_nombre(datos_ficha['regional'])
  regional = re.sub(r'^REGIONAL\s+', '', regional.strip(), flags=re.IGNORECASE)
  cod_centro, centro = _codigo_y_nombre(datos_ficha['centro'])
  raya = '─' * 30
  lineas = ['🎓 PROGRAMA DE FORMACIÓN',
            '═' * 30,
            f"🔖 Código {datos_ficha['codigo']} · Versión {datos_ficha['version']}",
            f"🧭 Modalidad: {titulo_es(datos_ficha['modalidad'])}",
            f"📌 Estado de la ficha: {datos_ficha['estado_ficha']}",
            raya,
            f"🌎 Regional: {titulo_es(regional)}" + (f" ({cod_regional})" if cod_regional else ''),
            f"🏫 Centro: {titulo_es(centro)}" + (f" ({cod_centro})" if cod_centro else ''),
            raya,
            f"📚 {n_raps} RAPs" + (f" ({n_productivos} de Etapa Productiva)" if n_productivos else '')]
  return '\n'.join(lineas)

def nota_fecha_reporte(info, gtime, ultimo_juicio=None, reporte_anterior=None, corte_lineas=()):
  """Nota de D4: vigencia del reporte de Sofia y posición de hoy en el calendario de la ficha."""
  f_rep = info.f_reporte
  procesado = gtime.date()
  if f_rep is None:
    cuando = ''
  elif procesado == f_rep:
    cuando = ' (el mismo día)'
  elif procesado > f_rep:
    cuando = f" ({duracion_texto(f_rep, procesado)} después)"
  else:
    cuando = ''
  lineas = ['📅 REPORTE DE SOFIA PLUS',
            f"📆 Datos al corte (fecha del reporte): {fecha_corta(f_rep)}",
            f"Procesado: {fecha_corta(procesado)}, {gtime.strftime('%H:%M:%S')}{cuando}"]
  if f_rep is not None and (procesado - f_rep).days >= 2:
    lineas.append(f"⚠️ Reporte con {duracion_texto(f_rep, procesado)} de antigüedad al procesarse")
  if ultimo_juicio:
    lineas.append(f"Último juicio registrado: {fecha_corta(ultimo_juicio)}")
  lineas.extend(corte_lineas)
  lineas.extend(f"⚠️ {a}" for a in getattr(info, 'avisos_fechas', []))
  if reporte_anterior:   # (fecha de la hoja, celdas conservadas)
    f_hoja, n_conservadas = reporte_anterior
    lineas.append(f"⚠️ REPORTE ANTERIOR: la hoja tenía datos del {fecha_corta(f_hoja)}. "
                  f"Los {n_conservadas} juicios más recientes que no trae este reporte se conservan en gris; "
                  "las estadísticas son solo de este reporte.")
  lineas.append('─' * 35)
  if info.valida and info.etapa != ETAPA_FINALIZADA and info.f_inicio and info.f_fin > info.f_inicio:
    total = (info.f_fin - info.f_inicio).days
    pasado = max(0, min(total, (f_rep - info.f_inicio).days))
    lineas.append(f"Tiempo de la ficha al corte: {round(pasado / total * 100)}% transcurrido "
                  f"({duracion_texto(info.f_inicio, f_rep)} de {duracion_texto(info.f_inicio, info.f_fin)})")
  lineas.append(linea_etapa(info))
  return '\n'.join(lineas)

def nota_fecha_fin(info):
  """Nota de D6: estado de la ficha en SOFIA Plus y fin de la etapa lectiva."""
  lineas = [f"Estado en SOFIA Plus: {info.estado_sofia}"]
  if info.valida:
    lineas.append(f"Fin lectiva: {fecha_corta(info.f_fin_lectiva)}")
  return '\n'.join(lineas)

def serial_a_datetime(numero):
  """Número de serie de Google Sheets/Excel (días desde 1899-12-30) -> datetime redondeado al segundo."""
  return (_dt.datetime(1899, 12, 30) + _dt.timedelta(days=float(numero), seconds=0.5)).replace(microsecond=0)

def es_numero(valor):
  return isinstance(valor, (int, float)) and not isinstance(valor, bool)

def revisar_fechas_juicios(datos_ficha):
  """Avisos sobre fechas de juicios que no se pueden leer o que no tienen sentido (señal de que la fecha se interpretó mal):
  ilegibles, anteriores al inicio de la ficha o posteriores a la fecha del reporte."""
  col = 'Fecha y Hora del Juicio Evaluativo'
  textos = [str(v).strip() for v in datos_ficha['juicios'][col].dropna() if str(v).strip() not in ('', '-')]
  fechas = [parsear_fecha(t) for t in textos]
  f_ini, f_rep = parsear_fecha(datos_ficha.get('fecha_inicio')), parsear_fecha(datos_ficha.get('fecha_reporte'))
  avisos = []
  n_mal = sum(1 for f in fechas if f is None)
  if n_mal:
    avisos.append(f"{n_mal} fechas de juicios no se pudieron leer (ejemplo: {next(t for t, f in zip(textos, fechas) if f is None)!r}).")
  n_antes = sum(1 for f in fechas if f is not None and f_ini and f < f_ini)
  if n_antes:
    avisos.append(f"{n_antes} juicios tienen fecha anterior al inicio de la ficha ({fecha_corta(f_ini)}): revisa el formato de fecha del archivo.")
  n_despues = sum(1 for f in fechas if f is not None and f_rep and f > f_rep)
  if n_despues:
    avisos.append(f"{n_despues} juicios tienen fecha posterior a la fecha del reporte ({fecha_corta(f_rep)}): revisa el formato de fecha del archivo.")
  return avisos

# --- Corte: procesar el reporte solo hasta una fecha y hora ---
CORTE_ACTUAL = None   # datetime (hora de Bogotá) o None; lo fija la interfaz en cada tabulación

def fecha_hora_de(valor):
  """datetime de un valor de Sofia ('2025/11/26 14:15:00' o '26/11/2025 14:15'); None si no se puede interpretar. Sin hora: 00:00."""
  fecha = parsear_fecha(valor)
  if fecha is None:
    return None
  m = re.search(r'(\d{1,2}):(\d{2})(?::(\d{2}))?', str(valor))
  h, mi, se = (int(m.group(1)), int(m.group(2)), int(m.group(3) or 0)) if m else (0, 0, 0)
  try:
    return _dt.datetime(fecha.year, fecha.month, fecha.day, h, mi, se)
  except ValueError:
    return None

def construir_corte(fecha, hora_texto):
  """datetime del corte a partir del selector de fecha y el texto 'HH:MM'. ValueError con un mensaje claro si algo falta o es inválido."""
  if fecha is None:
    raise ValueError("Elige la fecha del corte (o desmarca 'Procesar solo hasta').")
  m = re.fullmatch(r'\s*(\d{1,2}):(\d{2})\s*', str(hora_texto))
  if not m or int(m.group(1)) > 23 or int(m.group(2)) > 59:
    raise ValueError("La hora del corte debe tener el formato HH:MM, por ejemplo 18:30.")
  return _dt.datetime(fecha.year, fecha.month, fecha.day, int(m.group(1)), int(m.group(2)))

def aplicar_corte(datos_ficha, corte):
  """Procesa el reporte como si se hubiera descargado en 'corte': los juicios registrados después quedan POR EVALUAR sin funcionario ni fecha
  y la fecha del reporte (D4) pasa a ser la del corte. Devuelve un texto de aviso. El estado de los aprendices es el que trae el reporte."""
  df = datos_ficha['juicios']
  f_rep = parsear_fecha(datos_ficha.get('fecha_reporte'))
  datos_ficha['corte'] = None
  if f_rep is not None and corte.date() > f_rep:
    return f"el corte ({fecha_corta(corte.date())}) es posterior a la fecha del reporte ({fecha_corta(f_rep)}): no tiene efecto y se usa el reporte completo."
  col_f, col_func, col_j = 'Fecha y Hora del Juicio Evaluativo', 'Funcionario que registro el juicio evaluativo', 'Juicio de Evaluación'
  momentos = [fecha_hora_de(v) for v in df[col_f]]   # lista (no Series): pandas convertiría None en NaT
  posteriores = pd.Series([m is not None and m > corte for m in momentos], index=df.index)
  con_juicio = pd.Series([m is not None for m in momentos], index=df.index)
  if con_juicio.any() and int(posteriores.sum()) == int(con_juicio.sum()):
    raise ValueError(f"Ficha {datos_ficha.get('ficha', '')}: el corte ({corte.strftime('%d/%m/%Y %H:%M')}) es anterior al primer juicio del reporte; "
                     "la hoja quedaría sin juicios. Ajusta el corte.")
  df.loc[posteriores, col_j] = 'POR EVALUAR'
  df.loc[posteriores, col_func] = '  -   '
  df.loc[posteriores, col_f] = np.nan
  datos_ficha['fecha_reporte_original'] = datos_ficha.get('fecha_reporte')
  datos_ficha['fecha_reporte'] = f"{corte.day:02d}/{corte.month:02d}/{corte.year}"
  datos_ficha['corte'] = corte
  datos_ficha['juicios_posteriores'] = int(posteriores.sum())
  return ''

def lineas_corte(datos_ficha):
  """Líneas de nota que explican el corte (vacío si no se usó)."""
  corte = datos_ficha.get('corte')
  if corte is None:
    return []
  original = parsear_fecha(datos_ficha.get('fecha_reporte_original'))
  return [f"✂️ CORTE: procesado hasta el {fecha_corta(corte.date())}, {corte.strftime('%H:%M')} (el reporte original es del {fecha_corta(original)}).",
          f"{datos_ficha.get('juicios_posteriores', 0)} juicios posteriores no se tuvieron en cuenta. El estado de los aprendices es el del reporte original."]

def etapa_de_carpeta(fecha_fin_texto, fecha_reporte_texto=None):
  """ETAPA_LECTIVA o ETAPA_PRODUCTIVA para elegir la carpeta de salida: lectiva mientras la fecha del reporte no supere
  fin - 6 meses (D6 - 6 meses). Sin fecha de reporte legible usa la fecha de hoy."""
  f_fin = parsear_fecha(fecha_fin_texto)
  f_rep = parsear_fecha(fecha_reporte_texto) or _dt.date.today()
  if CORTE_ACTUAL is not None and CORTE_ACTUAL.date() < f_rep:   # con corte, la carpeta sigue la fecha de corte
    f_rep = CORTE_ACTUAL.date()
  if f_fin is None:
    raise ValueError(f"Fecha fin ilegible: {fecha_fin_texto!r}")
  return ETAPA_LECTIVA if f_rep <= f_fin - relativedelta(months=6) else ETAPA_PRODUCTIVA


### 3.2 Lectura de Sofia Plus, órdenes de RAPs y plantillas

In [ ]:
import io
import re
from googleapiclient.http import MediaFileUpload

def listar_gsheets(id_carpeta):
  """Hojas de cálculo de Google y archivos Excel (id, name, mimeType) en una carpeta de Drive."""
  query = f"'{id_carpeta}' in parents and (mimeType='application/vnd.google-apps.spreadsheet' or mimeType='application/vnd.ms-excel' or mimeType='application/vnd.openxmlformats-officedocument.spreadsheetml.sheet') and trashed = false"
  resultado = drive_service.files().list(
    q=query,
    fields="nextPageToken, files(id, name, mimeType)"
  ).execute()
  return resultado.get('files', [])

def tabla_juicios(valores):
  """DataFrame con la tabla de juicios a partir de todas las filas del reporte de Sofia."""
  df = pd.DataFrame(valores[FILA_TABLA_SOFIA:])
  df.columns = df.iloc[0]
  df = df[1:]
  return df.reset_index(drop=True)

def fechas_de_hoja_sin_ambiguedad(valores, crudos):
  """Una hoja de Google muestra las fechas según su configuración regional ('1/10/2026' puede ser 1 de octubre o 10 de enero).
  Se leen también los valores sin formato (número de serie) y se reemplazan las fechas por texto inequívoco:
  encabezado 'dd/mm/aaaa' y fecha de juicios 'aaaa-mm-dd hh:mm:ss' (igual que un Excel de Sofia)."""
  valores = [list(f) for f in valores]
  def poner(r, c, texto):
    if r < len(valores):
      valores[r] += [''] * (c + 1 - len(valores[r]))
      valores[r][c] = texto
  def cruda(r, c):
    return crudos[r][c] if r < len(crudos) and c < len(crudos[r]) else ''
  for clave in ('fecha_reporte', 'fecha_inicio', 'fecha_fin'):
    r, c = FILAS_ENCABEZADO_SOFIA[clave], COLUMNA_ENCABEZADO_SOFIA
    if es_numero(cruda(r, c)):
      poner(r, c, serial_a_datetime(cruda(r, c)).strftime('%d/%m/%Y'))
  if len(valores) > FILA_TABLA_SOFIA and 'Fecha y Hora del Juicio Evaluativo' in valores[FILA_TABLA_SOFIA]:
    c = valores[FILA_TABLA_SOFIA].index('Fecha y Hora del Juicio Evaluativo')
    for r in range(FILA_TABLA_SOFIA + 1, len(valores)):
      if es_numero(cruda(r, c)):
        poner(r, c, serial_a_datetime(cruda(r, c)).strftime('%Y-%m-%d %H:%M:%S'))
  return valores

def encabezado_desde_hoja(valores):
  """Datos del encabezado (ficha, programa, fechas...) de un reporte leído de una hoja de Google."""
  e = {k: valores[fila][COLUMNA_ENCABEZADO_SOFIA] for k, fila in FILAS_ENCABEZADO_SOFIA.items()}
  e['ficha'] = e['ficha'][0:7]
  return e

def encabezado_desde_excel(df_encabezado):
  """Datos del encabezado de un reporte leído de un Excel (las fechas llegan como fecha y se pasan a dd/mm/aaaa)."""
  e = {k: df_encabezado.iloc[fila, COLUMNA_ENCABEZADO_SOFIA] for k, fila in FILAS_ENCABEZADO_SOFIA.items()}
  e['ficha'] = str(e['ficha'])
  if isinstance(e['fecha_inicio'], (datetime.date, datetime.datetime)):
    e['fecha_inicio'] = e['fecha_inicio'].strftime('%d/%m/%Y')
  if isinstance(e['fecha_fin'], (datetime.date, datetime.datetime)):
    e['fecha_fin'] = e['fecha_fin'].strftime('%d/%m/%Y')
  return e

def asignar_orden(ra_texto, ra_to_orden):
  """Orden (N) del primer RA del diccionario ra_to_orden contenido en el texto; None si no hay coincidencia."""
  for ra_key in ra_to_orden:
    if ra_key in ra_texto:
      return ra_to_orden[ra_key]
  return None

def registrar_ficha(data_fichas, ordenRA, list_competencias, encabezado, juicios, fecha_con_guiones=False):
  """Procesa los juicios de una ficha (orden de RAPs, estados, competencias) y la guarda en data_fichas."""

  # --- VALIDACIÓN DE SEGURIDAD Y FORMATO DEL ARCHIVO ---
  juicios.columns = juicios.columns.astype(str).str.strip()

  columnas_requeridas = [
      'Resultado de Aprendizaje',
      'Juicio de Evaluación',
      'Número de Documento',
      'Tipo de Documento',
      'Nombre',
      'Apellidos',
      'Estado',
      'Fecha y Hora del Juicio Evaluativo'
  ]

  columnas_faltantes = [col for col in columnas_requeridas if col not in juicios.columns]
  if columnas_faltantes:
      ficha_err = encabezado.get('ficha', 'Desconocida')
      raise ValueError(
          f"El archivo no corresponde a un reporte válido de Sofia Plus (Faltan columnas obligatorias: {columnas_faltantes})."
      )
  # -----------------------------------------------------

  ficha = encabezado['ficha']
  codigo_programa = encabezado['codigo']
  datos = {k: v for k, v in encabezado.items() if k != 'ficha'}
  datos['juicios'] = juicios
  data_fichas[ficha] = datos
  dfRA = juicios
  auxRAs = extraer_ra_unicos(dfRA)
  n_RAs = len(auxRAs)
  if codigo_programa not in list(ordenRA.keys()):
    auxdf = pd.DataFrame(columns=['N', 'RA', 'Competencias'])
    auxdf['RA'] = auxRAs
    auxdf['N'] = range(n_RAs)
    auxdf['N'] = auxdf['N'].astype('int64')
    ordenRA[codigo_programa] = {}
    ordenRA[codigo_programa][str(n_RAs)] = {}
    ordenRA[codigo_programa][str(n_RAs)]['RAs'] = auxdf
  if fecha_con_guiones:
    dfRA['Fecha y Hora del Juicio Evaluativo'] = dfRA['Fecha y Hora del Juicio Evaluativo'].str.replace('-', '/')
  dfRA.insert(dfRA.columns.get_loc('Resultado de Aprendizaje') + 1, 'Orden', value=None)

  RAs_ordenados = ordenRA[codigo_programa][str(n_RAs)]['RAs']
  ra_to_orden = dict(zip(RAs_ordenados['RA'], RAs_ordenados['N']))
  dfRA['Orden'] = dfRA['Resultado de Aprendizaje'].apply(asignar_orden, args=(ra_to_orden,))

  dfRA = dfRA.sort_values(by=['Apellidos','Nombre','Número de Documento'])
  dfRA['Orden'] = pd.to_numeric(dfRA['Orden'], errors='coerce').fillna(0).astype(int)
  datos['cantidad_aprendices'] = dfRA['Número de Documento'].nunique()
  datos['cantidad_RAs'] = dfRA['Resultado de Aprendizaje'].nunique()

  dfRA['Estado'] = pd.Categorical(dfRA['Estado'], categories=ORDEN_ESTADOS, ordered=True)
  dfRA = dfRA.sort_values(by=['Estado', 'Apellidos','Nombre','Número de Documento','Orden'])
  dfRA = dfRA.reset_index(drop=True)
  datos['juicios'] = dfRA

  mapeo = dfRA.set_index('Resultado de Aprendizaje')['Competencia'].to_dict()
  RAs_ordenados['Competencias'] = RAs_ordenados['RA'].map(mapeo)
  datos['RAs'] = RAs_ordenados['RA']

  list_competencias[codigo_programa] = list(RAs_ordenados['Competencias'].unique())
  list_competencias[codigo_programa].sort()

def ordenar_RAs(id):
  datosRA = gc.open_by_key(id).sheet1
  datosRA = pd.DataFrame(datosRA.get_all_values())
  datosRA.columns = datosRA.iloc[0]
  datosRA = datosRA[1:]
  if 'N' not in datosRA.columns:
    datosRA.insert(0, 'N', range(len(datosRA)))
  datosRA['N'] = datosRA['N'].astype(int)
  datosRA = datosRA.sort_values(by=['N'])
  datosRA = datosRA.reset_index(drop=True)
  datosRA['Competencias'] = ''
  return datosRA

def extraer_ra_unicos(dfRA):
    col_fecha  = 'Fecha y Hora del Juicio Evaluativo'
    col_ra     = 'Resultado de Aprendizaje'
    col_func   = 'Funcionario que registro el juicio evaluativo'
    col_juicio = 'Juicio de Evaluación'
    es_aprobado = dfRA[col_juicio].astype(str).str.strip().str.upper() == 'APROBADO'
    ap = dfRA.loc[es_aprobado, [col_fecha, col_ra, col_func]].copy()
    ap[col_fecha] = pd.to_datetime(ap[col_fecha], errors='coerce').dt.normalize()
    ra = ap.sort_values([col_fecha, col_ra, col_func])[col_ra]
    bloques = ra.groupby((ra != ra.shift()).cumsum()).agg(RA='first', filas='size')
    idx = bloques.groupby('RA')['filas'].idxmax().sort_values()
    lista_ra = bloques.loc[idx, 'RA'].tolist()
    return list(dict.fromkeys(lista_ra + dfRA.loc[~es_aprobado, col_ra].dropna().tolist()))

def obtener_o_vincular_carpeta_orden():
  """Busca dinámicamente la carpeta genérica de Orden de Juicios por appProperties en Drive, o la crea."""
  global carpeta_raiz_instalacion_id
  if 'carpeta_raiz_instalacion_id' not in globals() or not carpeta_raiz_instalacion_id:
      carpeta_raiz_instalacion_id = obtener_o_vincular_raiz_instalacion()

  query = "appProperties has { key='es_carpeta_orden_rap' and value='true' } and mimeType='application/vnd.google-apps.folder' and trashed = false"
  search_result = drive_service.files().list(q=query, fields="files(id, name)").execute()
  folders = search_result.get('files', [])

  if folders:
    return unica_o_conflicto(folders, slot_orden())

  print(f"\n🛠️ Creando automáticamente la carpeta de Orden de RAPs en la Carpeta Raíz de Instalación...")
  try:
    folder_metadata = {
      'name': 'Orden de RAPs (Formatos)',
      'mimeType': 'application/vnd.google-apps.folder',
      'parents': [carpeta_raiz_instalacion_id],
      'appProperties': {
        'es_carpeta_orden_rap': 'true'
      }
    }
    new_folder = drive_service.files().create(body=folder_metadata, fields='id, name').execute()
    print(f"🎉 Carpeta de Órdenes creada: '{new_folder.get('name')}' con ID: {new_folder.get('id')}")
    return new_folder['id']
  except Exception as e:
    raise ValueError(f"❌ Error fatal al crear la carpeta de Orden de RAPs: {str(e)}")

def obtener_o_vincular_carpeta_plantillas():
  """Busca dinámicamente la carpeta genérica de Plantillas de Fichas por appProperties en Drive, o la crea."""
  global carpeta_raiz_instalacion_id
  if 'carpeta_raiz_instalacion_id' not in globals() or not carpeta_raiz_instalacion_id:
      carpeta_raiz_instalacion_id = obtener_o_vincular_raiz_instalacion()

  query = "appProperties has { key='es_carpeta_plantillas_sena' and value='true' } and mimeType='application/vnd.google-apps.folder' and trashed = false"
  search_result = drive_service.files().list(q=query, fields="files(id, name)").execute()
  folders = search_result.get('files', [])

  if folders:
    return unica_o_conflicto(folders, slot_plantillas())

  print(f"\n🛠️ Creando automáticamente la carpeta de Plantillas Oficiales en la Carpeta Raíz de Instalación...")
  try:
    folder_metadata = {
      'name': 'Plantillas de Juicios Evaluativos',
      'mimeType': 'application/vnd.google-apps.folder',
      'parents': [carpeta_raiz_instalacion_id],
      'appProperties': {
        'es_carpeta_plantillas_sena': 'true'
      }
    }
    new_folder = drive_service.files().create(body=folder_metadata, fields='id, name').execute()
    print(f"🎉 Carpeta de Plantillas creada: '{new_folder.get('name')}' con ID: {new_folder.get('id')}")
    return new_folder['id']
  except Exception as e:
    raise ValueError(f"❌ Error fatal al crear la carpeta de Plantillas: {str(e)}")

def cargar_ordenes_ra():
  """Lee los archivos de orden de RAPs desde la carpeta dinámica vinculada."""
  ordenRA = {}
  carpeta_orden_id = obtener_o_vincular_carpeta_orden()
  for item_origen in listar_gsheets(carpeta_orden_id):
    sheet = sheets_service.spreadsheets().get(spreadsheetId=item_origen['id']).execute().get('sheets', '')
    n_RAs = sheet[0]['properties']['gridProperties']['rowCount']-1
    titulo = sheet[0]['properties']['title']
    if titulo not in ordenRA.keys():
      ordenRA[titulo] = {}
    ordenRA[titulo][str(n_RAs)] = {}
    ordenRA[titulo][str(n_RAs)]['RAs'] = ordenar_RAs(item_origen['id'])
  return ordenRA

def extraer_id_drive(texto):
  """Extrae un ID de Google Drive a partir de un texto o una URL."""
  match = re.search(r'[-_a-zA-Z0-9]{25,}', texto)
  return match.group(0) if match else texto.strip()

def obtener_o_vincular_carpeta_procesar(codigo_programa=None, nom_programa=None):
  """Busca dinámicamente la carpeta genérica de Entrada 'Juicios a Procesar' por metadatos (appProperties). Si no existe, se crea."""
  global carpeta_raiz_instalacion_id
  if 'carpeta_raiz_instalacion_id' not in globals() or not carpeta_raiz_instalacion_id:
      carpeta_raiz_instalacion_id = obtener_o_vincular_raiz_instalacion()

  query = "appProperties has { key='juicios_procesar_generica' and value='true' } and mimeType='application/vnd.google-apps.folder' and trashed = false"
  search_result = drive_service.files().list(q=query, fields="files(id, name)").execute()
  folders = search_result.get('files', [])

  if folders:
    return unica_o_conflicto(folders, slot_procesar())

  print(f"\n🛠️ Creando automáticamente la carpeta de Entrada Genérica 'Juicios a Procesar' dentro de la Carpeta Raíz de Instalación...")
  try:
    folder_metadata = {
      'name': 'Juicios a Procesar',
      'mimeType': 'application/vnd.google-apps.folder',
      'parents': [carpeta_raiz_instalacion_id],
      'appProperties': {
        'juicios_procesar_generica': 'true'
      }
    }
    new_folder = drive_service.files().create(body=folder_metadata, fields='id, name').execute()
    print(f"🎉 Carpeta de Entrada creada: '{new_folder.get('name')}' con ID: {new_folder.get('id')}")
    return new_folder['id']
  except Exception as e:
    raise ValueError(f"❌ Error fatal al crear la carpeta de Entrada: {str(e)}")

def cargar_fichas_sofia(ordenRA, list_competencias):
  """Procesa los Juicios Evaluativos de Sofia que están en la carpeta de entrada de Drive de forma tolerante a fallos."""
  data_fichas = {}
  id_juicios = {}

  carpeta_origen_id = obtener_o_vincular_carpeta_procesar()

  for item_origen in listar_gsheets(carpeta_origen_id):
    if item_origen['name'].endswith('Rdo'):
      continue

    nombre_archivo = item_origen['name']
    mime = item_origen.get('mimeType', '')

    try:
      if 'google-apps.spreadsheet' in mime:
        hoja_origen = gc.open_by_key(item_origen['id']).sheet1
        valores = hoja_origen.get_all_values()
        try:   # fechas sin ambigüedad de configuración regional (ver fechas_de_hoja_sin_ambiguedad)
          valores = fechas_de_hoja_sin_ambiguedad(valores, hoja_origen.get_all_values(value_render_option='UNFORMATTED_VALUE'))
        except Exception as e_fechas:
          print(f"⚠️ '{nombre_archivo}': no se pudieron leer las fechas sin formato ({e_fechas}); se usan las fechas como las muestra la hoja.")
        encabezado = encabezado_desde_hoja(valores)
        df_tabla = tabla_juicios(valores)

        # Dry-run para validación del contenido estructurado
        temp_fichas = {}
        temp_competencias = {}
        registrar_ficha(temp_fichas, ordenRA, temp_competencias, encabezado, df_tabla, fecha_con_guiones=True)

        ficha = encabezado['ficha']
        id_juicios[ficha] = item_origen['id']
        data_fichas[ficha] = temp_fichas[ficha]
        list_competencias.update(temp_competencias)
      else:
        request = drive_service.files().get_media(fileId=item_origen['id'])
        file_content = request.execute()
        df_encabezado = pd.read_excel(io.BytesIO(file_content), nrows=FILA_TABLA_SOFIA, header=None)
        df = pd.read_excel(io.BytesIO(file_content), header=FILA_TABLA_SOFIA, dtype=str)
        encabezado = encabezado_desde_excel(df_encabezado)

        # Dry-run para validación del contenido estructurado
        temp_fichas = {}
        temp_competencias = {}
        registrar_ficha(temp_fichas, ordenRA, temp_competencias, encabezado, df, fecha_con_guiones=True)

        ficha = encabezado['ficha']
        id_juicios[ficha] = item_origen['id']
        data_fichas[ficha] = temp_fichas[ficha]
        list_competencias.update(temp_competencias)

    except ValueError as e_val:
      print(f"⚠️ OMITIDO: El archivo '{nombre_archivo}' en Drive no es un reporte válido: {str(e_val)}")
      continue
    except Exception as e_desc:
      print(f"⚠️ OMITIDO: No se pudo procesar el archivo '{nombre_archivo}' en Drive por un error inesperado: {str(e_desc)}")
      continue

  return data_fichas, id_juicios

def cargar_ficha_desde_excel(data_fichas, ordenRA, list_competencias, id_juicios):
  """Pide subir un reporte de Sofia (Excel), lo sube automáticamente a la carpeta de entrada vinculada en Drive y procesa los datos."""
  print("Selecciona el Archivo de Juicios Evaluativos (.xlsx) de tu computadora\n")
  subidos = files.upload()
  if not subidos:
    print("❌ No se cargó ningún archivo.")
    return
  nombre_archivo_local = list(subidos.keys())[0]

  try:
    df_encabezado = pd.read_excel(nombre_archivo_local, nrows=FILA_TABLA_SOFIA, header=None)
    df = pd.read_excel(nombre_archivo_local, header=FILA_TABLA_SOFIA, dtype=str)
    encabezado = encabezado_desde_excel(df_encabezado)
  except Exception as e:
    print(f"❌ Error al leer el archivo Excel: {e}")
    return

  carpeta_origen_id = obtener_o_vincular_carpeta_procesar()

  try:
    print(f"Subiendo {nombre_archivo_local} a la carpeta de entrada de Google Drive...")
    file_metadata = {
        'name': nombre_archivo_local,
        'parents': [carpeta_origen_id]
    }
    media = MediaFileUpload(nombre_archivo_local, mimetype='application/vnd.openxmlformats-officedocument.spreadsheetml.sheet', resumable=True)
    uploaded_file = drive_service.files().create(body=file_metadata, media_body=media, fields='id').execute()
    file_id_drive = uploaded_file.get('id')

    id_juicios[encabezado['ficha']] = file_id_drive
    print(f"✨ Archivo subido temporalmente a Drive con ID: {file_id_drive}")
  except Exception as e:
    print(f"⚠️ No se pudo sincronizar el archivo directamente con Google Drive: {str(e)}")

  registrar_ficha(data_fichas, ordenRA, list_competencias, encabezado, df, fecha_con_guiones=True)

def extraer_iniciales(frase, palabras_dobles=None):
    import re
    if palabras_dobles is None:
        palabras_dobles = {'software'}
    excluir = {
        'de', 'del', 'la', 'el', 'los', 'las', 'y', 'e', 'o', 'u',
        'en', 'a', 'con', 'por', 'para', 'sin', 'sobre', 'entre'
    }
    frase_limpia = re.sub(r'\([^)]*\)', '', frase)
    iniciales = []
    for palabra in frase_limpia.split():
        palabra_limpia = palabra.strip(',.;:')
        if palabra_limpia.lower() not in excluir and palabra_limpia:
            if palabra_limpia.lower() in palabras_dobles and len(palabra_limpia) >= 2:
                iniciales.append(palabra_limpia[:2].upper())
            else:
                iniciales.append(palabra_limpia[0].upper())
    return ''.join(iniciales)

def mapear_plantillas():
  ids_plantillas = {}
  carpeta_plantillas_id = obtener_o_vincular_carpeta_plantillas()
  for item in listar_gsheets(carpeta_plantillas_id):
    sheet = sheets_service.spreadsheets().get(spreadsheetId=item['id']).execute().get('sheets', '')
    if sheet[0]['properties']['title'] not in ids_plantillas.keys():
      ids_plantillas[sheet[0]['properties']['title']] = {}
    ids_plantillas[sheet[0]['properties']['title']][str(sheet[0]['properties']['gridProperties']['columnCount']-7)] = item['id']
  return ids_plantillas

def nombre_carpeta_etapa(nom_programa, fecha_fin_texto, fecha_reporte_texto=None):
    """Nombre lógico de la carpeta de salida: 'PROG-Lectiva' o 'PROG-Productiva'.
    La etapa se decide con la fecha del reporte (D4): Lectiva mientras no supere la fecha fin menos 6 meses (D6 - 6 meses).
    Sin fecha de reporte legible usa la fecha de hoy."""
    return f"{nom_programa}-Lectiva" if etapa_de_carpeta(fecha_fin_texto, fecha_reporte_texto) == ETAPA_LECTIVA else f"{nom_programa}-Productiva"

def candidatas_carpeta_etapa(parent_folder_id, subfolder_name):
    """([carpetas], 'propiedad'|'nombre'|None). Primero por identidad lógica (propiedad: sobrevive a renombrar y mover);
    solo si no hay ninguna, por nombre exacto bajo parent_folder_id, para carpetas creadas antes de existir la propiedad."""
    por_propiedad = buscar_carpetas_slot(slot_etapa(subfolder_name))
    if por_propiedad:
        return por_propiedad, 'propiedad'
    query = f"'{parent_folder_id}' in parents and name='{subfolder_name}' and mimeType='application/vnd.google-apps.folder' and trashed = false"
    por_nombre = drive_service.files().list(q=query, fields="files(id, name, parents, modifiedTime)").execute().get('files', [])
    return por_nombre, ('nombre' if por_nombre else None)

def nombre_etapa_contraria(nombre_etapa):
    """'MEII-Lectiva' ↔ 'MEII-Productiva'."""
    if nombre_etapa.endswith('-Lectiva'):
        return nombre_etapa[:-len('-Lectiva')] + '-Productiva'
    return nombre_etapa[:-len('-Productiva')] + '-Lectiva'

def buscar_subcarpeta_etapa(parent_folder_id, subfolder_name):
    """ID de la carpeta de etapa si existe (sin crearla); None si no existe; ConflictoCarpeta si hay varias."""
    slot = slot_etapa(subfolder_name)
    folders, via = candidatas_carpeta_etapa(parent_folder_id, subfolder_name)
    if not folders:
        return None
    folder_id = unica_o_conflicto(folders, slot)
    if via == 'nombre':
        # carpeta anterior sin propiedad: se marca (solo metadato) para que luego se reconozca aunque la renombren o muevan
        try:
            drive_service.files().update(fileId=folder_id, body={'appProperties': dict(slot.activar)}, fields='id').execute()
        except Exception:
            pass
    return folder_id

def obtener_o_crear_subcarpeta_etapa(parent_folder_id, subfolder_name):
    slot = slot_etapa(subfolder_name)
    folder_id = buscar_subcarpeta_etapa(parent_folder_id, subfolder_name)
    if folder_id:
        return folder_id

    print(f"✔ Creando subcarpeta de etapa inexistente: '{subfolder_name}'...")
    folder_metadata = {
        'name': subfolder_name,
        'mimeType': 'application/vnd.google-apps.folder',
        'parents': [parent_folder_id],
        'appProperties': dict(slot.activar)
    }
    new_folder = drive_service.files().create(body=folder_metadata, fields='id').execute()
    return new_folder['id']

def obtener_o_vincular_carpeta_procesados(codigo_programa, nom_programa):
    """Carpeta 'PROG-Juicios Procesados' del programa: se reconoce por propiedades (donde esté y como se llame); si no existe, se crea en la raíz."""
    global carpeta_raiz_instalacion_id
    slot = slot_procesados(codigo_programa)
    folders = buscar_carpetas_slot(slot)
    if folders:
        return unica_o_conflicto(folders, slot)
    if 'carpeta_raiz_instalacion_id' not in globals() or not carpeta_raiz_instalacion_id:
        carpeta_raiz_instalacion_id = obtener_o_vincular_raiz_instalacion()
    folder_metadata = {
        'name': f'{nom_programa}-Juicios Procesados',
        'mimeType': 'application/vnd.google-apps.folder',
        'parents': [carpeta_raiz_instalacion_id],
        'appProperties': {
            'codigo_programa': codigo_programa,
            'procesados': 'true'
        }
    }
    new_folder = drive_service.files().create(body=folder_metadata, fields='id').execute()
    return new_folder['id']

def verificar_carpetas_antes_de_procesar(programas):
    """Solo CONSULTA (no crea nada): si alguna carpeta que se va a usar es ambigua, abre el panel y lanza ConflictoCarpeta
    antes de subir o modificar cualquier archivo. programas: iterable de (codigo_programa, nombre_programa, fecha_fin[, fecha_reporte])."""
    carpetas = buscar_carpetas_slot(slot_plantillas())
    if len(carpetas) > 1:
        lanzar_conflicto(slot_plantillas(), carpetas)
    raiz = globals().get('carpeta_raiz_instalacion_id')
    vistos = set()
    for codigo, programa, fecha_fin, *resto in programas:
        try:
            nom = extraer_iniciales(programa)
            etapa = nombre_carpeta_etapa(nom, fecha_fin, resto[0] if resto else None)
            if (str(codigo), etapa) in vistos:
                continue
            vistos.add((str(codigo), etapa))
            carpetas = buscar_carpetas_slot(slot_procesados(codigo))
            if len(carpetas) > 1:
                lanzar_conflicto(slot_procesados(codigo), carpetas)
            if raiz:
                for nombre_etapa in (etapa, nombre_etapa_contraria(etapa)):   # el archivo se busca en ambas carpetas de etapa
                    carpetas, _ = candidatas_carpeta_etapa(raiz, nombre_etapa)
                    if len(carpetas) > 1:
                        lanzar_conflicto(slot_etapa(nombre_etapa), carpetas)
        except (ConflictoCarpeta, RaizPendiente):
            raise
        except Exception:
            continue   # un encabezado ilegible lo reporta el flujo normal del archivo

def determinar_etapa_carpeta_id(parent_folder_id, nom_programa, fecha_fin_str, fecha_reporte_str=None):
    subfolder_name = nombre_carpeta_etapa(nom_programa, fecha_fin_str, fecha_reporte_str)
    return obtener_o_crear_subcarpeta_etapa(parent_folder_id, subfolder_name), subfolder_name

metadata_carpetas_salida = {}


### 3.3 Notas existentes, fechas y nombres

In [ ]:
def normalizar_fecha(fecha_texto):
    if pd.isna(fecha_texto) or fecha_texto.strip() == '':
        return fecha_texto
    fecha_texto = fecha_texto.strip().replace('-', '/')
    partes = fecha_texto.split('/')
    # Si el primer elemento tiene 4 dígitos, es formato YYYY/MM/DD
    if len(partes[0]) == 4:
        return f"{partes[2]}/{partes[1]}/{partes[0]}"
    else:
        return fecha_texto  # Ya está en formato DD/MM/YYYY

def get_existing_notes(spreadsheet_id, sheet_name, start_row=FILA_INICIAL):
    """
    Lee las notas desde start_row hasta la última fila con datos en las columnas B:F.
    Asocia las notas al ID de la columna IDENTIFICACIÓN (Columna C).
    Retorna un diccionario: { 'ID': { 'COLUMNA': 'NOTA' } }
    """
    # 1. Obtener el ID de la hoja y el número total de filas actuales
    metadata = sheets_service.spreadsheets().get(spreadsheetId=spreadsheet_id).execute()
    sheet_properties = next(s['properties'] for s in metadata['sheets'] if s['properties']['title'] == sheet_name)
    max_rows = sheet_properties['gridProperties']['rowCount']

    column_names = ['T', 'IDENTIFICACIÓN', 'NOMBRES', 'APELLIDOS', 'ESTADO']
    notes_dict = {}

    # 2. Definir el rango dinámico desde start_row hasta el final
    dynamic_range = f'{sheet_name}!B{start_row}:F{max_rows}'

    response = sheets_service.spreadsheets().get(
        spreadsheetId=spreadsheet_id,
        ranges=[dynamic_range],
        fields='sheets.data.rowData.values(note,userEnteredValue)'
    ).execute()

    # Validar si hay datos en la respuesta
    sheets_data = response.get('sheets', [])
    if not sheets_data or 'data' not in sheets_data[0]:
        return notes_dict

    rows_data = sheets_data[0]['data'][0].get('rowData', [])

    for row in rows_data:
        values = row.get('values', [])
        if len(values) > 1:
            # La IDENTIFICACIÓN está en la Columna C (índice 1 del rango B:F)
            cell_id_data = values[1].get('userEnteredValue', {})
            student_id = cell_id_data.get('stringValue') or str(cell_id_data.get('numberValue', ''))

            if student_id and student_id != '0.0' and student_id.strip() != '':
                row_notes = {}
                # Iterar por las celdas de la fila en el rango B:F
                for idx, cell in enumerate(values):
                    note_text = cell.get('note')
                    if note_text:
                        col_name = column_names[idx]
                        row_notes[col_name] = note_text

                if row_notes:
                    notes_dict[student_id] = row_notes
    return notes_dict

def limpiar_nombre(texto_completo):
    palabras = texto_completo.split()
    total_palabras = len(palabras)

    # REGLA 1: Si tiene 2 palabras o menos, se queda completo
    if total_palabras <= 2:
        nombres = palabras
    else:
        # Lista de artículos/conectores comunes que unen nombres
        conectores_nombre = {"DE", "DEL", "LA", "LOS", "LAS"}

        # REGLA 3: Si la segunda palabra es un conector (ej: OMAR *DE* JESUS), tomamos 3 palabras
        if palabras[1] in conectores_nombre:
            nombres = palabras[:3]
        else:
            # REGLA 2: En cualquier otro caso, tomamos estrictamente las 2 primeras palabras
            nombres = palabras[:2]

    # Unir y dar formato estético
    resultado = " ".join(nombres).title()

    # Asegurar que el "de" quede en minúscula si existe
    return resultado.replace("De ", "de ")

def build_note_solicitud(sheet_id, student_id, target_row, notes_dict):
    """
    Genera las solicitudes para re-insertar las notas guardadas en una nueva fila.
    """
    new_requests = []
    col_map = {'T': 'B', 'IDENTIFICACIÓN': 'C', 'NOMBRES': 'D', 'APELLIDOS': 'E', 'ESTADO': 'F'}

    if student_id in notes_dict:
        for col_name, note_content in notes_dict[student_id].items():
            col_letter = col_map[col_name]
            cell_ref = f'{col_letter}{target_row}'
            new_requests.append(solicitud('emerging_note', sheet_id, cell_ref, valor=note_content))

    return new_requests


In [ ]:
def obtener_aprendices_hoja_actual(spreadsheet_id, sheet_name, start_row=FILA_INICIAL):
    """
    Realiza una lectura estructurada y exhaustiva de la hoja actual antes de ser sobreescrita.
    Indexa a todos los aprendices actualmente registrados mediante su número de documento,
    conservando sus estados, datos básicos y juicios/comentarios individuales por cada RAP.

    Retorna:
        dict: {
            'DOCUMENTO_ID': {
                'tipo_doc': str, 'nombres': str, 'apellidos': str, 'estado': str, 'porcentaje': str,
                'juicios_rap': { 'COLUMNA': { 'valor': str, 'nota': str } }
            }
        }
    """
    aprendices_hoja = {}
    try:
        # 1. Obtener metadatos para conocer las dimensiones reales de la cuadrícula
        metadata = sheets_service.spreadsheets().get(spreadsheetId=spreadsheet_id).execute()
        sheet_prop = next(s['properties'] for s in metadata['sheets'] if s['properties']['title'] == sheet_name)
        max_rows = sheet_prop['gridProperties']['rowCount']
        max_cols = sheet_prop['gridProperties']['columnCount']

        if max_rows < start_row:
            return aprendices_hoja # La hoja está vacía o es una plantilla nueva

        # 2. Generar el rango de columnas para los RAPs (desde H hasta el límite de la cuadrícula)
        num_columnas_rap = max_cols - 7
        if num_columnas_rap <= 0:
            return aprendices_hoja

        columnas_rap = generate_column_names('H', num_columnas_rap)
        columna_final_letra = columnas_rap[-1]

        # Rango completo que abarca desde la columna A (índice, tipo doc, documento) hasta la última de RAPs
        fila_rap = start_row - 2   # fila con la nota 'RAP <código> - <nombre>' de cada columna
        rango_lectura = f'{sheet_name}!A{fila_rap}:{columna_final_letra}{max_rows}'

        response = sheets_service.spreadsheets().get(
            spreadsheetId=spreadsheet_id,
            ranges=[rango_lectura],
            fields='sheets.data.rowData.values(note,userEnteredValue)'
        ).execute()

        sheets_data = response.get('sheets', [])
        if not sheets_data or 'data' not in sheets_data[0]:
            return aprendices_hoja

        rows_data = sheets_data[0]['data'][0].get('rowData', [])

        # RAP de cada columna según las notas de la fila 6 de la hoja actual (para ubicar los juicios por RAP y no por posición)
        claves_rap = {}
        if rows_data:
            for idx_celda, celda in enumerate(rows_data[0].get('values', [])):
                if idx_celda >= 7 and idx_celda - 7 < len(columnas_rap) and celda.get('note', '').strip():
                    claves_rap[columnas_rap[idx_celda - 7]] = clave_rap(celda['note'])[0]

        for row in rows_data[start_row - fila_rap:]:
            values = row.get('values', [])
            if len(values) > 5:
                # Obtener Tipo Doc (Columna B -> índice 1), Documento (Columna C -> índice 2)
                cell_tipo_doc = values[1].get('userEnteredValue', {}).get('stringValue', '')
                cell_doc_data = values[2].get('userEnteredValue', {})
                doc_id = cell_doc_data.get('stringValue') or str(cell_doc_data.get('numberValue', ''))
                # Remover posibles ceros decimales en formato de número
                if doc_id.endswith('.0'):
                    doc_id = doc_id[:-2]
                doc_id = doc_id.strip()

                if doc_id and doc_id != '0' and doc_id != '':
                    nombres = values[3].get('userEnteredValue', {}).get('stringValue', '').strip()
                    apellidos = values[4].get('userEnteredValue', {}).get('stringValue', '').strip()
                    estado = values[5].get('userEnteredValue', {}).get('stringValue', '').strip()
                    porcentaje = values[6].get('userEnteredValue', {}).get('stringValue', '').strip() if len(values) > 6 else '0%'

                    # Estructura básica del aprendiz
                    aprendiz_info = {
                        'tipo_doc': cell_tipo_doc,
                        'nombres': nombres,
                        'apellidos': apellidos,
                        'estado': estado,
                        'porcentaje': porcentaje,
                        'juicios_rap': {},
                        'nota_porcentaje': values[6].get('note', '').strip() if len(values) > 6 else ''
                    }

                    # Extraer RAPs de forma matricial (Inicia en columna H -> índice 7)
                    for j, col_letra in enumerate(columnas_rap):
                        idx_celda = 7 + j
                        if idx_celda < len(values):
                            cell_rap = values[idx_celda]
                            val_rap = cell_rap.get('userEnteredValue', {}).get('stringValue', '').strip()
                            nota_rap = cell_rap.get('note', '').strip()

                            if val_rap or nota_rap:
                                aprendiz_info['juicios_rap'][claves_rap.get(col_letra, f'#{col_letra}')] = {
                                    'valor': val_rap,
                                    'nota': nota_rap
                                }

                    aprendices_hoja[doc_id] = aprendiz_info
    except Exception as e:
        print(f"⚠️ No se pudo realizar la lectura estructural previa de la hoja '{sheet_name}': {str(e)}")

    return aprendices_hoja


def leer_fecha_reporte_hoja(spreadsheet_id, sheet_name):
    """Fecha del reporte (D4) que tiene la hoja ANTES de actualizarla (datetime.date o None si está vacía o no se pudo leer)."""
    try:
        r = sheets_service.spreadsheets().get(spreadsheetId=spreadsheet_id, ranges=[f"'{sheet_name}'!D4"],
                                              fields='sheets.data.rowData.values(formattedValue,userEnteredValue)').execute()
        celda = r['sheets'][0]['data'][0]['rowData'][0]['values'][0]
        escrito = celda.get('userEnteredValue', {})
        if es_numero(escrito.get('numberValue')):           # fecha guardada como fecha: se lee el número de serie (sin ambigüedad regional)
            return serial_a_datetime(escrito['numberValue']).date()
        valor = escrito.get('stringValue') or celda.get('formattedValue')
        return parsear_fecha(valor) if valor else None
    except Exception:
        return None


In [ ]:
def identificar_aprendices_no_en_ficha(aprendices_hoja, df_sofia_nuevo):
    """
    Compara el diccionario de aprendices recuperados de la hoja actual con el
    nuevo set de datos cargado desde el reporte de Sofia Plus.

    Identifica aquellos documentos que están presentes en la hoja de Sheets pero ya
    no aparecen en el nuevo reporte de Sofia Plus (aprendices históricos/retirados).

    Retorna:
        dict: Subconjunto de `aprendices_hoja` con los estudiantes ausentes clasificados como 'NO EN FICHA'.
    """
    aprendices_no_en_ficha = {}

    # Obtener el set de documentos únicos del nuevo reporte de Sofia
    documentos_sofia_nuevos = set(df_sofia_nuevo['Número de Documento'].astype(str).str.strip().unique())

    # Iterar sobre los alumnos existentes en la hoja de Sheets
    for doc_id, datos_hoja in aprendices_hoja.items():
        doc_id_clean = str(doc_id).strip()

        # Si el aprendiz no está en el nuevo reporte, califica para preservación especial
        if doc_id_clean not in documentos_sofia_nuevos:
            # Preservamos toda su estructura original pero actualizamos su estado temporal a 'NO EN FICHA'
            datos_preservados = datos_hoja.copy()
            # Se conserva el estado de formación real (columna F); solo si está vacío se marca 'NO EN FICHA'
            if not str(datos_preservados.get('estado', '')).strip():
                datos_preservados['estado'] = 'NO EN FICHA'

            aprendices_no_en_ficha[doc_id_clean] = datos_preservados

    if aprendices_no_en_ficha:
        print(f"🔍 Auditoría retrospectiva: Se identificaron {len(aprendices_no_en_ficha)} aprendices que no figuran en el reporte actual de SOFIA Plus (puede ser un reporte de fecha anterior); se conservan como referencia.")
    else:
        print("🔍 Auditoría retrospectiva: Todos los aprendices de la hoja actual se encuentran vigentes en el nuevo reporte de Sofia Plus.")

    return aprendices_no_en_ficha

In [ ]:
def preparar_bloque_aprendices_no_en_ficha(h, aprendices_no_en_ficha, inicio_fila, est, tabla_ra):
    """
    Genera las solicitudes de actualización, color y notas para los estudiantes
    históricos que ya no figuran en el reporte Sofia Plus actual, ubicándolos al final
    de la hoja con un formato visualmente atenuado (escala de grises).
    """
    solicitudes = []
    sin_ubicar = []
    col_por_texto, col_por_nombre = columnas_por_rap(h, tabla_ra)
    COLOR_HISTORICO_BASE = [240/255, 242/255, 244/255, 1.0] # Gris atenuado
    COLOR_HISTORICO_CELDA = [245/255, 247/255, 249/255, 1.0]

    for idx, (doc_id, info) in enumerate(aprendices_no_en_ficha.items()):
        fila_destino = str(inicio_fila + idx)

        # 1. Gris solo en las celdas A:G con contenido; las vacías quedan en blanco
        contenido_basico = {'A': True, 'B': info['tipo_doc'], 'C': doc_id, 'D': info['nombres'], 'E': info['apellidos'],
                            'F': info['estado'], 'G': info['porcentaje']}
        for letra_basica, contenido in contenido_basico.items():
            color_celda = COLOR_ESTADOS['NO EN FICHA'] if str(contenido).strip() else [1.0, 1.0, 1.0, 1.0]
            solicitudes.append(solicitud('color_cells', h.sheet_id, ga(letra_basica, fila_destino), valor=color_celda))

        # 2. Inyectar datos básicos
        solicitudes.append(solicitud('update_cells', h.sheet_id, ga('A', fila_destino), valor=f"H-{idx+1}"))
        solicitudes.append(solicitud('update_cells', h.sheet_id, ga('B', fila_destino), valor=info['tipo_doc']))
        solicitudes.append(solicitud('update_cells', h.sheet_id, ga('C', fila_destino), valor=doc_id))
        solicitudes.append(solicitud('update_cells', h.sheet_id, ga('D', fila_destino), valor=info['nombres']))
        solicitudes.append(solicitud('update_cells', h.sheet_id, ga('E', fila_destino), valor=info['apellidos']))
        solicitudes.append(solicitud('update_cells', h.sheet_id, ga('F', fila_destino), valor=info['estado']))
        solicitudes.append(solicitud('update_cells', h.sheet_id, ga('G', fila_destino), valor=info['porcentaje']))

        # 3. Restaurar las notas de los juicios ubicándolas por RAP (no por posición de columna)
        juicios_por_col = {}
        for clave, datos_celda_rap in info['juicios_rap'].items():
            col_destino = buscar_columna_rap(clave, col_por_texto, col_por_nombre)
            if col_destino is None and clave.startswith('#') and clave[1:] in h.columnas:
                col_destino = clave[1:]   # columna sin nota de RAP en la hoja anterior: se conserva su posición
            if col_destino is None:
                if datos_celda_rap.get('nota'):
                    sin_ubicar.append(f"documento {doc_id}, RAP '{clave}'")
                continue
            juicios_por_col[col_destino] = datos_celda_rap
        for col_letra in h.columnas:
            nota_rap = juicios_por_col.get(col_letra, {}).get('nota')
            if nota_rap:
                solicitudes.append(solicitud('emerging_note', h.sheet_id, ga(col_letra, fila_destino), valor=nota_rap))
                # Gris si la celda RAP tiene nota (si tiene texto manual, se pinta al restaurarlo)
                solicitudes.append(solicitud('color_cells', h.sheet_id, ga(col_letra, fila_destino), valor=COLOR_ESTADOS['NO EN FICHA']))

        # 4. Nota descriptiva en el porcentaje indicando el carácter histórico del estudiante
        # Se conserva la nota original de la celda del porcentaje (el mensaje de histórico va en la celda 'NO EN FICHA' del encabezado)
        texto_nota_porcentaje = info.get('nota_porcentaje') or f"👤 {info['nombres']} {info['apellidos']}"
        solicitudes.append(solicitud('emerging_note', h.sheet_id, ga('G', fila_destino), valor=texto_nota_porcentaje))

    if sin_ubicar:
        print("⚠️ Notas de aprendices históricos cuyo RAP ya no existe en la hoja nueva (no se restauraron):\n   " + "\n   ".join(sin_ubicar))
    return solicitudes

### 3.4 Cálculos e informes

In [ ]:
def emoji_tematica(tematica):
  """Emoji que acompaña a la temática en los informes."""
  return "🇬🇧" if "INGLÉS" in tematica.upper() else "⚡" if "ELEC" in tematica.upper() or "MED" in tematica.upper() else "📚"

def formato_raps(lista_raps):
  """[1, 2, 3] -> '1,2,3'"""
  return str(lista_raps).replace('[', '').replace(']', '').replace(' ', '')

def tematica_de(tabla_ra, k, por_defecto):
  """Temática del RAP en la posición k (base 0); por_defecto si el archivo de orden no tiene columna TEMÁTICA."""
  return tabla_ra['TEMÁTICA'][k] if 'TEMÁTICA' in tabla_ra.columns else por_defecto

def filas_por_aprendiz(dfRA, cantidad_aprendices):
  """Fila inicial y cantidad de filas de cada aprendiz en dfRA (que está ordenado por estado, apellidos, nombre y documento)."""
  conteo = dfRA['Número de Documento'].value_counts()
  inicios, cantidades = [], []
  m = 0
  for i in range(cantidad_aprendices):
    cantidad = int(conteo.get(dfRA['Número de Documento'][m], 0))
    inicios.append(m)
    cantidades.append(cantidad)
    m = m + cantidad
  return inicios, cantidades

def recorrer_raps(dfRA, inicio, cantidad_RAs):
  """Por cada RAP j (base 0) entrega (j, fila): la fila de dfRA de ese RAP para el aprendiz que inicia en 'inicio',
  o (j, None) si el aprendiz no tiene ese RAP (en ese caso las filas siguientes se corren una posición)."""
  n = 0
  for j in range(cantidad_RAs):
    if dfRA.loc[inicio+j-n,'Orden'] != j:
      n = n+1
      yield j, None
    else:
      yield j, inicio+j-n

def calcular_estadisticas(dfRA, cantidad_aprendices, cantidad_RAs, column_names, tabla_ra, inicios, cantidades=None, etapa=None):
  """Cálculos de la ficha: evaluados por RAP, instructores, pendientes, colores y conteo por estado.

  Regla de pendientes (solo aprendices en ESTADOS_ACTIVOS; RAP en POR EVALUAR o NO APROBADO). 'etapa' es info_ficha(...):
    - LECTIVA: un RAP lectivo es pendiente si ya empezó en el grupo (algún activo lo tiene APROBADO o NO APROBADO). REZAGADO.
    - PRODUCTIVA: todo RAP lectivo no aprobado es pendiente (la lectiva ya terminó). VENCIDO.
    - TERMINADA POR FECHA: además, el RAP de Etapa Productiva. VENCIDO.
    - El RAP de Etapa Productiva solo es pendiente cuando la ficha terminó por fecha.
    - Sin etapa válida (fechas ilegibles): criterio anterior (solo los RAPs que ya empezaron en el grupo)."""
  etapa_nombre = etapa.etapa if (etapa is not None and etapa.valida) else None
  f_reporte = etapa.f_reporte if etapa is not None else None
  fechas = dict(zip(dfRA.index, [parsear_fecha(v) for v in dfRA['Fecha y Hora del Juicio Evaluativo']]))
  productivos = {k+1 for k in range(cantidad_RAs) if es_rap_productivo(tabla_ra.Competencias[k])}   # RAP (base 1) de Etapa Productiva

  evaluados = {col: 0 for col in column_names}                 # aprobados por columna de RAP
  df_evaluadores = pd.DataFrame(index=range(cantidad_aprendices), columns=column_names)  # instructor que aprobó cada celda
  candidatos = {j+1: [] for j in range(cantidad_RAs)}          # RAP (base 1) -> aprendices activos (posición) en POR EVALUAR o NO APROBADO
  estado_pendiente = {}                                        # (aprendiz, RAP base 1) -> 'POR EVALUAR' | 'NO APROBADO'
  raps_iniciados = set()                                       # RAP (base 1) con al menos un aprendiz activo APROBADO o NO APROBADO
  raps_instructor = {}                                         # instructor -> RAPs (base 1) que aprobó
  ultimo_instructor = {}                                       # instructor -> fecha de su último juicio aprobado
  detalle = {j+1: SimpleNamespace(activos=0, aprobados=0, primero=None, ultimo=None) for j in range(cantidad_RAs)}   # por RAP, solo aprendices activos
  nombres, estados, ultimo_aprendiz, aprobados_aprendiz, filas_aprendiz = {}, {}, {}, {}, {}
  todas = [f for f in fechas.values() if f]
  ultimo_ficha = max(todas) if todas else None                 # fecha del último juicio registrado en el reporte

  for i in range(cantidad_aprendices):
    m = inicios[i]
    nombres[i] = nombre_completo(f"{dfRA.loc[m,'Nombre']} {dfRA.loc[m,'Apellidos']}")
    estados[i] = dfRA.loc[m,'Estado']
    activo = estados[i] in ESTADOS_ACTIVOS
    aprobados_aprendiz[i], filas_aprendiz[i] = 0, 0
    for j, fila in recorrer_raps(dfRA, m, cantidad_RAs):
      if fila is None:
        continue
      filas_aprendiz[i] += 1
      juicio = dfRA.loc[fila,'Juicio de Evaluación']
      fecha = fechas.get(fila)
      if fecha and (ultimo_aprendiz.get(i) is None or fecha > ultimo_aprendiz[i]):
        ultimo_aprendiz[i] = fecha
      if len(dfRA.loc[fila,'Funcionario que registro el juicio evaluativo']) > 7:
        funcionario = dfRA.loc[fila,'Funcionario que registro el juicio evaluativo']
        instructor_name = funcionario[funcionario.find('-')+2:].strip()
        DOC_INSTRUCTOR[clave_nombre(instructor_name)] = solo_documento(funcionario.partition('-')[0])
        if juicio == 'APROBADO':
          df_evaluadores.loc[i,column_names[j]] = instructor_name
          evaluados[column_names[j]] = evaluados[column_names[j]] + 1
          raps_instructor.setdefault(instructor_name, []).append(j+1)
          if fecha and (ultimo_instructor.get(instructor_name) is None or fecha > ultimo_instructor[instructor_name]):
            ultimo_instructor[instructor_name] = fecha
      if juicio == 'APROBADO':
        aprobados_aprendiz[i] += 1
      if activo:
        d = detalle[j+1]
        d.activos += 1
        if juicio == 'APROBADO':
          d.aprobados += 1
        if fecha:
          d.primero = fecha if d.primero is None else min(d.primero, fecha)
          d.ultimo = fecha if d.ultimo is None else max(d.ultimo, fecha)
        if juicio in ('POR EVALUAR', 'NO APROBADO'):
          candidatos[j+1].append(i)
          estado_pendiente[(i, j+1)] = juicio
        if juicio in ('APROBADO', 'NO APROBADO'):
          raps_iniciados.add(j+1)

  instructores_total = {key:len(value) for key,value in raps_instructor.items()}   # evaluaciones por instructor
  instructores_total = dict(sorted(instructores_total.items(), key=lambda item: item[1], reverse=True))
  raps_instructor = {key:sorted(list(set(value)),reverse=False) for key,value in raps_instructor.items()}

  conteos = {col: df_evaluadores[col].dropna().value_counts() for col in df_evaluadores.columns}  # evaluadores por columna
  colores_aprobados = {}   # columna -> {instructor: color}
  for col, evaluator_counts in conteos.items():
    colorsAprobados = iter(COLORES_APROBADOS)
    if not evaluator_counts.empty:
      colores_aprobados[col] = {evaluator.strip():next(colorsAprobados) for evaluator, count in evaluator_counts.items() }

  valor_estados = dfRA.groupby('Estado',observed=True)['TD_Documento'].nunique().reindex(ESTADOS_TODOS,fill_value=0).to_dict()

  def es_pendiente(rap):
    if etapa_nombre is None:                       # sin fechas válidas: criterio anterior
      return rap in raps_iniciados
    if rap in productivos:                         # Etapa Productiva: solo cuando la ficha terminó por fecha
      return etapa_nombre == ETAPA_FINALIZADA
    if etapa_nombre in (ETAPA_PRODUCTIVA, ETAPA_FINALIZADA):   # la lectiva terminó: todo RAP lectivo no aprobado
      return True
    return rap in raps_iniciados                   # lectiva en curso: solo los RAPs que ya empezaron en el grupo

  pendientes_col = {rap: lista for rap, lista in candidatos.items() if lista and es_pendiente(rap)}
  pendientes_fila = {j:[] for j in range(cantidad_aprendices)}   # aprendiz (posición) -> RAPs pendientes
  for key,value in pendientes_col.items():
    for i in value:
      pendientes_fila[i].append(key)
  pendientes_fila = {key:value for key,value in pendientes_fila.items() if len(value)>0}

  # Desde cuándo cuenta el tiempo de cada RAP pendiente: cierre de su etapa (VENCIDO) o último juicio del grupo (REZAGADO)
  atraso = {}
  for rap in pendientes_col:
    cierre = None
    if etapa_nombre is not None:
      cierre = etapa.f_fin if rap in productivos else etapa.f_fin_lectiva
    if cierre is not None and f_reporte > cierre:
      atraso[rap] = SimpleNamespace(vencido=True, desde=cierre)
    else:
      atraso[rap] = SimpleNamespace(vencido=False, desde=detalle[rap].ultimo)

  activos = [i for i in range(cantidad_aprendices) if estados[i] in ESTADOS_ACTIVOS]
  total_filas = cantidades if cantidades is not None else [filas_aprendiz[i] for i in range(cantidad_aprendices)]
  avance = {i: round(aprobados_aprendiz[i] / total_filas[i] * 100) for i in activos if total_filas[i]}   # % de avance de cada aprendiz activo
  etiqueta = {ETAPA_LECTIVA: 'REZAGADOS', ETAPA_PRODUCTIVA: 'VENCIDOS', ETAPA_FINALIZADA: 'VENCIDOS'}.get(etapa_nombre, '')

  instructor_principal = {k: conteos[col].idxmax().strip() for k, col in enumerate(df_evaluadores.columns) if not conteos[col].empty}  # RAP (base 0) -> instructor con más evaluados

  return SimpleNamespace(evaluados=evaluados, conteos=conteos, pendientes_col=pendientes_col, pendientes_fila=pendientes_fila,
                         raps_instructor=raps_instructor, instructores_total=instructores_total,
                         colores_aprobados=colores_aprobados, valor_estados=valor_estados,
                         instructor_principal=instructor_principal,
                         etapa=etapa, etiqueta=etiqueta, productivos=productivos, detalle=detalle, atraso=atraso,
                         estado_pendiente=estado_pendiente, nombres=nombres, estados=estados,
                         ultimo_aprendiz=ultimo_aprendiz, ultimo_instructor=ultimo_instructor, ultimo_ficha=ultimo_ficha,
                         activos=activos, avance=avance, iniciados=raps_iniciados)

# ----------------------- Textos de las notas e informes -----------------------

def dias_atraso(est, rap):
  """Días de atraso del RAP pendiente (None si falta alguna fecha)."""
  a = est.atraso.get(rap)
  f_reporte = est.etapa.f_reporte if est.etapa is not None else None
  return (f_reporte - a.desde).days if (a is not None and a.desde is not None and f_reporte is not None) else None

def tiempo_atraso(est, raps):
  """Tiempo (meses y días, formato corto) del RAP más atrasado de la lista."""
  if est.etapa is None or est.etapa.f_reporte is None:
    return ''
  mejor = None
  for rap in raps:
    d = dias_atraso(est, rap)
    if d is not None and (mejor is None or d > mejor[0]):
      mejor = (d, rap)
  if mejor is None:
    return 'sin fecha'
  return duracion_texto(est.atraso[mejor[1]].desde, est.etapa.f_reporte, corto=True)

def leyenda_tiempo(est):
  """Qué significa el tiempo que acompaña a cada pendiente, según la etapa."""
  if est.etapa is None or not est.etapa.valida:
    return ''
  if est.etapa.etapa == ETAPA_LECTIVA:
    return "⏱ = tiempo sin juicios del grupo en ese RAP (al corte)"
  return "⏱ = tiempo vencido al corte (desde que terminó la etapa del RAP)"

def notas_resumen_columnas(est, dfRA, tabla_ra, column_names, cantidad_aprendices):
  """Nota de la fila 2 de cada RAP: temática, evaluadores, avance de los aprendices activos, fechas y pendientes. {columna: texto}"""
  f_reporte = est.etapa.f_reporte if est.etapa is not None else None
  notas = {}
  for k, col in enumerate(column_names):
    rap = k + 1
    evaluator_counts = est.conteos[col]
    pendientes = est.pendientes_col.get(rap, [])
    if evaluator_counts.empty and not pendientes:
      continue
    lineas = [linea_corte(f_reporte)] if f_reporte else []
    lineas.append(f"📚 Temática: {tematica_de(tabla_ra, k, 'GENERAL')}")
    if evaluator_counts.empty:
      lineas.append("👤 Evaluadores: ninguno aún")
    else:
      lineas.append("👤 Evaluadores:")
      lineas += [f" ▫ {etiqueta_instructor(ev.strip())} ({count} {'evaluado' if count == 1 else 'evaluados'})" for ev, count in evaluator_counts.items()]
    d = est.detalle[rap]
    if d.activos:
      lineas.append(f"📊 Aprendices activos: {d.aprobados} de {d.activos} aprobados ({round(d.aprobados / d.activos * 100)}%)")
    if d.primero is not None:
      if d.primero == d.ultimo:
        lineas.append(f"🕒 Primer y último juicio: {fecha_corta(d.ultimo)}")
      else:
        lineas.append(f"🕒 Primer juicio {fecha_corta(d.primero)} · último {fecha_corta(d.ultimo)}")
    if pendientes and est.atraso[rap].vencido:
      cierre = est.atraso[rap].desde
      lineas.append(f"⛔ VENCIDO: la etapa terminó el {fecha_corta(cierre)}")
    lineas.append("───────────────────────────────────")
    if pendientes:
      lineas.append(f"⏳ PENDIENTES ({len(pendientes)}):")
      for i in pendientes:
        marca = '✖' if est.estado_pendiente.get((i, rap)) == 'NO APROBADO' else '▪'
        extra = ' · NO APROBADO' if marca == '✖' else ''
        lineas.append(f" {marca} #{i + 1} {est.nombres[i]}{extra}")
    else:
      lineas.append("✅ Sin pendientes")
    notas[col] = '\n'.join(lineas)
  return notas

def pendientes_agrupados(raps_pendientes, est, tabla_ra):
  """Agrupa los RAPs pendientes de un aprendiz por (temática, instructor principal). {(temática, instructor): [RAPs]}
  El instructor es None si nadie ha aprobado ese RAP."""
  agrupados = {}
  for rap in raps_pendientes:
    tematica = tematica_de(tabla_ra, rap - 1, 'GENERAL')
    agrupados.setdefault((tematica, est.instructor_principal.get(rap - 1)), []).append(rap)
  return agrupados

def textos_aprendiz(nombre, apellidos, porcentaje, aprobados, total_raps, raps_pendientes, est, tabla_ra, estado=None, i=None):
  """(nota individual, fragmento del informe G7) de un aprendiz. El fragmento es '' si no tiene RAPs pendientes.
  'i' es la posición (base 0) del aprendiz en la tabla; la nota y el informe lo muestran como #N."""
  nombre_formateado = nombre_completo(f"{nombre} {apellidos}")
  pos = f"#{i + 1} " if i is not None else ''
  nota = f"👤 {pos}{nombre_formateado}\n📊 Avance: {porcentaje}% ({aprobados} de {total_raps} RAPs aprobados)"
  ultimo = est.ultimo_aprendiz.get(i) if i is not None else None
  if ultimo:
    nota += f" · último juicio {fecha_corta(ultimo)}"
  fragmento = ''
  if estado is not None and estado not in ESTADOS_ACTIVOS:
    return nota + f"\n───────────────────────────────────\nℹ️ Estado: {estado}. No aplica seguimiento de pendientes.", fragmento
  f_corte = est.etapa.f_reporte if est.etapa is not None else None
  nota += f"\n{linea_corte(f_corte)}" if f_corte else ''
  nota += f"\n{linea_etapa_aprendiz(est)}" if linea_etapa_aprendiz(est) else ''
  nota += "\n───────────────────────────────────"
  if raps_pendientes is None:
    if aprobados == 0:
      return nota + "\n⏳ Sin juicios aprobados aún.", fragmento
    return nota + "\n✅ ¡Al día! No tiene RAPs pendientes de evaluación.", fragmento

  uno = len(raps_pendientes) == 1
  etiqueta = f" · {est.etiqueta[:-1] if uno else est.etiqueta}" if est.etiqueta else ''
  nota += f"\n⚠️ {len(raps_pendientes)} {'RAP PENDIENTE' if uno else 'RAPs PENDIENTES'} [{rangos_texto(raps_pendientes)}]{etiqueta}"
  if leyenda_tiempo(est):
    nota += f"\n{leyenda_tiempo(est)}"
  if any(est.estado_pendiente.get((i, rap)) == 'NO APROBADO' for rap in raps_pendientes):
    nota += "\n✖ = NO APROBADO"
  fragmento += f"\n{pos}{nombre_formateado} · {porcentaje}% ({aprobados}/{total_raps})\n"
  for (tematica, instructor), lista_raps in pendientes_agrupados(raps_pendientes, est, tabla_ra).items():
    emoji_materia = emoji_tematica(tematica)
    quien = etiqueta_instructor(instructor)
    nota += f"\n\n{emoji_materia} {tematica} · 👨‍🏫 {etiqueta_instructor(instructor, con_correo=True)}"
    for rap in lista_raps:
      codigo, nombre_rap = rap_corto(tabla_ra.RA[rap - 1])
      marca = ' ✖' if est.estado_pendiente.get((i, rap)) == 'NO APROBADO' else ''
      nombre_con_codigo = f"{codigo}-{nombre_rap}" if codigo else nombre_rap
      productivo = ' [Etapa Productiva]' if rap in est.productivos else ''
      tiempo = tiempo_atraso(est, [rap])
      nota += f"\n   RAP {rap}{marca} {nombre_con_codigo}{productivo}" + (f" · ⏱ {tiempo}" if tiempo else '')
    tiempo_grupo = tiempo_atraso(est, lista_raps)
    fragmento += f" ▪ {emoji_materia} {tematica} · {'RAP' if len(lista_raps) == 1 else 'RAPs'} {rangos_texto(lista_raps)} · {quien}" + (f" · ⏱ {tiempo_grupo}" if tiempo_grupo else '') + "\n"
  fragmento += "─" * 42 + "\n"
  return nota, fragmento

def linea_etapa_aprendiz(est):
  """Línea de etapa de la nota de cada aprendiz (vacía si no hay fechas válidas)."""
  if est.etapa is None or not est.etapa.valida:
    return ''
  if est.etapa.etapa == ETAPA_FINALIZADA:
    return f"🔒 {ETAPA_FINALIZADA} · la ficha terminó el {fecha_corta(est.etapa.f_fin)}"
  return f"📅 {linea_etapa(est.etapa).replace('Etapa: ', '')}"

def encabezado_informe_pendientes(est, ficha, aviso_anterior=''):
  """Encabezado de la nota de G7: etapa, qué significan los pendientes y avance del grupo."""
  etapa = est.etapa
  f_rep = etapa.f_reporte if etapa is not None else None
  lineas = [f"⚡ PENDIENTES · FICHA {ficha}" + (f" · corte {fecha_corta(f_rep)}" if f_rep else '')]
  if aviso_anterior:
    lineas.append(aviso_anterior)
  if etapa is not None and etapa.valida:
    lineas.append(linea_etapa(etapa))
    if etapa.etapa == ETAPA_LECTIVA:
      lineas.append("REZAGADOS: el grupo ya evaluó estos RAPs")
      lineas.append(leyenda_tiempo(est))
    elif etapa.etapa == ETAPA_PRODUCTIVA:   # el encabezado ya da las fechas: la leyenda del tiempo va en la misma línea
      lineas.append(f"VENCIDOS: la lectiva terminó el {fecha_corta(etapa.f_fin_lectiva)}. "
                    "El RAP de Etapa Productiva aún no cuenta · ⏱ = tiempo vencido")
    else:
      lineas.append(f"VENCIDOS: la lectiva terminó el {fecha_corta(etapa.f_fin_lectiva)} y la etapa productiva el {fecha_corta(etapa.f_fin)} · ⏱ = tiempo vencido")
  if est.avance:
    valores = list(est.avance.values())
    lineas.append(f"Avance real de los aprendices activos: {round(sum(valores) / len(valores))}% (rango {min(valores)}%–{max(valores)}%)")
  lineas.append(f"{len(est.activos)} aprendices activos · {len(est.pendientes_fila)} con pendientes")
  lineas.append("=" * 52)
  return '\n'.join(lineas) + '\n'

def informe_instructores(est, tabla_ra):
  """Informe de instructores evaluadores (nota de G2): evaluaciones, temáticas, RAPs y pendientes por posición."""
  f_reporte = est.etapa.f_reporte if est.etapa is not None else None
  datos_instructores = "🎓 INFORME DE INSTRUCTORES EVALUADORES\n"
  datos_instructores += (f"Corte: {fecha_corta(f_reporte)} · " if f_reporte else '') + "#N = posición del aprendiz en la tabla (columna A)\n"
  datos_instructores += "="*50 + "\n\n"
  for instructor_name, total_evals in est.instructores_total.items():
    instructor_name_clean = instructor_name.strip()
    datos_instructores += f"👤 {etiqueta_instructor(instructor_name_clean)}\n"
    ultimo = est.ultimo_instructor.get(instructor_name_clean)
    datos_instructores += f"   📝 {total_evals} {'evaluación' if total_evals == 1 else 'evaluaciones'}"
    if ultimo:
      datos_instructores += f" · último juicio {fecha_corta(ultimo)}"
    datos_instructores += "\n"
    raps_del_instructor = est.raps_instructor.get(instructor_name_clean, [])  # RAPs que aprobó al menos una vez

    inst_tematicas = {}   # Agrupar RAPs evaluados por temáticas
    for rap_num in raps_del_instructor:
      inst_tematicas.setdefault(tematica_de(tabla_ra, rap_num - 1, 'GENERAL'), []).append(rap_num)

    for tematica, raps_lista in inst_tematicas.items():
      emoji_materia = emoji_tematica(tematica)
      pendientes_ids_pos = []   # posición (columna A, base 1) de los aprendices con pendientes en estos RAPs
      for rap_num in raps_lista:
        for idx in est.pendientes_col.get(rap_num, []):
          if idx + 1 not in pendientes_ids_pos:
            pendientes_ids_pos.append(idx + 1)
      linea = f"   {emoji_materia} {tematica} · RAPs que evaluó: {rangos_texto(raps_lista)} ({len(raps_lista)} {'RAP' if len(raps_lista) == 1 else 'RAPs'})"
      if pendientes_ids_pos:
        n = len(pendientes_ids_pos)
        datos_instructores += linea + f"\n      ⚠️ {n} {'aprendiz' if n == 1 else 'aprendices'} con pendientes en estos RAPs: {posiciones_texto(pendientes_ids_pos)}\n"
      else:
        datos_instructores += linea + " · ✅ sin pendientes\n"
    datos_instructores += "\n"

  sin_evaluador = [rap for rap in est.pendientes_col if (rap - 1) not in est.instructor_principal]
  if sin_evaluador:
    datos_instructores += "⛔ RAPs PENDIENTES SIN NINGÚN JUICIO APROBADO\n"
    for rap in sin_evaluador:
      tematica = tematica_de(tabla_ra, rap - 1, 'GENERAL')
      posiciones = [i + 1 for i in est.pendientes_col[rap]]
      productivo = ' (Etapa Productiva)' if rap in est.productivos else ''
      datos_instructores += (f"   {emoji_tematica(tematica)} {tematica} · RAP {rap}{productivo} · "
                             f"{len(posiciones)} {'pendiente' if len(posiciones) == 1 else 'pendientes'}: {posiciones_texto(posiciones)}\n")
  return datos_instructores

# ----------------------- Notas de los títulos G5 (competencias) y G6 (RAPs) -----------------------

def partes_competencia(competencia):
  """(código, nombre) de una competencia de Sofia: '36180 - Enrique Low Murtra-Interactuar ...'."""
  if es_rap_productivo(competencia):
    return '', 'Resultados de aprendizaje de la Etapa Productiva'
  codigo, _, nombre = str(competencia).partition(' - ')
  if not nombre:
    codigo, nombre = '', str(competencia)
  nombre = ' '.join(nombre.split())
  if nombre.isupper():
    nombre = nombre.capitalize()
  return codigo.strip(), nombre

def nota_competencias(est, tabla_ra, competencias, n_raps):
  """Nota de G5: lista numerada de competencias (el número y el color son los de las filas 5 y 6), con sus RAPs y avance."""
  por_competencia = {}
  for k in range(n_raps):
    por_competencia.setdefault(tabla_ra.Competencias[k], []).append(k + 1)
  orden = sorted(por_competencia, key=lambda c: competencias.index(c) if c in competencias else len(competencias))
  lineas = [f"🎯 COMPETENCIAS ({len(por_competencia)})", "Fila 5 'COM' COMPETENCIAS, fila 6 'RAPs'. Igual color asocia la misma competencia", "═" * 53]
  for comp in orden:
    raps = por_competencia[comp]
    numero = competencias.index(comp) + 1 if comp in competencias else '–'
    codigo, nombre = partes_competencia(comp)
    aprobados = sum(est.detalle[r].aprobados for r in raps)
    activos = sum(est.detalle[r].activos for r in raps)
    avance = f" · ✅ {round(aprobados / activos * 100)}% aprobado" if activos else ''
    lineas.append(f"\n{numero}. {nombre}")
    lineas.append(f"   {'🔖 ' + codigo + ' · ' if codigo else ''}{'RAP' if len(raps) == 1 else 'RAPs'} {rangos_texto(raps)} ({len(raps)}){avance}")
  return recortar_nota('\n'.join(lineas))

def miles(n):
  """1211 -> '1.211'"""
  return f"{n:,}".replace(',', '.')

def nota_raps(est, n_raps):
  """Nota de G6: resumen de los RAPs de la ficha (cantidad, evaluados, completos, pendientes y sin iniciar)."""
  productivos = len(est.productivos)
  detalle = est.detalle.values()
  juicios_aprobados = sum(d.aprobados for d in detalle)
  juicios_activos = sum(d.activos for d in detalle)
  completos = sum(1 for d in est.detalle.values() if d.activos and d.aprobados == d.activos)
  sin_iniciar = sum(1 for rap in est.detalle if rap not in est.iniciados)
  lineas = ["📋 RESULTADOS DE APRENDIZAJE (RAPs)", "═" * 34,
            f"📚 {n_raps} RAPs" + (f": {n_raps - productivos} lectivos y {productivos} de Etapa Productiva" if productivos else '')]
  if juicios_activos:
    lineas.append(f"📊 Evaluados: {round(juicios_aprobados / juicios_activos * 100)}% ({miles(juicios_aprobados)} de {miles(juicios_activos)} juicios aprobados de los aprendices activos)")
  lineas += [f"✅ Completos (aprobados por todos los activos): {completos} de {n_raps}",
             f"⏳ Con pendientes: {len(est.pendientes_col)}",
             f"🆕 Sin iniciar (ningún activo con juicio aprobado o no aprobado): {sin_iniciar}",
             f"👨\u200d🏫 Instructores evaluadores: {len(est.instructores_total)}"]
  if est.ultimo_ficha:
    lineas.append(f"🕒 Último juicio registrado: {fecha_corta(est.ultimo_ficha)}")
  lineas.append("El número de la fila 6 es la posición del RAP")
  return '\n'.join(lineas)


### 3.5 Solicitudes de la hoja de salida

In [ ]:
def rango_rap(h):
  """Rango de la tabla de RAPs x aprendices, p. ej. 'H8:BO39'."""
  return f'{h.columnas[0]}{h.fila_inicial}:{h.columnas[-1]}{h.n_aprendices+h.fila_inicial-1}'

def solicitudes_estructura(h, props):
  """Inserta o borra columnas y filas para ajustar la hoja a la cantidad de RAPs y aprendices."""
  s = []
  if props['columnCount'] < 10:   # Plantilla genérica: una sola columna de RAP
    s.append(solicitud('insert_row_col',h.sheet_id,rango='H1',cantidad=h.n_RAs - 1,valor=['COLUMNS', 'AFTER']))
    s.append(solicitud('copy_format',h.sheet_id,rango='H1:H100',valor=f'I1:{h.columnas[-1]}100'))

  filas_actuales = props['rowCount']
  filas_necesarias = h.n_aprendices+h.fila_inicial-1
  if filas_actuales < filas_necesarias:
    # Insertamos las filas necesarias. Desactivamos inheritFromBefore para que no herede el formato de la fila superior (fila 7)
    filas_a_insertar = filas_necesarias - filas_actuales
    fila_insercion = filas_actuales - 1

    s.append({
      "insertDimension": {
        "range": {
          "sheetId": h.sheet_id,
          "dimension": "ROWS",
          "startIndex": fila_insercion,
          "endIndex": fila_insercion + filas_a_insertar
        },
        "inheritFromBefore": False
      }
    })
    # Copiamos explícitamente el formato patrón de la fila 8 (A8:H8) a todas las filas recién insertadas para mantener la uniformidad estricta
    s.append(solicitud('copy_format', h.sheet_id, rango='A8:H8', valor=f'A9:H{filas_necesarias}'))

  if filas_actuales > filas_necesarias:
    sobrantes = filas_actuales - filas_necesarias
    # Empezar la eliminación estrictamente una fila después del límite necesario real
    fila_inicio_recorte = filas_necesarias + 1
    s.append(solicitud('delete_row_col', h.sheet_id, rango=f'A{fila_inicio_recorte}', cantidad=sobrantes, valor='ROWS'))
  return s

def solicitudes_limpieza(h):
  """Quita filtro, notas y colores anteriores y numera los RAPs."""
  fin = h.n_aprendices+h.fila_inicial-1
  return [
    solicitud('clear_filter',h.sheet_id),
    solicitud('delete_emerging_note',h.sheet_id,rango_rap(h)),
    solicitud('delete_emerging_note',h.sheet_id,'E3:F6'),
    solicitud('delete_cells',h.sheet_id,'E3:F6'),
    solicitud('color_cells',h.sheet_id,'E3:F6',valor=COLOR_ESTADOS['COLOR BASE']),
    solicitud('color_cells',h.sheet_id,f'A{h.fila_inicial}:G{fin}',valor=COLOR_ESTADOS['EN FORMACION']),
    solicitud('color_cells',h.sheet_id,rango_rap(h),valor=[x/255 for x in [255, 255, 255, 0]]),
    solicitud('write_numbers_to_range',h.sheet_id,f'{h.columnas[0]}{h.fila_inicial-2}:{h.columnas[-1]}{h.fila_inicial-1}',valor=list(range(1,h.n_RAs+1))),
  ]

def solicitudes_encabezado(h, ficha, datos_ficha, gtime, est):
  """Programa, ficha, fechas y estado de la ficha (celdas A1 y D3:D6) con sus notas (A1, D4 y D6 vienen de las funciones de la sección 3.1b)."""
  etapa = est.etapa
  return [
    solicitud('update_cells',h.sheet_id,ga('A',1),valor=datos_ficha['programa']),
    solicitud('emerging_note',h.sheet_id,ga('A',1),valor=nota_programa(datos_ficha, h.n_RAs, len(est.productivos))),
    solicitud('update_cells',h.sheet_id,ga('D',3),valor=ficha),
    solicitud('emerging_note',h.sheet_id,ga('D',3),valor=f'TOTAL DE APRENDICES EN LA FICHA: {h.n_vigentes}'),
    solicitud('update_cells',h.sheet_id,ga('D',4),valor=datos_ficha['fecha_reporte']),
    solicitud('emerging_note',h.sheet_id,ga('D',4),valor=nota_fecha_reporte(etapa, gtime, est.ultimo_ficha, (h.f_hoja_anterior, h.conservadas) if h.reporte_anterior else None, lineas_corte(datos_ficha))),
    solicitud('update_cells',h.sheet_id,ga('D',5),valor=datos_ficha['fecha_inicio']),
    solicitud('update_cells',h.sheet_id,ga('D',6),valor=datos_ficha['fecha_fin']),
    solicitud('emerging_note',h.sheet_id,ga('D',6),valor=nota_fecha_fin(etapa)),
  ]

def solicitudes_columnas_rap(h, tabla_ra, competencies):
  """Encabezado de cada columna de RAP: competencia (nota, número y color) y RAP (nota y color)."""
  s = []
  for k in range(h.n_RAs):
    col = h.columnas[k]
    competencia_actual = tabla_ra.Competencias[k]

    # Manejo dinámico y robusto: si la competencia no está en la lista de competencias conocidas, se agrega de inmediato
    if competencia_actual not in competencies:
        competencies.append(competencia_actual)
        competencies.sort()

    posicion = competencies.index(competencia_actual)
    color = [x/255 for x in COLORES_COMPETENCIA[posicion % len(COLORES_COMPETENCIA)]]

    s.append(solicitud('emerging_note',h.sheet_id,ga(col,5),valor=competencia_actual))
    s.append(solicitud('emerging_note',h.sheet_id,ga(col,6),valor=f'RAP {tabla_ra.RA[k]}'))
    s.append(solicitud('color_cells',h.sheet_id,ga(col,5),valor=color))
    s.append(solicitud('update_cells',h.sheet_id,ga(col,5),valor=(1+posicion)))
    s.append(solicitud('color_cells',h.sheet_id,ga(col,6),valor=color))
    s.append(solicitud('delete_cells',h.sheet_id,rango_rap(h)))
  return s

def solicitudes_notas_titulos(h, est, tabla_ra, competencias):
  """Notas de los títulos G5 (lista de competencias) y G6 (resumen de RAPs)."""
  return [solicitud('emerging_note',h.sheet_id,ga('G',5),valor=nota_competencias(est, tabla_ra, competencias, h.n_RAs)),
          solicitud('emerging_note',h.sheet_id,ga('G',6),valor=nota_raps(est, h.n_RAs))]

def solicitudes_notas_columnas(h, est, notas):
  """Notas de las filas 2 (resumen del RAP) y 7 (evaluados/total) de cada columna con evaluadores."""
  s = []
  for col, texto in notas.items():
    s.append(solicitud('emerging_note',h.sheet_id,ga(col,2),valor=texto))
    s.append(solicitud('emerging_note',h.sheet_id,ga(col,7),valor=f'{est.evaluados[col]}/{h.n_vigentes}'))
  return s

ESTADOS_COLOR_FILA = [e for e in ORDEN_ESTADOS if e != 'EN FORMACION']   # 'EN FORMACION' conserva el color base de la fila
ESTADOS_ACTIVOS = ['INDUCCION', 'EN FORMACION', 'CONDICIONADO']          # aprendices a los que se les hace seguimiento de pendientes

def texto_juicio(juicio, funcionario, fecha):
  """Nota de la celda de juicio, p. ej. 'APROBADO por Natalia Marin Otalvaro - 20 Jun 2024'.
  Un POR EVALUAR con funcionario asignado es un levantamiento (un NO APROBADO que revirtió a pendiente): 'Juicio LEVANTADO por ...'."""
  nombre = funcionario.partition('-')[2].strip().title()
  etiqueta = 'Juicio LEVANTADO' if juicio == 'POR EVALUAR' else juicio
  fecha_txt = 'sin fecha'
  if isinstance(fecha, str) and fecha.strip():
    f = pd.to_datetime(fecha, errors='coerce')
    if not pd.isna(f):
      fecha_txt = f"{f.day} {MESES_ABREV[f.month-1]} {f.year}"
  return f"{etiqueta} por {nombre} - {fecha_txt}"

def solicitudes_aprendices(h, dfRA, est, tabla_ra, inicios, cantidades, notas_guardadas, conservar=None):
  """Fila de cada aprendiz (datos, colores por juicio, porcentaje y notas). Devuelve (solicitudes, informe G7)."""
  s = []
  aviso = '\n'.join(([getattr(h, 'linea_corte', '')] if getattr(h, 'linea_corte', '') else []) +
                    ([f"⚠️ REPORTE ANTERIOR: la hoja tenía datos del {fecha_corta(h.f_hoja_anterior)}. Este informe usa solo este reporte."]
                     if h.reporte_anterior else []))
  informe = encabezado_informe_pendientes(est, h.ficha, aviso)
  h.conservadas = 0
  # Solo los aprendices vigentes (los históricos 'NO EN FICHA' se escriben aparte, al final de la tabla)
  for i in range(len(inicios)):
    p = str(i+h.fila_inicial)
    m, cantidad_RAs_aprendiz = inicios[i], cantidades[i]
    documento_aprendiz = str(dfRA.loc[m,'Número de Documento'])

    if dfRA.loc[m,'Estado'] in ESTADOS_COLOR_FILA:
      s.append(solicitud('color_cells',h.sheet_id,f'A{p}:G{p}',valor=COLOR_ESTADOS[dfRA.loc[m,'Estado']]))
    s.append(solicitud('update_cells',h.sheet_id,ga('A',p),valor=(i+1)))
    s.append(solicitud('update_cells',h.sheet_id,ga('B',p),valor=dfRA.loc[m,'Tipo de Documento'] ))
    s.append(solicitud('update_cells',h.sheet_id,ga('C',p),valor=dfRA.loc[m,'Número de Documento']))
    s.append(solicitud('update_cells',h.sheet_id,ga('D',p),valor=dfRA.loc[m,'Nombre']))
    s.append(solicitud('update_cells',h.sheet_id,ga('E',p),valor=dfRA.loc[m,'Apellidos']))
    s.append(solicitud('update_cells',h.sheet_id,ga('F',p),valor=dfRA.loc[m,'Estado']))

    RA_aprobados = 0
    for j, fila in recorrer_raps(dfRA, m, h.n_RAs):
      col = h.columnas[j]
      # Reporte anterior a la hoja: si la hoja tenía una nota distinta (o el reporte no trae juicio), se conserva la de la hoja en gris
      vieja = conservar.get(normalizar_documento(documento_aprendiz), {}).get(col) if conservar else None
      if vieja:
        trae_juicio = fila is not None and len(dfRA.loc[fila,'Funcionario que registro el juicio evaluativo']) > 7
        nueva = (texto_juicio(dfRA.loc[fila,'Juicio de Evaluación'], dfRA.loc[fila,'Funcionario que registro el juicio evaluativo'],
                              dfRA.loc[fila,'Fecha y Hora del Juicio Evaluativo']) if trae_juicio else None)
        if nueva is None or ' '.join(nueva.split()).casefold() != ' '.join(vieja.split()).casefold():
          s.append(solicitud('emerging_note',h.sheet_id,ga(col,p),valor=vieja))
          s.append(solicitud('color_cells',h.sheet_id,ga(col,p),valor=COLOR_ESTADOS['NO EN FICHA']))
          h.conservadas += 1
          if trae_juicio and dfRA.loc[fila,'Juicio de Evaluación'] == 'APROBADO':
            RA_aprobados = RA_aprobados + 1   # el porcentaje se calcula solo con el reporte
          continue
      if fila is None:
        s.append(solicitud('color_cells',h.sheet_id,ga(col,p),valor=COLOR_ESTADOS['NA']))
        continue
      if len(dfRA.loc[fila,'Funcionario que registro el juicio evaluativo']) > 7:
        funcionario = dfRA.loc[fila,'Funcionario que registro el juicio evaluativo']
        fecha = dfRA.loc[fila,'Fecha y Hora del Juicio Evaluativo']
        funcionario_evaluador = funcionario.split('-')[1].strip()
        s.append(solicitud('emerging_note',h.sheet_id,ga(col,p),valor=texto_juicio(dfRA.loc[fila,'Juicio de Evaluación'], funcionario, fecha)))
        if dfRA.loc[fila,'Juicio de Evaluación'] == 'APROBADO':
          RA_aprobados = RA_aprobados + 1
          s.append(solicitud('color_cells',h.sheet_id,ga(col,p),valor=est.colores_aprobados[col][funcionario_evaluador]))
        elif dfRA.loc[fila,'Juicio de Evaluación'] == 'POR EVALUAR':
          s.append(solicitud('color_cells',h.sheet_id,ga(col,p),valor=COLOR_ESTADOS['POR EVALUAR']))
        else:
          s.append(solicitud('color_cells',h.sheet_id,ga(col,p),valor=COLOR_ESTADOS['NO APROBADO']))

    porEvaluado = round(RA_aprobados/cantidad_RAs_aprendiz*100)
    s.append(solicitud('update_cells',h.sheet_id,ga('G',p),valor=f'{porEvaluado}%'))
    # Color de la celda de porcentaje según RAPs pendientes (solo activos); no activos en gris
    n_pendientes = len(est.pendientes_fila.get(i, []))
    if dfRA.loc[m,'Estado'] not in ESTADOS_ACTIVOS:
      color_g = COLOR_ESTADOS[dfRA.loc[m,'Estado']]   # no activos: el color de su categoría
    elif n_pendientes == 0:
      color_g = COLOR_ESTADOS['AL DIA']
    elif n_pendientes == 1:
      color_g = COLOR_ESTADOS['PENDIENTES 1']
    elif n_pendientes == 2:
      color_g = COLOR_ESTADOS['PENDIENTES 2']
    else:
      color_g = COLOR_ESTADOS['PENDIENTES 3+']
    s.append(solicitud('color_cells',h.sheet_id,ga('G',p),valor=color_g))
    nota, fragmento = textos_aprendiz(dfRA.loc[m,'Nombre'], dfRA.loc[m,'Apellidos'], porEvaluado, RA_aprobados,
                                      cantidad_RAs_aprendiz, est.pendientes_fila.get(i), est, tabla_ra, dfRA.loc[m,'Estado'], i=i)
    informe += fragmento
    s.append(solicitud('emerging_note',h.sheet_id,ga('G',p),valor=recortar_nota(nota)))
    if notas_guardadas:
      s.extend(build_note_solicitud(h.sheet_id, documento_aprendiz, p, notas_guardadas))
  return s, recortar_nota(informe)

def solicitudes_resumen_estados(h, est):
  """Resumen de aprendices por estado en E3:F6 (texto, nota y color)."""
  s = []
  iter_celdasEstados = iter(CELDAS_ESTADOS)
  aviso = (f"\n\n⚠️ REPORTE ANTERIOR: la hoja tenía datos del {fecha_corta(h.f_hoja_anterior)}. Las estadísticas son solo de este reporte; "
           f"{h.conservadas} juicios de la hoja que este reporte no trae igual se conservan en gris.") if h.reporte_anterior else ''
  primera = True
  for key,value in est.valor_estados.items():
    if value > 0:
      celda = next(iter_celdasEstados)
      KEY = key
      if key =='CONDICIONADO' or key =='CANCELADO' or key =='APLAZADO' or key =='TRASLADADO' or key =='CERTIFICADO':
        KEY = key + 'S'
      s.append(solicitud('update_cells',h.sheet_id,celda,valor=f'{KEY}: {value}'))
      s.append(solicitud('emerging_note',h.sheet_id,celda,valor=f'{round(value/h.n_vigentes*100)}% de {h.n_vigentes}' + (aviso if primera else '')))
      primera = False
      s.append(solicitud('color_cells',h.sheet_id,celda,valor=COLOR_ESTADOS[key]))
  # 'NO EN FICHA' ocupa la siguiente celda libre del rango dinámico E3:F6; sin nota ni porcentaje (no hace parte de la estadística del reporte)
  if h.n_historicos > 0:
    celda = next(iter_celdasEstados, None)
    if celda:
      s.append(solicitud('update_cells',h.sheet_id,celda,valor=f'NO EN FICHA: {h.n_historicos}'))
      nota_historicos = ("⚠️ APRENDICES QUE NO FIGURAN EN ESTE REPORTE\n\n"
                         "Estos aprendices están en la hoja, pero no aparecen en el reporte de SOFIA Plus que se cargó. "
                         "Posiblemente porque el reporte es de una fecha anterior a la de la hoja.\n\n"
                         "Se conserva su avance anterior solo como referencia: no entra en los porcentajes, los pendientes ni las estadísticas de este reporte.")
      if h.nombres_historicos:
        nota_historicos += "\n\n" + "\n".join(f" ▪ {n}" for n in h.nombres_historicos)
      s.append(solicitud('emerging_note',h.sheet_id,celda,valor=nota_historicos))
      s.append(solicitud('color_cells',h.sheet_id,celda,valor=COLOR_ESTADOS['NO EN FICHA']))
  return s

def normalizar_documento(valor):
  """Documento de identidad como texto comparable: sin espacios ni '.0' final."""
  if isinstance(valor, float) and valor.is_integer():
    valor = int(valor)
  texto = str(valor).strip()
  if texto.endswith('.0'):
    texto = texto[:-2]
  return texto

def clave_rap(texto):
  """(texto completo, nombre) de un RAP a partir de 'RAP 1671 - 2. ANALIZAR...' o '1671 - 2. ANALIZAR...'.
  El nombre no cambia aunque cambie el código, por eso sirve de respaldo para ubicar la columna."""
  texto = str(texto).strip()
  if texto.upper().startswith('RAP '):
    texto = texto[4:].strip()
  _, _, nombre = texto.partition(' - ')
  return texto, (nombre or texto).strip().upper()

def capturar_texto_manual(spreadsheet_id, sheet_name, start_row=FILA_INICIAL):
  """Texto (o números) escrito a mano en la tabla de RAPs (desde H{start_row} hacia abajo y a la derecha), leído ANTES de modificar la hoja.
  Cada dato se identifica por el documento del aprendiz (columna C) y por el RAP (nota de la fila start_row-2).
  Retorna {documento: {texto del RAP: valor}}. Si falla la lectura se propaga el error (para no borrar datos sin haberlos guardado)."""
  texto_manual = {}
  metadata = sheets_service.spreadsheets().get(spreadsheetId=spreadsheet_id).execute()
  props = next(s['properties'] for s in metadata['sheets'] if s['properties']['title'] == sheet_name)
  max_rows = props['gridProperties']['rowCount']
  max_cols = props['gridProperties']['columnCount']
  if max_rows < start_row or max_cols <= 7:
    return texto_manual

  fila_rap = start_row - 2   # fila con la nota 'RAP <código> - <nombre>' de cada columna
  ultima_col = generate_column_names('H', max_cols - 7)[-1]
  respuesta = sheets_service.spreadsheets().get(
    spreadsheetId=spreadsheet_id,
    ranges=[f"'{sheet_name}'!A{fila_rap}:{ultima_col}{max_rows}"],
    fields='sheets.data.rowData.values(note,userEnteredValue)'
  ).execute()
  datos = respuesta.get('sheets', [])
  if not datos or 'data' not in datos[0]:
    return texto_manual
  filas = datos[0]['data'][0].get('rowData', [])
  if not filas:
    return texto_manual

  claves = {}   # índice de celda (base 0 desde la columna A) -> texto del RAP
  for idx, celda in enumerate(filas[0].get('values', [])):
    if idx >= 7 and celda.get('note', '').strip():
      claves[idx] = clave_rap(celda['note'])[0]

  for fila in filas[start_row - fila_rap:]:
    valores = fila.get('values', [])
    if len(valores) <= 7:
      continue
    celda_doc = valores[2].get('userEnteredValue', {})
    documento = normalizar_documento(celda_doc.get('stringValue', celda_doc.get('numberValue', '')))
    if documento in ('', '0'):
      continue
    for idx, texto_rap in claves.items():
      if idx >= len(valores):
        continue
      celda = valores[idx].get('userEnteredValue', {})
      if 'stringValue' in celda:
        valor = celda['stringValue']
        if not valor.strip():
          continue
      elif 'numberValue' in celda:
        valor = celda['numberValue']
      else:
        continue
      texto_manual.setdefault(documento, {})[texto_rap] = valor
  return texto_manual

def columnas_por_rap(h, tabla_ra):
  """Columna de la hoja nueva por RAP: ({texto completo: columna}, {nombre: columna})."""
  col_por_texto, col_por_nombre = {}, {}
  for k in range(h.n_RAs):
    texto, nombre = clave_rap(tabla_ra.RA[k])
    col_por_texto[texto] = h.columnas[k]
    col_por_nombre[nombre] = h.columnas[k]
  return col_por_texto, col_por_nombre

def buscar_columna_rap(texto_rap, col_por_texto, col_por_nombre):
  """Columna de un RAP: primero por texto completo; si cambió el código, por nombre. None si no existe."""
  return col_por_texto.get(texto_rap) or col_por_nombre.get(clave_rap(texto_rap)[1])

def solicitudes_texto_manual(h, tabla_ra, texto_manual, filas_por_documento, filas_historicos=()):
  """Vuelve a escribir el texto manual en la fila (por documento) y la columna (por RAP) que le corresponden en la hoja nueva.
  Los que no se puedan ubicar se avisan en consola. En filas de aprendices históricos la celda se pinta de gris."""
  s = []
  if not texto_manual:
    return s
  col_por_texto, col_por_nombre = columnas_por_rap(h, tabla_ra)
  sin_ubicar = []
  for documento, raps in texto_manual.items():
    fila = filas_por_documento.get(documento)
    for texto_rap, valor in raps.items():
      columna = buscar_columna_rap(texto_rap, col_por_texto, col_por_nombre)
      if fila is None or columna is None:
        sin_ubicar.append(f"documento {documento}, RAP '{texto_rap}': {valor!r}")
        continue
      s.append(solicitud('update_cells', h.sheet_id, ga(columna, fila), valor=valor))
      if fila in filas_historicos:
        s.append(solicitud('color_cells', h.sheet_id, ga(columna, fila), valor=COLOR_ESTADOS['NO EN FICHA']))
  if sin_ubicar:
    print("⚠️ Texto manual que no se pudo ubicar en la hoja nueva (no se restauró):\n   " + "\n   ".join(sin_ubicar))
  return s


## 4. Ejecución

**4.1 Cargar datos:** órdenes de RAPs y juicios de Sofia (carpeta o, si está vacía, un archivo Excel).

In [ ]:
# ============================================================
# EJECUCIÓN INICIAL — Asegurar Carpeta Raíz primero
# ============================================================

try:
    # 1. Obtener la Carpeta Raíz de Instalación vinculada (si falta o se pidió cambiarla, se abre el panel gráfico)
    carpeta_raiz_instalacion_id = obtener_o_vincular_raiz_instalacion()

    # 2. Cargar órdenes de RAPs
    ordenRA = cargar_ordenes_ra()

    # 3. Cargar datos de fichas desde Sofia
    list_competencias = {}
    data_fichas, id_juicios = cargar_fichas_sofia(ordenRA, list_competencias)

    # 4. Inicialización pasiva y 100% silenciosa en caso de no encontrarse archivos
    if not id_juicios:
        data_fichas = {}
        id_juicios = {}
except (RaizPendiente, ConflictoCarpeta) as e_vinculo:
    # Falta resolver una vinculación de carpetas: se omite la precarga (el panel gráfico permite resolverlo y
    # la interfaz de la sección 5 vuelve a cargar todo al procesar).
    print(f"⚠️ {e_vinculo}")
    carpeta_raiz_instalacion_id = globals().get('carpeta_raiz_instalacion_id')
    ordenRA, list_competencias, data_fichas, id_juicios = {}, {}, {}, {}

**4.2 Preparar hojas de salida:** plantillas y copia para las fichas nuevas.

In [ ]:
def get_folder_path_recursive(folder_id, drive_service):
    path = []
    try:
        current_folder = drive_service.files().get(fileId=folder_id, fields='name, parents').execute()
        parents = current_folder.get('parents')
        if parents:
            for parent_id in parents:
                parent_path = get_folder_path_recursive(parent_id, drive_service)
                if parent_path:
                    path.extend(parent_path)
        path.append(current_folder['name'])
    except Exception:
        pass
    return path

def get_drive_folder_path(folder_id, drive_service):
    path_list = get_folder_path_recursive(folder_id, drive_service)
    if path_list:
        return " > ".join(path_list)
    return "Desconocido"

def preparar_hojas_salida(data_fichas, ids_plantillas):
  global nom, metadata_carpetas_salida

  sheets_ids = {}
  ids = {}
  rowsColumnsCounts = {}

  # Aseguramos el ID de la carpeta raíz de instalación al inicio
  global carpeta_raiz_instalacion_id
  if 'carpeta_raiz_instalacion_id' not in globals() or not carpeta_raiz_instalacion_id:
      carpeta_raiz_instalacion_id = obtener_o_vincular_raiz_instalacion()

  for key, info in data_fichas.items():
    codigo_programa = info['codigo']
    nom_programa = extraer_iniciales(info['programa'])
    nom = nom_programa

    # Ajuste: Pasar directamente 'carpeta_raiz_instalacion_id' en lugar de buscar/crear carpeta del programa
    folder_etapa_id, nombre_etapa_carpeta = determinar_etapa_carpeta_id(carpeta_raiz_instalacion_id, nom_programa, info['fecha_fin'])

    query = f"(name = '{nom_programa}-{key}' or name contains '{nom_programa}-{key} ') and mimeType='application/vnd.google-apps.spreadsheet' and trashed = false"
    search_result = drive_service.files().list(q=query, fields="files(id, name, parents)").execute()
    items_origen = search_result.get('files', [])

    existing_file = None
    operacion = "actualizado"
    if items_origen:
        existing_file = items_origen[0]
        file_id = existing_file['id']

        actual_parents = existing_file.get('parents', [])
        if folder_etapa_id not in actual_parents:
            drive_service.files().update(
                fileId=file_id,
                addParents=folder_etapa_id,
                removeParents=",".join(actual_parents),
                fields='id, parents'
            ).execute()
    else:
        operacion = "creado"
        if codigo_programa in ids_plantillas.keys():
          # Si el programa tiene su propia plantilla oficial, la copiamos normalmente
          id_plantilla = ids_plantillas[codigo_programa][str(info['cantidad_RAs'])]
          copied_file = drive_service.files().copy(
            fileId=id_plantilla,
            body={
              'name': f'{nom_programa}-{key}',
              'parents': [folder_etapa_id]
            }
          ).execute()
          file_id = copied_file['id']

          sheet_metadata = sheets_service.spreadsheets().get(spreadsheetId=file_id).execute()
          sheet_id = sheet_metadata['sheets'][0]['properties']['sheetId']
          request_body = {
            'requests': [{
              'updateSheetProperties': {
                'properties': {
                  'sheetId': sheet_id,
                  'title': key
                },
                'fields': 'title'
              }
            }]
          }
          sheets_service.spreadsheets().batchUpdate(spreadsheetId=file_id, body=request_body).execute()
        else:
          # Si no tiene plantilla oficial, creamos un Spreadsheet nuevo vacío y aplicamos la plantilla embebida estructurada en memoria
          new_spreadsheet = drive_service.files().create(
            body={
              'name': f'{nom_programa}-{key}',
              'mimeType': 'application/vnd.google-apps.spreadsheet',
              'parents': [folder_etapa_id]
            },
            fields='id'
          ).execute()
          file_id = new_spreadsheet['id']

          # Recrear la estructura de la pestaña directamente
          crear_hoja_desde_plantilla_embebida(file_id, key)

          # Eliminar opcionalmente la hoja por defecto vacía ('Hoja 1' o 'Sheet1') para mantener limpia la salida
          try:
             meta_tmp = sheets_service.spreadsheets().get(spreadsheetId=file_id).execute()
             for s in meta_tmp.get('sheets', []):
                 if s['properties']['title'] in ['Hoja 1', 'Sheet1']:
                     sheets_service.spreadsheets().batchUpdate(
                         spreadsheetId=file_id,
                         body={'requests': [{'deleteSheet': {'sheetId': s['properties']['sheetId']}}]}
                     ).execute()
          except Exception:
             pass

    sheet_metadata = sheets_service.spreadsheets().get(spreadsheetId=file_id).execute()
    sheet = sheet_metadata.get('sheets', '')
    sheets_ids[key] = sheet[0]['properties']['sheetId']
    ids[key] = file_id
    rowsColumnsCounts[key] = sheet[0]['properties']['gridProperties']

    metadata_carpetas_salida[key] = {
        'operacion': operacion,
        'nombre_archivo': f'{nom_programa}-{key}',
        'ruta_carpeta': f"Juicios Evaluativos tabulados > {nombre_etapa_carpeta}",
        'link_carpeta': f"https://drive.google.com/drive/folders/{folder_etapa_id}"
    }

  return sheets_ids, ids, rowsColumnsCounts

### 4.4 Herramienta para Lectura de Metadatos de Carpetas o Archivos
Ejecuta esta celda e ingresa el ID de cualquier carpeta o archivo de Google Drive para validar sus propiedades personalizadas (`properties` / `appProperties`).

In [ ]:
def get_folder_path_recursive(folder_id, drive_service):
    path = []
    try:
        current_folder = drive_service.files().get(fileId=folder_id, fields='name, parents').execute()
        parents = current_folder.get('parents')
        if parents:
            for parent_id in parents:
                parent_path = get_folder_path_recursive(parent_id, drive_service)
                if parent_path:
                    path.extend(parent_path)
        path.append(current_folder['name'])
    except Exception:
        pass
    return path

def get_drive_folder_path(folder_id, drive_service):
    path_list = get_folder_path_recursive(folder_id, drive_service)
    if path_list:
        return " > ".join(path_list)
    return "Desconocido"

def preparar_hojas_salida(data_fichas, ids_plantillas):
  global nom, metadata_carpetas_salida, ID_PLANTILLA_GENERICA

  sheets_ids = globals().get('sheets_ids', {})
  ids = globals().get('ids', {})
  rowsColumnsCounts = globals().get('rowsColumnsCounts', {})

  # Aseguramos el ID de la carpeta raíz de instalación al inicio
  global carpeta_raiz_instalacion_id
  if 'carpeta_raiz_instalacion_id' not in globals() or not carpeta_raiz_instalacion_id:
      carpeta_raiz_instalacion_id = obtener_o_vincular_raiz_instalacion()

  # Obtener carpeta de plantillas
  carpeta_plantillas_id = obtener_o_vincular_carpeta_plantillas()

  for key, info in data_fichas.items():
    codigo_programa = info['codigo']
    nom_programa = extraer_iniciales(info['programa'])
    nom = nom_programa

    # Determinar la carpeta de la etapa lectiva o productiva correspondiente
    folder_etapa_id, nombre_etapa_carpeta = determinar_etapa_carpeta_id(carpeta_raiz_instalacion_id, nom_programa, info['fecha_fin'], info.get('fecha_reporte'))

    # CONTROL DE SEGURIDAD: Validar si el ID registrado en memoria realmente existe en Drive
    file_id = ids.get(key)
    archivo_valido = False
    if file_id:
        try:
            drive_service.files().get(fileId=file_id, fields='id').execute()
            archivo_valido = True
        except Exception:
            archivo_valido = False
            if key in ids: del ids[key]
            if key in sheets_ids: del sheets_ids[key]
            if key in rowsColumnsCounts: del rowsColumnsCounts[key]

    query = f"(name = '{nom_programa}-{key}' or name contains '{nom_programa}-{key} ') and mimeType='application/vnd.google-apps.spreadsheet' and trashed = false"
    search_result = drive_service.files().list(q=query, fields="files(id, name, parents)").execute()
    items_origen = search_result.get('files', [])

    existing_file = None
    operacion = "actualizado"
    if items_origen and archivo_valido:
        existing_file = items_origen[0]
        file_id = existing_file['id']

        actual_parents = existing_file.get('parents', {})
        if folder_etapa_id not in actual_parents:
            drive_service.files().update(
                fileId=file_id,
                addParents=folder_etapa_id,
                removeParents=",".join(actual_parents),
                fields='id, parents'
            ).execute()
    else:
        operacion = "creado"
        # Determinar qué plantilla usar: específica del programa o la genérica
        if codigo_programa in ids_plantillas.keys() and str(info['cantidad_RAs']) in ids_plantillas[codigo_programa]:
          id_plantilla = ids_plantillas[codigo_programa][str(info['cantidad_RAs'])]
        else:
          # Resolver dinámicamente la plantilla genérica
          if not ID_PLANTILLA_GENERICA:
              ID_PLANTILLA_GENERICA = obtener_id_plantilla_generica_dinamica()

          # Verificar si la plantilla genérica física existe en Drive
          plantilla_existe = False
          if ID_PLANTILLA_GENERICA:
              try:
                  drive_service.files().get(fileId=ID_PLANTILLA_GENERICA, fields='id').execute()
                  plantilla_existe = True
              except Exception:
                  plantilla_existe = False

          # En caso de no encontrar la plantilla física, la generamos automáticamente en Drive
          if not plantilla_existe:
              print(f"⚠️ No se detectó plantilla genérica física en Drive. Creándola dinámicamente en la carpeta de plantillas...")
              nueva_plantilla_metadata = {
                  'name': 'Plantilla Generica Juicios Evaluativos',
                  'mimeType': 'application/vnd.google-apps.spreadsheet',
                  'parents': [carpeta_plantillas_id]
              }
              nueva_plantilla = drive_service.files().create(body=nueva_plantilla_metadata, fields='id').execute()
              ID_PLANTILLA_GENERICA = nueva_plantilla['id']

              # Inyectar estructura y estilos G123 a la nueva plantilla
              generar_plantilla_juicios_evaluativos(ID_PLANTILLA_GENERICA, {
                  'PROGRAMA': 'MANTENIMIENTO ELECTRÓNICO E INSTRUMENTAL INDUSTRIAL',
                  'TRIMESTRE': 'TRI',
                  'TEMATICAS': 'TEMÁTICAS'
              })
              print(f"📌 ¡Nueva plantilla genérica guardada físicamente en Drive con ID: {ID_PLANTILLA_GENERICA}!")

          id_plantilla = ID_PLANTILLA_GENERICA

        copied_file = drive_service.files().copy(
          fileId=id_plantilla,
          body={
            'name': f'{nom_programa}-{key}',
            'parents': [folder_etapa_id]
          }
        ).execute()
        file_id = copied_file['id']

        sheet_metadata = sheets_service.spreadsheets().get(spreadsheetId=file_id).execute()
        sheet_id = sheet_metadata['sheets'][0]['properties']['sheetId']
        request_body = {
          'requests': [{
            'updateSheetProperties': {
              'properties': {
                'sheetId': sheet_id,
                'title': key
              },
              'fields': 'title'
            }
          }]
        }
        sheets_service.spreadsheets().batchUpdate(spreadsheetId=file_id, body=request_body).execute()

    sheet_metadata = sheets_service.spreadsheets().get(spreadsheetId=file_id).execute()
    sheet = sheet_metadata.get('sheets', '')
    sheets_ids[key] = sheet[0]['properties']['sheetId']
    ids[key] = file_id
    rowsColumnsCounts[key] = sheet[0]['properties']['gridProperties']

    metadata_carpetas_salida[key] = {
        'operacion': operacion,
        'nombre_archivo': f'{nom_programa}-{key}',
        'ruta_carpeta': f"Juicios Evaluativos tabulados > {nombre_etapa_carpeta}",
        'link_carpeta': f"https://drive.google.com/drive/folders/{folder_etapa_id}",
        'folder_id': folder_etapa_id,
        'nombre_carpeta_final': nombre_etapa_carpeta
    }

  globals()['sheets_ids'] = sheets_ids
  globals()['ids'] = ids
  globals()['rowsColumnsCounts'] = rowsColumnsCounts
  return sheets_ids, ids, rowsColumnsCounts

# Diccionario global para acumular los resultados de los archivos procesados y reportarlos de forma agrupada al final
reporte_procesamiento = {}

def actualizar_hoja(ficha, body, gtime, datos_ficha):
  """Envía las solicitudes a Google Sheets, acumula metadatos para el reporte consolidado y mueve los archivos."""
  try:
    sheets_service.spreadsheets().batchUpdate(spreadsheetId=ids[ficha], body={'requests': body}).execute()
    # Nombre del archivo de salida con la fecha y hora de esta actualización, p. ej. 'MEII-2396648 [25-10-2026 21:22:37]'
    try:
      nombre_nuevo = f"{extraer_iniciales(datos_ficha['programa'])}-{ficha} [{gtime.strftime('%d-%m-%Y %H:%M:%S')}]"
      drive_service.files().update(fileId=ids[ficha], body={'name': nombre_nuevo}).execute()
      if ficha in metadata_carpetas_salida:
        metadata_carpetas_salida[ficha]['nombre_archivo'] = nombre_nuevo
    except Exception as e_nombre:
      print(f"⚠️ No se pudo actualizar el nombre del archivo de la ficha {ficha}: {str(e_nombre)}")

    info_entrada = None
    # 1. ARCHIVO DE ENTRADA
    if id_juicios and ficha in id_juicios:
      codigo_programa = datos_ficha['codigo']
      nom_programa = extraer_iniciales(datos_ficha['programa'])
      file_id = id_juicios[ficha]

      file_original_metadata = drive_service.files().get(fileId=file_id, fields='name').execute()
      nombre_original = file_original_metadata.get('name', 'Archivo Desconocido')

      global carpeta_raiz_instalacion_id
      if 'carpeta_raiz_instalacion_id' not in globals() or not carpeta_raiz_instalacion_id:
          carpeta_raiz_instalacion_id = obtener_o_vincular_raiz_instalacion()

      # Carpeta de procesados: se reconoce por sus appProperties (donde esté y como se llame); si no existe, se crea
      folder_id = obtener_o_vincular_carpeta_procesados(codigo_programa, nom_programa)

      nuevo_nombre = f'{nom_programa}-Jui Eval-{ficha}-FA[{gtime.day}-{gtime.month}-{gtime.year}] [{gtime.strftime("%H:%M:%S")}]_Rdo'
      file_metadata = {
        'name': nuevo_nombre,
        'appProperties': {
          'codigo_programa': codigo_programa
        }
      }

      file_info = drive_service.files().get(fileId=file_id, fields='parents').execute()
      previous_parents = ",".join(file_info.get('parents', []))

      drive_service.files().update(
        fileId=file_id,
        body=file_metadata,
        addParents=folder_id,
        removeParents=previous_parents,
        fields='id, parents'
      ).execute()

      ruta_entrada_drive = get_drive_folder_path(folder_id, drive_service)
      info_entrada = {
          'nombre_original': nombre_original,
          'nuevo_nombre': nuevo_nombre,
          'ruta_drive': ruta_entrada_drive,
          'link_carpeta': f"https://drive.google.com/drive/folders/{folder_id}",
          'folder_id': folder_id
      }

    # 2. ARCHIVO DE SALIDA
    meta_salida = metadata_carpetas_salida.get(ficha, {})
    info_salida = None
    if meta_salida:
        salida_metadata = drive_service.files().get(fileId=ids[ficha], fields='parents').execute()
        parents_salida = salida_metadata.get('parents', [])
        if parents_salida:
            ruta_salida_drive = get_drive_folder_path(parents_salida[0], drive_service)
            link_carpeta_real = f"https://drive.google.com/drive/folders/{parents_salida[0]}"
            folder_final_id = parents_salida[0]
        else:
            ruta_salida_drive = "Raíz"
            link_carpeta_real = "https://drive.google.com/"
            folder_final_id = "root"

        info_salida = {
            'nombre': meta_salida['nombre_archivo'],
            'operacion': meta_salida['operacion'].upper(),
            'ruta_drive': ruta_salida_drive,
            'link_carpeta': link_carpeta_real,
            'folder_id': folder_final_id,
            'nombre_carpeta_final': meta_salida.get('nombre_carpeta_final', 'General'),
            'duplicados': meta_salida.get('duplicados', [])
        }

    # Guardamos toda la información indexada bajo el nombre del programa (en mayúsculas y limpio)
    prog_nombre = datos_ficha['programa'].strip().upper().rstrip('.')

    if prog_nombre not in reporte_procesamiento:
        reporte_procesamiento[prog_nombre] = {
            'salidas': {},
            'entradas': {}
        }

    # Agrupar salidas por carpeta de destino de etapa
    folder_id_key = info_salida['folder_id'] if info_salida else 'unknown'
    if folder_id_key not in reporte_procesamiento[prog_nombre]['salidas']:
        reporte_procesamiento[prog_nombre]['salidas'][folder_id_key] = {
            'nombre_carpeta_final': info_salida['nombre_carpeta_final'] if info_salida else 'Desconocida',
            'ruta_drive': info_salida['ruta_drive'] if info_salida else 'Desconocida',
            'link_carpeta': info_salida['link_carpeta'] if info_salida else '',
            'fichas': []
        }
    reporte_procesamiento[prog_nombre]['salidas'][folder_id_key]['fichas'].append({
        'ficha': ficha,
        'operacion': info_salida['operacion'] if info_salida else 'ACTUALIZADO',
        'link_archivo': f"https://docs.google.com/spreadsheets/d/{ids[ficha]}",
        'duplicados': info_salida['duplicados'] if info_salida else []
    })

    # Agrupar entradas por carpeta de procesados
    if info_entrada:
        ent_folder_id = info_entrada['folder_id']
        if ent_folder_id not in reporte_procesamiento[prog_nombre]['entradas']:
            reporte_procesamiento[prog_nombre]['entradas'][ent_folder_id] = {
                'ruta_drive': info_entrada['ruta_drive'],
                'link_carpeta': info_entrada['link_carpeta']
            }

  except Exception as e:
    print(f'   ❌ Error actualizando la hoja de cálculo de la ficha {ficha}: {str(e)}')
    raise e

def construir_solicitudes_ficha(ficha, datos_ficha, gtime):
  """Lista completa de solicitudes (en orden) que deja la hoja de la ficha actualizada e integra estudiantes históricos."""
  dfRA = datos_ficha['juicios']
  dfRA['TD_Documento'] = (dfRA['Tipo de Documento'] + dfRA['Número de Documento'].astype(str))

  # 1. Lectura estructural previa de los aprendices en Sheets antes de la sobreescritura
  aprendices_hoja_actual = obtener_aprendices_hoja_actual(ids[ficha], ficha, start_row=FILA_INICIAL)

  # 2. Identificación de alumnos históricos ausentes en el nuevo reporte ('NO EN FICHA')
  aprendices_no_en_ficha = identificar_aprendices_no_en_ficha(aprendices_hoja_actual, dfRA)

  n_aprendices_vigentes = datos_ficha['cantidad_aprendices']
  n_aprendices_historicos = len(aprendices_no_en_ficha)
  datos_ficha['historicos'] = n_aprendices_historicos
  n_aprendices_totales = n_aprendices_vigentes + n_aprendices_historicos

  n_RAs = datos_ficha['cantidad_RAs']
  codigo_programa = datos_ficha['codigo']

  h = SimpleNamespace(ficha=ficha, sheet_id=sheets_ids[ficha], columnas=generate_column_names('H', n_RAs),
                      n_aprendices=n_aprendices_totales, n_vigentes=n_aprendices_vigentes,
                      n_historicos=n_aprendices_historicos, n_RAs=n_RAs, fila_inicial=FILA_INICIAL,
                      nombres_historicos=[f"{v['nombres'].title()} {v['apellidos'].title()} ({v['estado']})" for v in aprendices_no_en_ficha.values()])
  tabla_ra = ordenRA[codigo_programa][str(n_RAs)]['RAs']
  h.conservadas = 0
  h.linea_corte = lineas_corte(datos_ficha)[0] if datos_ficha.get('corte') else ''

  # Reporte anterior a la hoja: la fecha D4 de la hoja se lee ANTES de sobrescribirla
  f_hoja = leer_fecha_reporte_hoja(ids[ficha], ficha)
  f_nueva = parsear_fecha(datos_ficha.get('fecha_reporte'))
  h.f_hoja_anterior = f_hoja
  h.reporte_anterior = bool(f_hoja and f_nueva and f_nueva < f_hoja)
  conservar = {}
  if h.reporte_anterior:
    print(f"   ⏪ Ficha {ficha}: el reporte ({fecha_corta(f_nueva)}) es anterior a los datos de la hoja ({fecha_corta(f_hoja)}). "
          "Se conservan en gris las notas de la hoja que el reporte no trae igual; las estadísticas usan solo este reporte.")
    col_por_texto, col_por_nombre = columnas_por_rap(h, tabla_ra)
    sin_ubicar = 0
    for doc, info_hoja in aprendices_hoja_actual.items():
      if doc in aprendices_no_en_ficha:
        continue
      for clave, celda_rap in info_hoja['juicios_rap'].items():
        if not celda_rap.get('nota'):
          continue
        col = buscar_columna_rap(clave, col_por_texto, col_por_nombre)
        if col is None and clave.startswith('#') and clave[1:] in h.columnas:
          col = clave[1:]
        if col is None:
          sin_ubicar += 1
          continue
        conservar.setdefault(normalizar_documento(doc), {})[col] = celda_rap['nota']
    if sin_ubicar:
      print(f"   ⚠️ {sin_ubicar} notas de la hoja no se pudieron ubicar por RAP (no se conservaron).")

  # Notas manuales de estudiantes vigentes que hay que conservar
  try:
    notas_guardadas = get_existing_notes(ids[ficha], ficha, start_row=h.fila_inicial)
  except Exception as e:
    notas_guardadas = {}
    print(f"⚠ No se pudieron recuperar notas previas para {ficha}: {str(e)}")

  # Texto manual de la tabla de RAPs (H8 en adelante), guardado antes de modificar la hoja
  texto_manual = capturar_texto_manual(ids[ficha], ficha, start_row=h.fila_inicial)

  inicios, cantidades = filas_por_aprendiz(dfRA, n_aprendices_vigentes)

  # Las estadísticas se calculan únicamente considerando a los aprendices vigentes
  # Etapa de la ficha a la fecha del reporte (D4): define qué RAPs son pendientes y cómo se explican en las notas
  etapa = info_ficha(datos_ficha)
  etapa.avisos_fechas = revisar_fechas_juicios(datos_ficha)
  etapa.advertencias.extend(etapa.avisos_fechas)
  for aviso in etapa.advertencias:
    print(f"   ⚠️ Ficha {ficha}: {aviso}")
  est = calcular_estadisticas(dfRA, n_aprendices_vigentes, n_RAs, h.columnas, tabla_ra, inicios, cantidades, etapa)
  notas_cols = notas_resumen_columnas(est, dfRA, tabla_ra, h.columnas, n_aprendices_vigentes)

  # El bloque de aprendices se calcula antes (cuenta cuántos juicios se conservan) y se agrega después del encabezado
  sol_aprendices, informe_aprendices = solicitudes_aprendices(h, dfRA, est, tabla_ra, inicios, cantidades, notas_guardadas, conservar)
  for nombre_instructor in est.instructores_total:   # deja registrado en el LOG a los instructores sin correo
    correo_de(nombre_instructor)

  body = []
  body += solicitudes_estructura(h, rowsColumnsCounts[ficha])
  body += solicitudes_limpieza(h)
  body += solicitudes_encabezado(h, ficha, datos_ficha, gtime, est)
  body += solicitudes_columnas_rap(h, tabla_ra, list_competencias[codigo_programa])
  body += solicitudes_notas_titulos(h, est, tabla_ra, list_competencias[codigo_programa])
  body += solicitudes_notas_columnas(h, est, notas_cols)

  # Escribir primero el bloque de aprendices vigentes en Sheets
  body += sol_aprendices

  # Escribir el bloque de aprendices históricos 'NO EN FICHA' al final de la tabla física
  if n_aprendices_historicos > 0:
      fila_inicio_historicos = h.fila_inicial + n_aprendices_vigentes
      sol_historicos = preparar_bloque_aprendices_no_en_ficha(h, aprendices_no_en_ficha, fila_inicio_historicos, est, tabla_ra)
      body += sol_historicos

  body.append(solicitud('emerging_note', h.sheet_id, ga('G', 7), valor=informe_aprendices))
  body.append(solicitud('emerging_note', h.sheet_id, ga('G', 2), valor=informe_instructores(est, tabla_ra)))
  body += solicitudes_resumen_estados(h, est)
  body.append(solicitud('set_filter', h.sheet_id, f'A{h.fila_inicial-1}:{h.columnas[-1]}{n_aprendices_totales+h.fila_inicial-1}'))
  # Reinsertar el texto manual: fila por documento de identidad, columna por RAP
  filas_por_documento = {normalizar_documento(dfRA.loc[m, 'Número de Documento']): h.fila_inicial + i for i, m in enumerate(inicios)}
  filas_historicos = set()
  for idx, doc_id in enumerate(aprendices_no_en_ficha):
    fila_historico = h.fila_inicial + n_aprendices_vigentes + idx
    filas_por_documento[normalizar_documento(doc_id)] = fila_historico
    filas_historicos.add(fila_historico)
  body += solicitudes_texto_manual(h, tabla_ra, texto_manual, filas_por_documento, filas_historicos)

  return body

def imprimir_reporte_consolidado():
  """Reporte compacto: el programa como título y, con sangría, las carpetas de salida (etapa) y de entrada, con enlaces."""
  from IPython.display import display, Markdown
  if not reporte_procesamiento:
      display(Markdown("### ⚠️ No se procesó ningún archivo."))
      return

  def enlace(texto, url, nota=''):
      """Enlace Markdown; la nota aparece al pasar el cursor sobre el enlace."""
      titulo = f' "{nota.replace(chr(34), chr(39))}"' if nota else ''
      return f"[{texto}]({url}{titulo})"

  bloques = []
  for prog_nombre, datos_prog in reporte_procesamiento.items():
      sublineas = []

      for folder_id, salida_info in datos_prog['salidas'].items():
          fichas_actualizadas = [enlace(f['ficha'], f['link_archivo'], 'Abrir en Drive') for f in salida_info['fichas'] if f['operacion'] == 'ACTUALIZADO']
          fichas_creadas = [enlace(f['ficha'], f['link_archivo'], 'Abrir en Drive') for f in salida_info['fichas'] if f['operacion'] == 'CREADO']

          etapa_nombre_upper = "GENERAL"
          if "LECTIVA" in salida_info['nombre_carpeta_final'].upper():
              etapa_nombre_upper = "Etapa LECTIVA"
          elif "PRODUCTIVA" in salida_info['nombre_carpeta_final'].upper():
              etapa_nombre_upper = "Etapa PRODUCTIVA"

          partes_operacion = []
          if fichas_actualizadas:
              txt_act = f"ACTUALIZADO para ficha {fichas_actualizadas[0]}" if len(fichas_actualizadas) == 1 else f"ACTUALIZADOS para fichas {', '.join(fichas_actualizadas[:-1])} y {fichas_actualizadas[-1]}"
              partes_operacion.append(txt_act)
          if fichas_creadas:
              txt_cre = f"CREADO para ficha {fichas_creadas[0]}" if len(fichas_creadas) == 1 else f"CREADOS para fichas {', '.join(fichas_creadas[:-1])} y {fichas_creadas[-1]}"
              partes_operacion.append(txt_cre)

          etapa_enlace = enlace(etapa_nombre_upper, salida_info['link_carpeta'], f"Ruta en Drive: {salida_info.get('ruta_drive', '')}")
          sublineas.append(f"🟢 Juicios {', '.join(partes_operacion)} en {etapa_enlace}")
          for f in salida_info['fichas']:
              for dup in f.get('duplicados', []):
                  sublineas.append(f"⚠️ Ficha {f['ficha']}: hay un archivo duplicado por nombre, " + enlace(dup['nombre'], dup['link'], 'Abrir en Drive')
                                   + (f" en {dup['carpeta']}" if dup.get('carpeta') else '') + ", que NO se está actualizando.")

      if datos_prog['entradas']:
          nom_iniciales = extraer_iniciales(prog_nombre)
          enlaces_entrada = " · ".join(enlace(f"{nom_iniciales} Guardados", e['link_carpeta'], f"Ruta en Drive: {e.get('ruta_drive', '')}") for e in datos_prog['entradas'].values())
          sublineas.append(f"📥 Juicios Evaluativos SOFIA Plus de {enlaces_entrada}")

      # Título y, con sangría por espacios, las líneas que dependen de él (en un solo párrafo)
      bloques.append(f"✨ **{prog_nombre}**  \n" + "  \n".join("&emsp;&emsp;" + l for l in sublineas))

  display(Markdown("\n\n---\n\n".join(bloques)))
  log_correos_instructores()


In [ ]:
import time

# Memoria caché para evitar consultas repetitivas a la API de Drive
_cache_carpetas_id = {}
_cache_rutas_carpetas = {}

def get_folder_path_recursive(folder_id, drive_service):
    if folder_id in _cache_rutas_carpetas:
        return _cache_rutas_carpetas[folder_id]
    path = []
    try:
        current_folder = drive_service.files().get(fileId=folder_id, fields='name, parents').execute()
        parents = current_folder.get('parents')
        if parents:
            for parent_id in parents:
                parent_path = get_folder_path_recursive(parent_id, drive_service)
                if parent_path:
                    path.extend(parent_path)
        path.append(current_folder['name'])
    except Exception:
        pass
    _cache_rutas_carpetas[folder_id] = path
    return path

def get_drive_folder_path(folder_id, drive_service):
    path_list = get_folder_path_recursive(folder_id, drive_service)
    if path_list:
        return " > ".join(path_list)
    return "Desconocido"

def preparar_hojas_salida(data_fichas, ids_plantillas):
  global nom, metadata_carpetas_salida, ID_PLANTILLA_GENERICA

  sheets_ids = globals().get('sheets_ids', {})
  ids = globals().get('ids', {})
  rowsColumnsCounts = globals().get('rowsColumnsCounts', {})

  # Aseguramos el ID de la carpeta raíz de instalación al inicio de forma limpia
  global carpeta_raiz_instalacion_id
  if 'carpeta_raiz_instalacion_id' not in globals() or not carpeta_raiz_instalacion_id:
      carpeta_raiz_instalacion_id = obtener_o_vincular_raiz_instalacion()

  # Obtener carpeta oficial de plantillas
  carpeta_plantillas_id = obtener_o_vincular_carpeta_plantillas()

  # Consultar en lote las carpetas de etapa ya existentes en la raíz para evitar queries repetitivas
  query_etapas = f"'{carpeta_raiz_instalacion_id}' in parents and mimeType='application/vnd.google-apps.folder' and trashed = false"
  existentes_raiz = drive_service.files().list(q=query_etapas, fields="files(id, name)").execute().get('files', [])
  dict_etapas_existentes = {f['name']: f['id'] for f in existentes_raiz}

  for key, info in data_fichas.items():
    codigo_programa = str(info['codigo'])
    nom_programa = extraer_iniciales(info['programa'])
    nom = nom_programa
    n_RAs = info['cantidad_RAs']
    n_RAs_str = str(n_RAs)

    # Determinar carpeta de destino según etapa usando nuestra caché en memoria
    nombre_etapa_carpeta = nombre_carpeta_etapa(nom_programa, info['fecha_fin'], info.get('fecha_reporte'))

    if nombre_etapa_carpeta in dict_etapas_existentes:
        folder_etapa_id = dict_etapas_existentes[nombre_etapa_carpeta]
    else:
        folder_etapa_id = obtener_o_crear_subcarpeta_etapa(carpeta_raiz_instalacion_id, nombre_etapa_carpeta)
        dict_etapas_existentes[nombre_etapa_carpeta] = folder_etapa_id

    # Evitar peticiones GET redundantes por cada archivo si ya lo tenemos indexado localmente
    file_id = ids.get(key)
    archivo_valido = False
    if file_id:
        archivo_valido = True  # Asumimos válido para ahorrar cuotas API

    query = f"(name = '{nom_programa}-{key}' or name contains '{nom_programa}-{key} ') and mimeType='application/vnd.google-apps.spreadsheet' and trashed = false"
    search_result = drive_service.files().list(q=query, fields="files(id, name, parents)").execute()
    items_origen = search_result.get('files', [])

    operacion = "actualizado"
    if items_origen and archivo_valido:
        existing_file = items_origen[0]
        file_id = existing_file['id']

        actual_parents = existing_file.get('parents', [])
        if folder_etapa_id not in actual_parents:
            drive_service.files().update(
                fileId=file_id,
                addParents=folder_etapa_id,
                removeParents=",".join(actual_parents),
                fields='id, parents'
            ).execute()
            time.sleep(0.5)  # Espaciado de seguridad
    else:
        operacion = "creado"

        if codigo_programa not in ids_plantillas:
            ids_plantillas[codigo_programa] = {}

        if n_RAs_str not in ids_plantillas[codigo_programa]:
            if not ID_PLANTILLA_GENERICA:
                ID_PLANTILLA_GENERICA = obtener_id_plantilla_generica_dinamica()

            if not ID_PLANTILLA_GENERICA:
                # Crear plantilla física genérica fallback únicamente de ser necesario
                nueva_plantilla_metadata = {
                    'name': 'Plantilla Generica Juicios Evaluativos',
                    'mimeType': 'application/vnd.google-apps.spreadsheet',
                    'parents': [carpeta_plantillas_id]
                }
                nueva_plantilla = drive_service.files().create(body=nueva_plantilla_metadata, fields='id').execute()
                ID_PLANTILLA_GENERICA = nueva_plantilla['id']
                generar_plantilla_juicios_evaluativos(ID_PLANTILLA_GENERICA, {
                    'PROGRAMA': info['programa'].upper(),
                    'TRIMESTRE': 'TRI',
                    'TEMATICAS': 'TEMÁTICAS'
                })
                time.sleep(1.0)

            # Copiar genérica para crear plantilla específica
            copia_especifica = drive_service.files().copy(
                fileId=ID_PLANTILLA_GENERICA,
                body={
                    'name': f"Plantilla Juicios Evaluativos {nom_programa} RAPs{n_RAs}",
                    'parents': [carpeta_plantillas_id]
                }
            ).execute()
            id_nueva_especifica = copia_especifica['id']
            time.sleep(0.5)

            sheet_meta_esp = sheets_service.spreadsheets().get(spreadsheetId=id_nueva_especifica).execute()
            sheet_id_esp = sheet_meta_esp['sheets'][0]['properties']['sheetId']

            sheets_service.spreadsheets().batchUpdate(spreadsheetId=id_nueva_especifica, body={
                'requests': [{
                    'updateSheetProperties': {
                        'properties': {'sheetId': sheet_id_esp, 'title': codigo_programa},
                        'fields': 'title'
                    }
                }]
            }).execute()

            requests_especifica = []
            if n_RAs > 1:
                requests_especifica.append(solicitud('insert_row_col', sheet_id_esp, rango='H1', cantidad=n_RAs - 1, valor=['COLUMNS', 'AFTER']))
                requests_especifica.append(solicitud('copy_format', sheet_id_esp, rango='H1:H8', valor=f'I1:{generate_column_names("H", n_RAs)[-1]}8'))

            requests_especifica.append({
                'updateSheetProperties': {
                    'properties': {
                        'sheetId': sheet_id_esp,
                        'gridProperties': {'rowCount': 8, 'columnCount': 7 + n_RAs}
                    },
                    'fields': 'gridProperties(rowCount,columnCount)'
                }
            })

            sheets_service.spreadsheets().batchUpdate(spreadsheetId=id_nueva_especifica, body={'requests': requests_especifica}).execute()
            ids_plantillas[codigo_programa][n_RAs_str] = id_nueva_especifica
            time.sleep(0.5)

        id_plantilla_usar = ids_plantillas[codigo_programa][n_RAs_str]
        copied_file = drive_service.files().copy(
          fileId=id_plantilla_usar,
          body={'name': f'{nom_programa}-{key}', 'parents': [folder_etapa_id]}
        ).execute()
        file_id = copied_file['id']
        time.sleep(0.5)

        sheet_metadata = sheets_service.spreadsheets().get(spreadsheetId=file_id).execute()
        sheet_id = sheet_metadata['sheets'][0]['properties']['sheetId']
        sheets_service.spreadsheets().batchUpdate(spreadsheetId=file_id, body={
          'requests': [{
            'updateSheetProperties': {
              'properties': {'sheetId': sheet_id, 'title': key},
              'fields': 'title'
            }
          }]
        }).execute()

    sheet_metadata = sheets_service.spreadsheets().get(spreadsheetId=file_id).execute()
    sheet = sheet_metadata.get('sheets', '')
    sheets_ids[key] = sheet[0]['properties']['sheetId']
    ids[key] = file_id
    rowsColumnsCounts[key] = sheet[0]['properties']['gridProperties']

    metadata_carpetas_salida[key] = {
        'operacion': operacion,
        'nombre_archivo': f'{nom_programa}-{key}',
        'ruta_carpeta': f"Juicios Evaluativos tabulados > {nombre_etapa_carpeta}",
        'link_carpeta': f"https://drive.google.com/drive/folders/{folder_etapa_id}",
        'folder_id': folder_etapa_id,
        'nombre_carpeta_final': nombre_etapa_carpeta
    }
    time.sleep(0.2)  # Pausa de cortesía para estabilizar tasa de peticiones

  globals()['sheets_ids'] = sheets_ids
  globals()['ids'] = ids
  globals()['rowsColumnsCounts'] = rowsColumnsCounts
  return sheets_ids, ids, rowsColumnsCounts

### 🛠️ Refactorización Estructural: Módulos y Clases de Soporte (Fase 2)

In [ ]:
# ======================================================================
# FASE 2: MODULARIZACIÓN Y LIMPIEZA DE CÓDIGO
# ======================================================================

class GoogleDriveManager:
    """Clase encargada de centralizar la interacción con Google Drive API.
       Administra la Carpeta Raíz, Entrada, Salidas y Plantillas de forma segura."""
    def __init__(self, drive_service):
        self.drive = drive_service
        self.raiz_id = None

    def asegurar_raiz(self):
        """Busca o vincula la carpeta raíz de instalación, controlando la desvinculación por sufijo."""
        query = "appProperties has { key='carpeta_raiz_instalacion_script_juicios' and value='true' } and mimeType='application/vnd.google-apps.folder' and trashed = false"
        search_result = self.drive.files().list(q=query, fields="files(id, name)").execute()
        folders = search_result.get('files', [])

        if folders:
            folder_id = folders[0]['id']
            folder_name = folders[0].get('name', '')
            if folder_name.endswith(' _xYz_'):
                try:
                    self.drive.files().update(
                        fileId=folder_id,
                        body={'appProperties': {'carpeta_raiz_instalacion_script_juicios': 'false'}}
                    ).execute()
                except Exception:
                    pass
            else:
                self.raiz_id = folder_id
                return self.raiz_id

        # Creación automática fallback por defecto
        folder_metadata = {
            'name': 'Ecosistema Juicios Evaluativos - Raíz',
            'mimeType': 'application/vnd.google-apps.folder',
            'parents': ['root'],
            'appProperties': {'carpeta_raiz_instalacion_script_juicios': 'true'}
        }
        new_folder = self.drive.files().create(body=folder_metadata, fields='id').execute()
        self.raiz_id = new_folder['id']
        return self.raiz_id

    def obtener_subcarpeta_por_propiedad(self, prop_key, folder_name):
        """Busca una subcarpeta por propiedad privada de la app; si no existe, la crea bajo la raíz."""
        if not self.raiz_id: self.asegurar_raiz()
        query = f"appProperties has {{ key='{prop_key}' and value='true' }} and mimeType='application/vnd.google-apps.folder' and trashed = false"
        search_result = self.drive.files().list(q=query, fields="files(id, name)").execute()
        folders = search_result.get('files', [])
        if folders:
            return folders[0]['id']

        # Crear subcarpeta vinculada
        meta = {
            'name': folder_name,
            'mimeType': 'application/vnd.google-apps.folder',
            'parents': [self.raiz_id],
            'appProperties': {prop_key: 'true'}
        }
        new_f = self.drive.files().create(body=meta, fields='id').execute()
        return new_f['id']

class SofiaEvaluator:
    """Clase encargada de agrupar el parsing, depuración y estructuración
       de los juicios evaluativos y estadísticas de Sofia Plus."""
    @staticmethod
    def extraer_ra_unicos(df_juicios):
        col_fecha = 'Fecha y Hora del Juicio Evaluativo'
        col_ra = 'Resultado de Aprendizaje'
        col_func = 'Funcionario que registro el juicio evaluativo'
        col_juicio = 'Juicio de Evaluación'

        es_aprobado = df_juicios[col_juicio].astype(str).str.strip().str.upper() == 'APROBADO'
        ap = df_juicios.loc[es_aprobado, [col_fecha, col_ra, col_func]].copy()
        ap[col_fecha] = pd.to_datetime(ap[col_fecha], errors='coerce').dt.normalize()
        ra = ap.sort_values([col_fecha, col_ra, col_func])[col_ra]
        bloques = ra.groupby((ra != ra.shift()).cumsum()).agg(RA='first', filas='size')
        idx = bloques.groupby('RA')['filas'].idxmax().sort_values()
        lista_ra = bloques.loc[idx, 'RA'].tolist()
        return list(dict.fromkeys(lista_ra + df_juicios.loc[~es_aprobado, col_ra].dropna().tolist()))

class GoogleSheetsTemplateEngine:
    """Motor de renderizado para gestionar plantillas físicas, inyección de estilos
       y construcción de transacciones batchUpdate hacia Google Sheets API."""
    def __init__(self, sheets_service):
        self.sheets = sheets_service

    def aplicar_limites_cuadrícula(self, sheet_id, spreadsheet_id, filas=8, columnas=8):
        request = {
            'updateSheetProperties': {
                'properties': {
                    'sheetId': sheet_id,
                    'gridProperties': {'rowCount': filas, 'columnCount': columnas}
                },
                'fields': 'gridProperties(rowCount,columnCount)'
            }
        }
        self.sheets.spreadsheets().batchUpdate(spreadsheetId=spreadsheet_id, body={'requests': [request]}).execute()

### 🖥️ Interfaz Gráfica de Procesamiento (Pre-Versión Interactiva)
Usa este panel interactivo para cargar tus archivos de Sofia Plus, monitorear el estado del procesamiento y ejecutar la tabulación de manera más cómoda.

### ⚙  Implementación optimizada de la fase 4.2: gestión de carpetas y hojas de salida
En esta celda se integra la versión final de `preparar_hojas_salida`, con caché en memoria de Drive, transiciones de etapa Lectiva/Productiva con preservación de enlaces, clonación y autogeneración dinámica de plantillas.

In [ ]:
import time
import datetime
from dateutil.relativedelta import relativedelta

# Variables de memoria caché global para optimizar llamadas a la API de Drive
_cache_carpetas_etapas = {}
_cache_rutas_carpetas = {}

def obtener_ruta_drive_con_cache(folder_id, drive_service):
    """Obtiene la ruta amigable de una carpeta utilizando caché en memoria para ahorrar cuota de API."""
    if folder_id in _cache_rutas_carpetas:
        return _cache_rutas_carpetas[folder_id]
    path = []
    try:
        current_folder = drive_service.files().get(fileId=folder_id, fields='name, parents').execute()
        parents = current_folder.get('parents')
        if parents:
            for parent_id in parents:
                parent_path = obtener_ruta_drive_con_cache(parent_id, drive_service)
                if parent_path:
                    path.extend(parent_path.split(' > '))
        path.append(current_folder['name'])
    except Exception:
        pass
    ruta_completa = " > ".join(path) if path else "Desconocido"
    _cache_rutas_carpetas[folder_id] = ruta_completa
    return ruta_completa

def preparar_hojas_salida(data_fichas, ids_plantillas):
    """
    Fase 4.2: Asegura y prepara las hojas físicas de salida.
    Optimización de alto rendimiento: Utiliza properties personalizadas de Google Drive para indexación directa
    y un sistema de validación híbrida en caché para evitar lecturas de metadatos redundantes.
    """
    global nom, metadata_carpetas_salida, ID_PLANTILLA_GENERICA

    sheets_ids = globals().get('sheets_ids', {})
    ids = globals().get('ids', {})
    rowsColumnsCounts = globals().get('rowsColumnsCounts', {})

    # 1. Asegurar Carpeta Raíz de Instalación
    global carpeta_raiz_instalacion_id
    if 'carpeta_raiz_instalacion_id' not in globals() or not carpeta_raiz_instalacion_id:
        carpeta_raiz_instalacion_id = obtener_o_vincular_raiz_instalacion()

    carpeta_plantillas_id = obtener_o_vincular_carpeta_plantillas()

    for key, info in data_fichas.items():
        codigo_programa = str(info['codigo'])
        nom_programa = extraer_iniciales(info['programa'])
        nom = nom_programa
        n_RAs = info['cantidad_RAs']
        n_RAs_str = str(n_RAs)

        # 3. Determinar etapa académica actual (Lectiva vs Productiva)
        nombre_etapa_carpeta = nombre_carpeta_etapa(nom_programa, info['fecha_fin'], info.get('fecha_reporte'))

        # Identidad lógica de la carpeta (propiedad) → compatibilidad por nombre → creación; renombrar o mover no la pierde
        folder_etapa_id = obtener_o_crear_subcarpeta_etapa(carpeta_raiz_instalacion_id, nombre_etapa_carpeta)

        # 4. y 5. El archivo de salida se localiza ENTRE las dos carpetas de etapa (Lectiva y Productiva), estén donde estén:
        #    INTENTO 1: propiedades de Drive; INTENTO 2: nombre + validación híbrida (pestaña / celda D3).
        nombre_contraria = nombre_etapa_contraria(nombre_etapa_carpeta)
        folder_contraria_id = buscar_subcarpeta_etapa(carpeta_raiz_instalacion_id, nombre_contraria)
        file_id, nombre_archivo_encontrado, via_archivo, padres_archivo, duplicados = localizar_archivo_salida(
            key, nom_programa, [(folder_etapa_id, nombre_etapa_carpeta), (folder_contraria_id, nombre_contraria)])
        operacion = "actualizado"

        if via_archivo == 'propiedad':
            print(f"📌 Vinculación Indexada (1 Petición): Se asoció '{nombre_archivo_encontrado}' mediante propiedades de Drive para la ficha '{key}'.")
        elif via_archivo == 'hibrido':
            print(f"📌 Vinculación Híbrida Optimizada con Caché: Se asoció '{nombre_archivo_encontrado}' para la ficha '{key}'.")

            # Marcar con propiedad para futuras búsquedas ultrarrápidas directas
            try:
                drive_service.files().update(
                    fileId=file_id,
                    body={'properties': {'ficha_asociada': key}}
                ).execute()
            except Exception:
                pass

        # Transición de etapa: si el archivo está en la carpeta que no le corresponde (por fecha, o porque lo movieron), se reubica
        if file_id and folder_etapa_id not in padres_archivo:
            quitar = [p for p in padres_archivo if p == folder_contraria_id]
            drive_service.files().update(
                fileId=file_id,
                addParents=folder_etapa_id,
                removeParents=",".join(quitar) if quitar else None,
                fields='id, parents'
            ).execute()
            for k_cache in [k for k in _cache_validacion_hibrida if k.endswith(f"_{nom_programa}_{key}")]:
                del _cache_validacion_hibrida[k_cache]   # el archivo ya no está donde decía la caché
            print(f"🔀 Transición de etapa: '{nombre_archivo_encontrado}' (ficha {key}) se movió a '{nombre_etapa_carpeta}'.")

        for dup in duplicados:
            print(f"⚠️ ARCHIVO DUPLICADO POR NOMBRE (ficha {key}): '{dup['nombre']}' en '{dup['carpeta']}' NO se está actualizando. "
                  f"Se actualiza el archivo vinculado '{nombre_archivo_encontrado}'.")

        # 6. Si no existe ningún archivo con esa ficha, lo creamos de cero usando la plantilla
        if not file_id:
            operacion = "creado"
            if codigo_programa in ids_plantillas and n_RAs_str in ids_plantillas[codigo_programa]:
                id_plantilla_usar = ids_plantillas[codigo_programa][n_RAs_str]
            else:
                # Generación de la plantilla genérica si no existe
                if not ID_PLANTILLA_GENERICA:
                    ID_PLANTILLA_GENERICA = obtener_id_plantilla_generica_dinamica()

                plantilla_existe = False
                if ID_PLANTILLA_GENERICA:
                    try:
                        drive_service.files().get(fileId=ID_PLANTILLA_GENERICA, fields='id').execute()
                        plantilla_existe = True
                    except Exception:
                        pass

                if not plantilla_existe:
                    print(f"⚙ Generando plantilla genérica física base 'G123'...")
                    nueva_meta = {
                        'name': 'Plantilla Generica Juicios Evaluativos',
                        'mimeType': 'application/vnd.google-apps.spreadsheet',
                        'parents': [carpeta_plantillas_id]
                    }
                    nueva_plantilla = drive_service.files().create(body=nueva_meta, fields='id').execute()
                    ID_PLANTILLA_GENERICA = nueva_plantilla['id']
                    generar_plantilla_juicios_evaluativos(ID_PLANTILLA_GENERICA, {
                        'PROGRAMA': info['programa'].upper(),
                        'TRIMESTRE': 'TRI',
                        'TEMATICAS': 'TEMÁTICAS'
                    })
                    time.sleep(1.0)

                print(f"⚒ Construyendo plantilla específica para '{nom_programa}' con {n_RAs} RAPs...")
                copia_especifica = drive_service.files().copy(
                    fileId=ID_PLANTILLA_GENERICA,
                    body={
                        'name': f"Plantilla Juicios Evaluativos {nom_programa} RAPs{n_RAs}",
                        'parents': [carpeta_plantillas_id]
                    }
                ).execute()
                id_nueva_especifica = copia_especifica['id']
                time.sleep(0.5)

                sheet_meta_esp = sheets_service.spreadsheets().get(spreadsheetId=id_nueva_especifica).execute()
                sheet_id_esp = sheet_meta_esp['sheets'][0]['properties']['sheetId']
                sheets_service.spreadsheets().batchUpdate(spreadsheetId=id_nueva_especifica, body={
                    'requests': [{
                        'updateSheetProperties': {
                            'properties': {'sheetId': sheet_id_esp, 'title': codigo_programa},
                            'fields': 'title'
                        }
                    }]
                }).execute()

                requests_especifica = []
                if n_RAs > 1:
                    requests_especifica.append(solicitud('insert_row_col', sheet_id_esp, rango='H1', cantidad=n_RAs - 1, valor=['COLUMNS', 'AFTER']))
                    requests_especifica.append(solicitud('copy_format', sheet_id_esp, rango='H1:H8', valor=f'I1:{generate_column_names("H", n_RAs)[-1]}8'))

                requests_especifica.append({
                    'updateSheetProperties': {
                        'properties': {
                            'sheetId': sheet_id_esp,
                            'gridProperties': {'rowCount': 8, 'columnCount': 7 + n_RAs}
                        },
                        'fields': 'gridProperties(rowCount,columnCount)'
                    }
                })

                sheets_service.spreadsheets().batchUpdate(spreadsheetId=id_nueva_especifica, body={'requests': requests_especifica}).execute()
                if codigo_programa not in ids_plantillas:
                    ids_plantillas[codigo_programa] = {}
                ids_plantillas[codigo_programa][n_RAs_str] = id_nueva_especifica
                time.sleep(0.5)

            id_plantilla_usar = ids_plantillas[codigo_programa][n_RAs_str]
            copied_file = drive_service.files().copy(
                fileId=id_plantilla_usar,
                body={
                    'name': f'{nom_programa}-{key}',
                    'parents': [folder_etapa_id],
                    'properties': {'ficha_asociada': key}
                }
            ).execute()
            file_id = copied_file['id']
            time.sleep(0.5)

            sheet_metadata = sheets_service.spreadsheets().get(spreadsheetId=file_id).execute()
            sheet_id = sheet_metadata['sheets'][0]['properties']['sheetId']
            sheets_service.spreadsheets().batchUpdate(spreadsheetId=file_id, body={
                'requests': [{
                    'updateSheetProperties': {
                        'properties': {'sheetId': sheet_id, 'title': key},
                        'fields': 'title'
                    }
                }]
            }).execute()

        sheet_metadata = sheets_service.spreadsheets().get(spreadsheetId=file_id).execute()
        sheet = sheet_metadata.get('sheets', '')
        sheets_ids[key] = sheet[0]['properties']['sheetId']
        ids[key] = file_id
        rowsColumnsCounts[key] = sheet[0]['properties']['gridProperties']

        metadata_carpetas_salida[key] = {
            'operacion': operacion,
            'nombre_archivo': nombre_archivo_encontrado,
            'ruta_carpeta': f"Juicios Evaluativos tabulados > {nombre_etapa_carpeta}",
            'link_carpeta': f"https://drive.google.com/drive/folders/{folder_etapa_id}",
            'folder_id': folder_etapa_id,
            'nombre_carpeta_final': nombre_etapa_carpeta,
            'nombre_pestaña_real': key,
            'duplicados': duplicados
        }
        time.sleep(0.2)

    globals()['sheets_ids'] = sheets_ids
    globals()['ids'] = ids
    globals()['rowsColumnsCounts'] = rowsColumnsCounts
    return sheets_ids, ids, rowsColumnsCounts

In [ ]:
# Memoria caché para evitar consultas redundantes de metadatos en la validación híbrida
_cache_validacion_hibrida = {}

def vincular_archivo_salida_hibrido(ficha_id, nom_programa, folder_etapa_id):
    """
    Busca y vincula un archivo de salida existente en Drive usando una estrategia híbrida optimizada:
    1. Método Primario (Rápido): Busca archivos con el nombre 'PROG-FICHA' y valida si su pestaña se llama igual que la ficha usando la caché.
    2. Método de Respaldo (Celda D3): Si falla, lee la celda D3 del archivo encontrado para validar la ficha.
    Retorna el ID del archivo si lo encuentra y valida, de lo contrario retorna None.
    """
    cache_key = f"{folder_etapa_id}_{nom_programa}_{ficha_id}"
    if cache_key in _cache_validacion_hibrida:
        return _cache_validacion_hibrida[cache_key]

    query = f"(name = '{nom_programa}-{ficha_id}' or name contains '{nom_programa}-{ficha_id} ') and mimeType='application/vnd.google-apps.spreadsheet' and '{folder_etapa_id}' in parents and trashed = false"
    search_result = drive_service.files().list(q=query, fields="files(id, name)").execute()
    items = search_result.get('files', [])

    if not items:
        _cache_validacion_hibrida[cache_key] = None
        return None

    file_id = items[0]['id']
    try:
        # --- Método Primario: Comprobar títulos de pestañas en metadatos ligeros ---
        sheet_metadata = sheets_service.spreadsheets().get(spreadsheetId=file_id, fields='sheets.properties(sheetId,title)').execute()
        sheets = sheet_metadata.get('sheets', [])

        if sheets and sheets[0]['properties']['title'] == ficha_id:
            _cache_validacion_hibrida[cache_key] = file_id
            return file_id

        # --- Método de Respaldo: Leer Celda D3 si la pestaña fue renombrada ---
        print(f"⚠️ Pestaña de la ficha {ficha_id} no coincide. Ejecutando respaldo (Lectura de celda D3) en {nom_programa}-{ficha_id}...")
        rango_evaluar = f"'{sheets[0]['properties']['title']}'!D3"
        resultado_celda = sheets_service.spreadsheets().values().get(
            spreadsheetId=file_id,
            range=rango_evaluar
        ).execute()

        valor_d3 = str(resultado_celda.get('values', [['']])[0][0]).strip()
        if ficha_id in valor_d3:
            print(f"✅ ¡Respaldo Exitoso! Celda D3 confirmó la identidad de la ficha {ficha_id}.")
            _cache_validacion_hibrida[cache_key] = file_id
            return file_id
    except Exception as e:
        print(f"❌ No se pudo validar de forma híbrida el archivo '{nom_programa}-{ficha_id}': {str(e)}")

    _cache_validacion_hibrida[cache_key] = None
    return None


def base_nombre_salida(nombre):
    """Parte estable del nombre de un archivo de salida: 'MEII-2996916 [27-10-2026 21:22:37]' → 'MEII-2996916'."""
    return (nombre or '').split('[')[0].strip()


def localizar_archivo_salida(ficha_id, nom_programa, carpetas):
    """Busca el archivo de salida de la ficha ENTRE las carpetas de etapa dadas [(id, nombre_lógico)], la primera es la que
    le corresponde hoy. Primero por propiedad de Drive (ficha_asociada) y, si no hay, por nombre + validación híbrida.
    Retorna (file_id|None, nombre, vía|None, [IDs de sus carpetas], duplicados). `duplicados` son los demás archivos de la
    misma ficha con el mismo nombre base (o con la misma propiedad): no se actualizan, solo se informan."""
    carpetas = [(i, n) for i, n in carpetas if i]
    ids = [i for i, _ in carpetas]
    nombres = dict(carpetas)
    en_carpetas = ' or '.join(f"'{i}' in parents" for i in ids)
    base = f"{nom_programa}-{ficha_id}"
    hoja = "mimeType='application/vnd.google-apps.spreadsheet' and trashed = false"

    q_prop = f"({en_carpetas}) and properties has {{ key='ficha_asociada' and value='{ficha_id}' }} and {hoja}"
    con_prop = drive_service.files().list(q=q_prop, fields="files(id, name, parents, createdTime)").execute().get('files', [])
    # si hubiera varios con la propiedad: el que ya está en la carpeta que corresponde y, luego, el más antiguo
    con_prop.sort(key=lambda f: (0 if ids[0] in (f.get('parents') or []) else 1, f.get('createdTime', ''), f['id']))

    file_id, nombre, via, padres = None, base, None, []
    if con_prop:
        file_id, nombre, via, padres = con_prop[0]['id'], con_prop[0]['name'], 'propiedad', list(con_prop[0].get('parents') or [])
    else:
        for carpeta_id in ids:
            hallado = vincular_archivo_salida_hibrido(ficha_id, nom_programa, carpeta_id)
            if hallado:
                file_id, via, padres = hallado, 'hibrido', [carpeta_id]
                break

    q_nom = f"({en_carpetas}) and (name = '{base}' or name contains '{base} ') and {hoja}"
    por_nombre = drive_service.files().list(q=q_nom, fields="files(id, name, parents)").execute().get('files', [])
    duplicados, vistos = [], set()
    for f in por_nombre + con_prop:
        if f['id'] == file_id or f['id'] in vistos:
            continue
        if f in por_nombre and base_nombre_salida(f.get('name')) != base:
            continue   # 'contains' es laxo: solo cuenta la parte estable del nombre
        vistos.add(f['id'])
        carpeta = next((nombres[p] for p in (f.get('parents') or []) if p in nombres), '')
        duplicados.append({'id': f['id'], 'nombre': f.get('name', ''), 'carpeta': carpeta,
                           'link': f"https://docs.google.com/spreadsheets/d/{f['id']}"})
    return file_id, nombre, via, padres, duplicados

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import time
import io
import os
import html
import asyncio
import threading
import pandas as pd
import datetime
from google.colab import auth, drive

# ==========================================
# CONFIGURACIÓN DE ADVERTENCIAS Y LOGS
# ==========================================
import warnings
# Silenciar advertencia específica de pandas para inferencia de formato de fechas
warnings.filterwarnings("ignore", category=UserWarning, module="pandas")

# ==========================================
# ESTILOS CSS PERSONALIZADOS PARA APLICACIÓN WEB
# ==========================================
css_estilos = widgets.HTML("""
<style>
    .app-container { font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; color: #1e293b; }
    .app-card { background: #ffffff; border-radius: 12px; box-shadow: 0 4px 6px -1px rgba(0,0,0,0.1), 0 2px 4px -1px rgba(0,0,0,0.06); border: 1px solid #f1f5f9; padding: 20px; margin-bottom: 20px; }
    .app-header { background: linear-gradient(135deg, #1e3a8a 0%, #3b82f6 100%); color: white; padding: 20px; border-radius: 12px 12px 0 0; margin-bottom: 20px; }
    .app-header h2 { margin: 0; font-size: 22px; font-weight: 700; letter-spacing: -0.5px; }
    .app-header p { margin: 6px 0 0 0; font-size: 13px; opacity: 0.9; }
    .banner-error { background-color: #fef2f2; border-left: 5px solid #ef4444; color: #991b1b; padding: 12px; border-radius: 6px; margin: 10px 0px; font-size: 13px; }
    .banner-info { background-color: #eff6ff; border-left: 5px solid #3b82f6; color: #1e40af; padding: 12px; border-radius: 6px; margin: 10px 0px; font-size: 13px; }

    /* Estilos para textos de pestañas (Modos): Color y tamaño más grande */
    .p-TabBar-tabLabel {
        overflow: visible !important;
        white-space: nowrap !important;
        font-size: 15px !important;
        font-weight: bold !important;
        color: #1e3a8a !important;
    }
    .lm-TabBar-tab.p-mod-current .p-TabBar-tabLabel {
        color: #2563eb !important;
        font-size: 16px !important;
    }
    .lm-TabBar-tab {
        min-width: 250px !important;
    }

    /* Contenedor Scrollable Compacto para evitar saturación de pantalla */
    .scroll-progress-container {
        max-height: 180px;
        overflow-y: auto;
        border: 1px solid #e2e8f0;
        border-radius: 8px;
        padding: 10px;
        background: #f8fafc;
        margin-top: 10px;
    }

    /* Tarjeta Compacta Ultra-Slim para las Fichas */
    .ficha-progress-box {
        background: #ffffff;
        border: 1px solid #e2e8f0;
        border-radius: 6px;
        padding: 6px 12px; margin-bottom: 6px;
        display: flex;
        align-items: center;
        justify-content: space-between;
    }
    .ficha-progress-box.processing { border-left: 4px solid #3b82f6; background-color: #eff6ff; }
    .ficha-progress-box.success { border-left: 4px solid #22c55e; background-color: #f0fdf4; }
</style>
""")

# --- 1. SELECTOR DE MODOS UNIFICADO (Pestañas nativas de ipywidgets) ---
tab_modos = widgets.Tab(layout={'margin_bottom': '15px', 'width': '100%', 'min_width': '600px'})

# Variables de Estado de Carga Multi-Archivo
archivo_seleccionado = {
    'origen': 'manual',
    'archivos': []
}

# Contenedor dinámico para poder recrear el FileUpload y reiniciar su contador físico (el número entre paréntesis)
box_uploader_container = widgets.HBox([], layout={'margin_bottom': '5px', 'height': '44px', 'align_items': 'center'})

# Declaración de funciones del uploader para evitar NameError
def _a_bytes(contenido):
    """Copia el binario del widget. bytes() sobre memoryview copia; bytes ya es inmutable."""
    if contenido is None or isinstance(contenido, str):
        return None
    try:
        return bytes(contenido)
    except Exception:
        return None

def _lote_si_cuadran(entradas):
    """entradas: [(nombre, contenido, tam_declarado|None)]. None si el lote está incompleto o mal pareado."""
    archivos = []
    for nombre, contenido, tam in entradas:
        nombre = (nombre or '').strip()
        crudo = _a_bytes(contenido)
        if not nombre or crudo is None:
            return None
        if isinstance(tam, int) and tam >= 0 and len(crudo) != tam:
            return None  # metadata nueva con el buffer viejo, o el último archivo aún no llega
        archivos.append({'nombre': nombre, 'contenido': crudo})
    return archivos

_NO_APLICA = object()

def _desde_metadata_data(uploader):
    """ipywidgets 7 (Colab): la fuente real es metadata+data, no value.

    value se arma cuando cambia _counter y a veces ocurre ANTES de que llegue
    el último buffer. zip() se queda corto y ese archivo desaparece del conteo.
    """
    if not hasattr(uploader, 'metadata') and not hasattr(uploader, 'data'):
        return _NO_APLICA
    meta = list(getattr(uploader, 'metadata', None) or [])
    data = list(getattr(uploader, 'data', None) or [])
    if not meta and not data:
        return _NO_APLICA
    if len(meta) != len(data):
        return None
    entradas = []
    for m, d in zip(meta, data):
        if not isinstance(m, dict):
            return None
        entradas.append((m.get('name'), d, m.get('size')))
    return _lote_si_cuadran(entradas)

def _desde_value(uploader):
    """ipywidgets 8 (value es tupla) y respaldo del dict de ipywidgets 7."""
    valor = getattr(uploader, 'value', None)
    if isinstance(valor, dict):
        if not valor:
            return []
        entradas = []
        for nombre, datos in valor.items():
            if isinstance(datos, dict):
                meta = datos.get('metadata') if isinstance(datos.get('metadata'), dict) else {}
                entradas.append((nombre, datos.get('content'), meta.get('size', datos.get('size'))))
            else:
                entradas.append((nombre, datos, None))
        return _lote_si_cuadran(entradas)
    if isinstance(valor, (list, tuple)):
        if not valor:
            return []
        entradas = []
        for item in valor:
            if not isinstance(item, dict):
                return None
            meta = item.get('metadata') if isinstance(item.get('metadata'), dict) else {}
            entradas.append((item.get('name'), item.get('content'), item.get('size', meta.get('size'))))
        return _lote_si_cuadran(entradas)
    return []

def archivos_del_lote(uploader):
    """Lote ACTUAL del botón. None = todavía incompleto (no usar, se reintenta).

    No devuelve la memoria acumulada: cada clic del botón reemplaza la lista
    interna del widget. La suma entre clics la hace fusionar_archivos.
    """
    primario = _desde_metadata_data(uploader)
    if primario is None:
        return None
    if primario is not _NO_APLICA:
        return primario
    return _desde_value(uploader)

def fusionar_archivos(memoria, lote):
    """Suma el lote a lo ya elegido, por nombre. Nunca quita un archivo previo.
    Si el mismo nombre vuelve a llegar, se queda el contenido más reciente."""
    indice = {}
    for a in memoria or []:
        indice[a['nombre']] = {'nombre': a['nombre'], 'contenido': a['contenido']}
    for a in lote or []:
        indice[a['nombre']] = {'nombre': a['nombre'], 'contenido': a['contenido']}
    return list(indice.values())

# Generación de lectura: un clic nuevo anula los reintentos del clic anterior.
_sync_gen = {'n': 0}
# Evita que igualar el (N) del botón dispare otra lectura y apague el botón un instante.
_ajustando_contador = {'activo': False}

def _agendar(fn, delay):
    """Corre fn en el hilo del kernel tras `delay` segundos, para que los buffers terminen de llegar."""
    loop = None
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        try:
            candidato = asyncio.get_event_loop_policy().get_event_loop()
            if candidato.is_running():
                loop = candidato
        except Exception:
            loop = None
    if loop is not None:
        try:
            loop.call_later(delay, fn)
            return
        except Exception:
            pass
    def _disparar():
        if loop is not None:
            try:
                loop.call_soon_threadsafe(fn)
                return
            except Exception:
                pass
        try:
            fn()
        except Exception as e:
            print(f"⚠️ No se pudo leer el selector de archivos: {e}")
    t = threading.Timer(delay, _disparar)
    t.daemon = True
    t.start()

def _refrescar_etiqueta_archivos():
    archivos = archivo_seleccionado.get('archivos', [])
    n = len(archivos)
    if n == 0:
        lbl_archivo_estado.value = "<i>Ningún archivo seleccionado aún</i>"
        btn_procesar.disabled = True
        return
    items = "".join(
        f"<li style='margin:0 0 2px 0;'>{html.escape(a['nombre'])}</li>" for a in archivos
    )
    lbl_archivo_estado.value = (
        f"Se han cargado <b>{n}</b> archivo(s). <b>Estos se procesan</b> al iniciar:"
        f"<ul style='margin:4px 0 0 18px; padding:0; font-size:12px; max-height:140px; overflow:auto;'>{items}</ul>"
    )
    btn_procesar.disabled = False

def _etiqueta_leyendo():
    n = len(archivo_seleccionado.get('archivos', []))
    if n == 0:
        return "<i>Leyendo archivos seleccionados…</i>"
    return f"<i>Sumando archivos… <b>{n}</b> ya confirmados. Espera a que la lista se complete.</i>"

def _aviso_lote_incompleto(uploader):
    meta = list(getattr(uploader, 'metadata', None) or [])
    en_mem = {a['nombre'] for a in archivo_seleccionado.get('archivos', [])}
    nombres = [m.get('name') for m in meta if isinstance(m, dict) and m.get('name')]
    faltan = [n for n in nombres if n not in en_mem] or nombres
    lista = "".join(f"<li>{html.escape(n)}</li>" for n in faltan)
    extra = f"<ul style='margin:4px 0 0 18px;'>{lista}</ul>" if lista else ""
    lbl_archivo_estado.value = (
        "<div class='banner-error'><b>No se pudo leer completo el último grupo de archivos.</b> "
        "Vuelve a seleccionarlos: se suman por nombre y no se duplican."
        f"{extra}</div>"
    )
    btn_procesar.disabled = not bool(en_mem)

def sincronizar_memoria_desde_uploader():
    """Lectura inmediata (sin espera). Suma el lote si ya está completo. No borra lo acumulado."""
    hijos = box_uploader_container.children
    if not hijos:
        return len(archivo_seleccionado.get('archivos', []))
    lote = archivos_del_lote(hijos[0])
    if lote:
        archivo_seleccionado['archivos'] = fusionar_archivos(archivo_seleccionado.get('archivos', []), lote)
        archivo_seleccionado['origen'] = 'manual'
    return len(archivo_seleccionado.get('archivos', []))

def _intentar_sync(gen, reintentos, avisar=True):
    """Devuelve True si el lote ya quedó leído (o venía vacío). False si hay que reintentar."""
    if gen != _sync_gen['n']:
        return False
    hijos = box_uploader_container.children
    if not hijos:
        return False
    uploader = hijos[0]
    try:
        lote = archivos_del_lote(uploader)
    except Exception as e:
        if reintentos > 0:
            _agendar(lambda g=gen, r=reintentos - 1: _intentar_sync(g, r), 0.15)
            return False
        print(f"⚠️ No se pudo leer el selector de archivos: {e}")
        if avisar:
            _aviso_lote_incompleto(uploader)
        return False
    if lote is None:
        if reintentos > 0:
            _agendar(lambda g=gen, r=reintentos - 1: _intentar_sync(g, r), 0.15)
            return False
        if avisar:
            _aviso_lote_incompleto(uploader)
        return False
    if not lote:
        return True
    archivo_seleccionado['archivos'] = fusionar_archivos(archivo_seleccionado.get('archivos', []), lote)
    archivo_seleccionado['origen'] = 'manual'
    actualizar_fecha_corte_por_defecto()
    # El (N) del botón es acumulativo y no se reinicia; lo igualamos a la lista real.
    n = len(archivo_seleccionado['archivos'])
    if getattr(uploader, '_counter', None) != n:
        _ajustando_contador['activo'] = True
        try:
            uploader._counter = n
        except Exception:
            pass
        finally:
            _ajustando_contador['activo'] = False
    _refrescar_etiqueta_archivos()
    return True

def _cambio_vacio(change):
    """Ignora el eco vacío al recrear el botón después de procesar."""
    nuevo = change.get('new')
    nombre = change.get('name')
    if nombre == '_counter':
        return nuevo in (0, None)
    if nombre in ('value', 'data', 'metadata') and nuevo in (None, (), [], {}):
        return True
    return False

def al_cambiar_archivo(change):
    if _ajustando_contador['activo'] or _cambio_vacio(change):
        return
    _sync_gen['n'] += 1
    gen = _sync_gen['n']
    # No procesar mientras el último archivo del grupo puede no haber llegado todavía.
    btn_procesar.disabled = True
    lbl_archivo_estado.value = _etiqueta_leyendo()
    # Primero en el acto (data y metadata a veces ya vienen juntos). Si falta el
    # último buffer, se reintenta solo: no se muestra error todavía.
    if _intentar_sync(gen, 0, avisar=False):
        return
    _agendar(lambda g=gen: _intentar_sync(g, 12), 0.2)

def crear_nuevo_uploader():
    """Recrea el widget FileUpload desde cero para forzar el reinicio de su contador interno."""
    nuevo_uploader = widgets.FileUpload(
        accept='.xlsx, .xls',
        multiple=True,
        description="Seleccionar Archivos Locales",
        layout={'width': '250px', 'height': '38px'},
        button_style='primary'
    )
    # value llega incompleto; data y _counter avisan cuando el lote termina de bajar.
    nuevo_uploader.observe(al_cambiar_archivo, names=['value', 'data', '_counter'])
    box_uploader_container.children = [nuevo_uploader]
    return nuevo_uploader

# Instanciar el uploader inicial ahora que al_cambiar_archivo ya está declarado
file_uploader = crear_nuevo_uploader()

# Obtener enlace dinámico de la carpeta de entrada de Drive de forma pasiva
try:
    carpeta_origen_id_temp = obtener_o_vincular_carpeta_procesar()
    link_procesar_drive = f"https://drive.google.com/drive/folders/{carpeta_origen_id_temp}"
    html_carpeta_link = f"<a href='{link_procesar_drive}' target='_blank' style='font-weight: bold; color: #1d4ed8; text-decoration: underline;'>Juicios a Procesar</a>"
except Exception:
    html_carpeta_link = "<b>Juicios a Procesar</b>"

# Pestaña 1: Cargar Reporte Local
lbl_archivo_estado = widgets.HTML("<i>Ningún archivo seleccionado aún</i>", layout={'margin': '5px 0px 5px 5px'})
box_manual_instrucciones = widgets.HTML("""
<div class='banner-info'>
    <b>📥 Carga Directa Local</b><br/>
    Sube uno o múltiples archivos de Sofia Plus (.xls o .xlsx) de tu computadora. Mantén presionado Ctrl o Shift al seleccionarlos.
</div>
""")
vbox_manual = widgets.VBox([
    box_manual_instrucciones,
    box_uploader_container,
    lbl_archivo_estado
])

# Pestaña 2: Procesar desde Google Drive
btn_sincronizar_drive = widgets.Button(description="↔  Sincronizar Carpeta", button_style='info', layout={'width': '200px', 'height': '38px'})
lbl_estado_lote_drive = widgets.HTML("<i>Sincroniza para auditar archivos en la nube...</i>", layout={'margin': '10px 0px'})
def _html_instrucciones_drive():
    return f"""
<div class='banner-info'>
    <b>☁  Procesamiento en Lote desde Drive</b><br/>
    Coloca los reportes de Sofia Plus dentro de la carpeta {html_carpeta_link} en Drive. El script los leerá directamente desde la nube conservando tus notas manuales.
</div>
"""
box_drive_instrucciones = widgets.HTML(_html_instrucciones_drive())
vbox_drive = widgets.VBox([
    box_drive_instrucciones,
    widgets.HBox([btn_sincronizar_drive], layout={'margin_bottom': '10px', 'height': '44px', 'align_items': 'center'}),
    lbl_estado_lote_drive
])

# Configurar títulos de las pestañas
tab_modos.children = [vbox_manual, vbox_drive]
tab_modos.set_title(0, "📥 Carga Manual (Local)")
tab_modos.set_title(1, "☁  Lote en la Nube (Drive)")

# --- 2. CONSOLA Y MONITOREO DETALLADO DE FICHAS ---
progress_bar = widgets.IntProgress(value=0, min=0, max=100, step=1, description='General:', bar_style='info', orientation='horizontal', layout={'width': '95%', 'margin': '15px 0px'})
btn_procesar = widgets.Button(description=" Iniciar Tabulación", button_style='success', icon='play', disabled=True, layout={'width': '250px', 'height': '42px'})
# Corte: procesar el reporte solo hasta una fecha y hora (junto al botón de inicio)
chk_corte = widgets.Checkbox(value=False, description='Procesar solo hasta:', indent=False, layout={'width': '150px'})
dp_corte = widgets.DatePicker(value=datetime.datetime.now(bogota_tz).date(), disabled=True, layout={'width': '140px'})
txt_hora_corte = widgets.Text(value='23:59', placeholder='HH:MM', disabled=True, layout={'width': '62px'})
_corte_estado = {'programatico': False, 'manual': False}

def _fecha_reporte_drive(f):
    """Fecha del reporte (celda C2) de un archivo de la carpeta de Drive; None si no se puede leer."""
    if f.get('mimeType') == 'application/vnd.google-apps.spreadsheet':
        r = sheets_service.spreadsheets().values().get(spreadsheetId=f['id'], range='C2', valueRenderOption='UNFORMATTED_VALUE').execute()
        v = (r.get('values') or [[None]])[0][0]
        return serial_a_datetime(v).date() if es_numero(v) else parsear_fecha(v)
    cont = drive_service.files().get_media(fileId=f['id']).execute()
    return parsear_fecha(encabezado_desde_excel(pd.read_excel(io.BytesIO(cont), nrows=FILA_TABLA_SOFIA, header=None))['fecha_reporte'])

def fecha_reporte_mas_reciente():
    """La fecha de reporte (C2) más reciente entre los archivos cargados (carga local) o de la carpeta de Drive, según la pestaña activa."""
    fechas = []
    try:
        if archivo_seleccionado.get('origen') == 'drive':
            for f in listar_gsheets(obtener_o_vincular_carpeta_procesar()):
                if not f['name'].endswith('Rdo'):
                    try:
                        fechas.append(_fecha_reporte_drive(f))
                    except Exception:
                        continue
        else:
            fechas = [parsear_fecha(enc[3]) for enc in _encabezados_de_archivos(archivo_seleccionado.get('archivos') or [])]
    except Exception:
        return None
    fechas = [x for x in fechas if x]
    return max(fechas) if fechas else None

def actualizar_fecha_corte_por_defecto(forzar=False):
    """Con el corte activo, el selector propone la fecha de reporte más reciente de los archivos: guía hasta dónde llegan los datos.
    No pisa una fecha que el usuario ya eligió, salvo al marcar la casilla (forzar)."""
    if not chk_corte.value or (_corte_estado['manual'] and not forzar):
        return
    f = fecha_reporte_mas_reciente()
    if f:
        _corte_estado['programatico'] = True
        dp_corte.value = f
        _corte_estado['programatico'] = False

def _al_cambiar_fecha_corte(change):
    if not _corte_estado['programatico']:
        _corte_estado['manual'] = True

def _al_marcar_corte(change):
    dp_corte.disabled = txt_hora_corte.disabled = not change['new']
    if change['new']:
        _corte_estado['manual'] = False
        actualizar_fecha_corte_por_defecto(forzar=True)
chk_corte.observe(_al_marcar_corte, names='value')
dp_corte.observe(_al_cambiar_fecha_corte, names='value')
box_corte = widgets.HBox([chk_corte, dp_corte, txt_hora_corte], layout={'align_items': 'center', 'margin': '0px 0px 0px 14px'})
output_consola = widgets.Output(layout={'border': '1px solid #e2e8f0', 'background_color': '#f8fafc', 'padding': '12px', 'height': '150px', 'overflow_y': 'auto', 'margin_top': '10px'})
# Contador en vivo y detalle técnico (log completo) en un acordeón cerrado por defecto
lbl_progreso_ficha = widgets.HTML("", layout={'margin': '4px 0px'})
output_detalle = widgets.Output(layout={'height': '260px', 'overflow_y': 'auto'})
acordeon_detalle = widgets.Accordion(children=[output_detalle], selected_index=None)
acordeon_detalle.set_title(0, 'Detalle técnico (log completo)')

# Contenedor con Scroll de progreso para albergar las fichas de forma compacta
container_progreso_fichas = widgets.VBox([], layout={'margin_top': '10px'}).add_class('scroll-progress-container')

# --- Controladores de Eventos de la Interfaz ---

def al_cambiar_pestana(change):
    index = change['new']
    if index == 0:
        archivo_seleccionado['origen'] = 'manual'
        tiene_archivos_memoria = bool(archivo_seleccionado.get('archivos'))
        btn_procesar.disabled = not tiene_archivos_memoria
    else:
        archivo_seleccionado['origen'] = 'drive'
        btn_procesar.disabled = True
        verificar_archivos_lote_drive()
    actualizar_fecha_corte_por_defecto()

tab_modos.observe(al_cambiar_pestana, names='selected_index')

def verificar_archivos_lote_drive(b=None):
    if 'drive_service' not in globals() or drive_service is None:
        lbl_estado_lote_drive.value = "<div class='banner-error'>⚠️  Google Drive no está conectado. Por favor ejecuta el entorno de Colab de nuevo.</div>"
        btn_procesar.disabled = True
        return

    lbl_estado_lote_drive.value = "<i>Sincronizando y auditando carpeta en tiempo real...</i>"
    btn_procesar.disabled = True

    try:
        carpeta_origen_id = obtener_o_vincular_carpeta_procesar()
        archivos = listar_gsheets(carpeta_origen_id)
        archivos_validos = [f for f in archivos if not f['name'].endswith('Rdo')]

        if archivos_validos:
            num_archivos = len(archivos_validos)
            lista_html = "".join([f"<li style='margin-bottom: 2px;'>{f['name']}</li>" for f in archivos_validos])

            lbl_estado_lote_drive.value = f"""
            <div style='background-color: #f0fdf4; border-left: 5px solid #22c55e; padding: 12px; border-radius: 6px; color: #166534; font-size: 13px;'>
                <b>✅ Escaneo de Drive exitoso.</b> Se encontraron <b>{num_archivos}</b> reporte(s) en la carpeta:
                <ul style='margin: 5px 0 0 15px; padding-left: 5px;'>{lista_html}</ul>
            </div>
            """
            btn_procesar.disabled = False
        else:
            lbl_estado_lote_drive.value = f"""
            <div class='banner-error'>
                ❌ No se encontraron reportes en la carpeta de entrada <a href='https://drive.google.com/drive/folders/{carpeta_origen_id}' target='_blank' style='font-weight: bold; color: #b91c1c; text-decoration: underline;'>Juicios a Procesar</a> de Drive.
            </div>
            """
            btn_procesar.disabled = True
    except Exception as e:
        lbl_estado_lote_drive.value = f"<div class='banner-error'>⚠️  Error al escanear Drive: {str(e)}</div>"
        btn_procesar.disabled = True

def _encabezados_de_archivos(archivos):
    """[(código, programa, fecha_fin, fecha_reporte)] leyendo solo el encabezado de cada Excel; los ilegibles se omiten (los reporta el flujo normal)."""
    salida = []
    for item in archivos:
        try:
            enc = encabezado_desde_excel(pd.read_excel(io.BytesIO(item['contenido']), nrows=FILA_TABLA_SOFIA, header=None))
            salida.append((enc['codigo'], enc['programa'], enc['fecha_fin'], enc['fecha_reporte']))
        except Exception:
            continue
    return salida

def liberar_uploader_completo():
    """Reinicia por completo la memoria de carga y el uploader de archivos locales (reseteando el indicador entre paréntesis)."""
    global file_uploader
    archivo_seleccionado['archivos'] = []
    file_uploader = crear_nuevo_uploader()

def iniciar_tabulacion_interactiva(b):
    global carpeta_raiz_instalacion_id, ordenRA, list_competencias, data_fichas, id_juicios, CORTE_ACTUAL

    # Corte opcional (fecha y hora): se valida antes de tocar nada para no perder los archivos cargados por un dato mal escrito
    try:
        corte_elegido = construir_corte(dp_corte.value, txt_hora_corte.value) if chk_corte.value else None
    except ValueError as e_corte:
        lbl_progreso_ficha.value = f"⚠️ {e_corte}"
        return
    CORTE_ACTUAL = corte_elegido

    if archivo_seleccionado['origen'] == 'manual':
        _sync_gen['n'] += 1   # cancela una lectura que todavía esté en curso
        sincronizar_memoria_desde_uploader()
    origen_proceso = archivo_seleccionado['origen']   # el origen puede cambiar si el usuario cambia de pestaña durante el proceso
    btn_procesar.disabled = True
    progress_bar.value = 5
    progress_bar.bar_style = 'info'
    container_progreso_fichas.children = []
    # Una sola línea de estado: se reemplaza, no se acumula. El detalle va al log.
    lbl_progreso_ficha.value = "Iniciando Pipeline de Tabulación Automatizado..."
    with output_consola:
        clear_output()
    with output_detalle:
        clear_output()
        print("Iniciando Pipeline de Tabulación Automatizado...")

    try:
        # 1. Asegurar carpeta raíz (si falta o se pidió cambiarla, se abre el panel y el proceso se detiene sin perder la selección)
        limpiar_caches_carpetas(solo_rutas=True)   # las rutas del reporte reflejan renombrados o movimientos recientes
        carpeta_raiz_instalacion_id = obtener_o_vincular_raiz_instalacion()
        progress_bar.value = 15

        # 2. Cargar formatos de órdenes de RAPs
        ordenRA = cargar_ordenes_ra()
        progress_bar.value = 25

        local_data_fichas = {}
        local_id_juicios = {}
        list_competencias = {}

        # 3. Parsing de Archivos según origen
        if origen_proceso == 'manual':
            # Copia fija: una lectura tardía del botón no puede meter ni quitar archivos a mitad del proceso
            archivos_a_procesar = list(archivo_seleccionado.get('archivos') or [])
            if not archivos_a_procesar:
                raise ValueError("No se detectó ningún archivo en memoria. Por favor, selecciona los archivos locales nuevamente.")

            # Antes de subir nada: si alguna carpeta que se usará es ambigua, se pide resolverla ahora
            verificar_carpetas_antes_de_procesar(_encabezados_de_archivos(archivos_a_procesar))

            for item in archivos_a_procesar:
                nombre = item['nombre']
                cont = item['contenido']

                try:
                    df_encabezado = pd.read_excel(io.BytesIO(cont), nrows=FILA_TABLA_SOFIA, header=None)
                    df = pd.read_excel(io.BytesIO(cont), header=FILA_TABLA_SOFIA, dtype=str)
                    encabezado = encabezado_desde_excel(df_encabezado)
                    ficha = encabezado['ficha']
                    codigo_programa = encabezado['codigo']
                    nom_programa = extraer_iniciales(encabezado['programa'])

                    # Dry-run estructural de validación de formato antes de subir
                    temp_fichas = {}
                    temp_competencias = {}
                    registrar_ficha(temp_fichas, ordenRA, temp_competencias, encabezado, df, fecha_con_guiones=True)
                    if corte_elegido is not None:
                        temp_fichas[ficha]['ficha'] = ficha
                        aviso_corte = aplicar_corte(temp_fichas[ficha], corte_elegido)
                        if aviso_corte:
                            with output_detalle:
                                print(f"⚠️ Ficha {ficha}: {aviso_corte}")

                    # 3.1 OBTENER O CREAR CARPETA PROCESADOS FINAL DIRECTAMENTE (Evita duplicados en Entrada)
                    folder_dest_id = obtener_o_vincular_carpeta_procesados(codigo_programa, nom_programa)

                    # 3.2 SUBIR DIRECTAMENTE A LA CARPETA PROCESADOS
                    gtime_temp = datetime.datetime.now(bogota_tz)
                    nuevo_nombre = f'{nom_programa}-Jui Eval-{ficha}-FA[{gtime_temp.day}-{gtime_temp.month}-{gtime_temp.year}] [{gtime_temp.strftime("%H:%M:%S")}]_Rdo'

                    file_metadata = {
                        'name': nuevo_nombre,
                        'parents': [folder_dest_id],
                        'appProperties': {
                            'codigo_programa': codigo_programa
                        }
                    }
                    from googleapiclient.http import MediaIoBaseUpload
                    media = MediaIoBaseUpload(io.BytesIO(cont), mimetype='application/vnd.openxmlformats-officedocument.spreadsheetml.sheet', resumable=True)
                    uploaded_file = drive_service.files().create(body=file_metadata, media_body=media, fields='id').execute()

                    # Registramos el ID del archivo cargado directamente en Procesados
                    local_id_juicios[ficha] = uploaded_file.get('id')
                    local_data_fichas[ficha] = temp_fichas[ficha]
                    list_competencias.update(temp_competencias)
                except (RaizPendiente, ConflictoCarpeta):
                    raise   # no es un problema del archivo: se detiene el proceso para resolver la carpeta
                except ValueError as e_val:
                    with output_detalle:
                        print(f"⚠️  ADVERTENCIA: Se omitió el archivo '{nombre}' debido a un problema de validación:\n{str(e_val)}\n")
                    continue
                except Exception as e_desc:
                    with output_detalle:
                        print(f"⚠️  ADVERTENCIA: No se pudo procesar el archivo '{nombre}' (error inesperado): {str(e_desc)}\n")
                    continue

            if not local_data_fichas:
                liberar_uploader_completo()
                raise ValueError("Ninguno de los archivos cargados corresponde a un reporte válido de Sofia Plus.")
        else:
            local_data_fichas, local_id_juicios = cargar_fichas_sofia(ordenRA, list_competencias)
            if not local_id_juicios:
                raise ValueError("No hay reportes en la carpeta de Drive de entrada.")
            if corte_elegido is not None:
                for f_corte, d_corte in local_data_fichas.items():
                    d_corte['ficha'] = f_corte
                    aviso_corte = aplicar_corte(d_corte, corte_elegido)
                    if aviso_corte:
                        with output_detalle:
                            print(f"⚠️ Ficha {f_corte}: {aviso_corte}")
            verificar_carpetas_antes_de_procesar([(d['codigo'], d['programa'], d['fecha_fin'], d.get('fecha_reporte')) for d in local_data_fichas.values()])

        progress_bar.value = 40
        globals()['id_juicios'] = local_id_juicios
        globals()['data_fichas'] = local_data_fichas

        # 4. Inicializar visualización de progreso
        widgets_fichas = {}
        lista_widgets = []
        for f_id, datos_f in local_data_fichas.items():
            nom_prog = extraer_iniciales(datos_f['programa'])
            etiqueta_barra = f"Juicios {nom_prog}- Ficha {f_id}"
            lbl_f = widgets.HTML(f"<b>{etiqueta_barra}</b>", layout={'width': '55%'})
            p_f = widgets.IntProgress(value=0, min=0, max=100, bar_style='warning', layout={'width': '40%', 'height': '10px'})
            box_f = widgets.HBox([lbl_f, p_f], layout={'align_items': 'center', 'margin_bottom': '4px'}).add_class('ficha-progress-box')
            widgets_fichas[f_id] = (p_f, box_f)
            lista_widgets.append(box_f)

        container_progreso_fichas.children = lista_widgets
        progress_bar.value = 50

        # 5. Generar hojas físicas de salida
        ids_plantillas = mapear_plantillas()
        with output_detalle:
            preparar_hojas_salida(local_data_fichas, ids_plantillas)
        progress_bar.value = 65

        # 6. Tabular secuencialmente
        gtime_actual = datetime.datetime.now(bogota_tz)
        globals()['reporte_procesamiento'] = {}

        num_fichas = len(local_data_fichas)
        for idx, (f_id, datos_f) in enumerate(local_data_fichas.items()):
            p_f, box_f = widgets_fichas[f_id]
            box_f.add_class('processing')
            p_f.value = 15
            p_f.bar_style = 'info'

            lbl_progreso_ficha.value = f"Tabulando ficha {f_id}… ({idx+1}/{num_fichas})"
            with output_detalle:
                body_requests = construir_solicitudes_ficha(f_id, datos_f, gtime_actual)
                p_f.value = 60
                actualizar_hoja(f_id, body_requests, gtime_actual, datos_f)
                extra = f" · {datos_f['historicos']} histórico(s) conservado(s)" if datos_f.get('historicos') else ''
                print(f"✅ Ficha {f_id} ({idx+1}/{num_fichas}){extra}")
            p_f.value = 100
            p_f.bar_style = 'success'
            box_f.remove_class('processing')
            box_f.add_class('success')

            progress_bar.value = int(65 + (35 * (idx + 1) / num_fichas))

        progress_bar.bar_style = 'success'
        CORTE_ACTUAL = None
        lbl_progreso_ficha.value = "✅ Pipeline de tabulación finalizado con éxito."
        with output_detalle:
            print("✅ Pipeline de tabulación finalizado con éxito.")
        with output_consola:
            clear_output()
            imprimir_reporte_consolidado()

        # Resetear completamente los campos de carga al terminar el proceso de forma exitosa
        if origen_proceso == 'manual':
            liberar_uploader_completo()   # solo se libera la carga local cuando lo procesado fue la carga local
            lbl_archivo_estado.value = "<i>✨ Proceso culminado de forma exitosa. Memoria liberada de forma segura para nuevas cargas.</i>"
            btn_procesar.disabled = True
        else:
            verificar_archivos_lote_drive()

    except (RaizPendiente, ConflictoCarpeta) as e_vinculo:
        # Falta elegir/vincular una carpeta: el panel ya está abierto. No se descarta la selección de archivos; se reintenta al terminar.
        CORTE_ACTUAL = None
        progress_bar.bar_style = 'warning'
        lbl_progreso_ficha.value = f"⚠️ {e_vinculo}"
        with output_detalle:
            print(f"⚠️ {e_vinculo}")
        btn_procesar.disabled = False

    except Exception as e:
        CORTE_ACTUAL = None
        progress_bar.bar_style = 'danger'
        lbl_progreso_ficha.value = f"❌ ERROR EN PIPELINE DE AUTOMATIZACIÓN: {str(e)}"
        with output_detalle:
            print(f"❌ ERROR EN PIPELINE DE AUTOMATIZACIÓN: {str(e)}")

        if origen_proceso == 'manual':
            liberar_uploader_completo()
            lbl_archivo_estado.value = f"<div class='banner-error'><b>Error de Ejecución:</b> {str(e)}</div>"

        # Reactivar el botón para permitir reintentos inmediatos ante un error de validación
        btn_procesar.disabled = False

# --- Panel de vinculación de carpetas (oculto salvo que haga falta) ---
btn_abrir_panel = widgets.Button(description='⚙ Carpetas', tooltip='Abrir el panel de carpeta raíz y carpetas internas',
                                 layout=widgets.Layout(width='120px', height='28px'))
btn_abrir_panel.on_click(lambda b: abrir_panel_vinculacion())
_estado_panel['desplegado'] = True   # el panel va dentro de la tarjeta principal

def _actualizar_enlace_procesar():
    global html_carpeta_link
    try:
        html_carpeta_link = f"<a href='https://drive.google.com/drive/folders/{obtener_o_vincular_carpeta_procesar()}' target='_blank' style='font-weight: bold; color: #1d4ed8; text-decoration: underline;'>Juicios a Procesar</a>"
    except Exception:
        html_carpeta_link = "<b>Juicios a Procesar</b>"
    box_drive_instrucciones.value = _html_instrucciones_drive()

def _aviso_vinculacion(evento, mensaje):
    if mensaje:
        lbl_progreso_ficha.value = mensaje
    if evento in ('interna_cambiada', 'conflicto_resuelto'):
        _actualizar_enlace_procesar()

registrar_hook_vinculacion(_aviso_vinculacion)

# --- Vinculación de Eventos ---
btn_sincronizar_drive.on_click(verificar_archivos_lote_drive)
btn_procesar.on_click(iniciar_tabulacion_interactiva)

# --- Estructuración de la Interfaz Visual (HTML/CSS) ---
ui_titulo = widgets.HTML("""
<div class='app-header'>
    <h2>⚙  Panel de Control - Juicios Evaluativos SENA</h2>
    <p>Pipeline web optimizado para tabulación, formateo y restauración de notas en tiempo real</p>
</div>
""")

card_proceso = widgets.VBox([
    widgets.HBox([btn_abrir_panel], layout={'justify_content': 'flex-end'}),
    contenedor_panel_vinculacion,
    widgets.HTML("<span style='font-size: 15px; font-weight: bold;'>Gestión de Reportes y Tabulación Automatizada</span><hr style='margin: 5px 0px 12px 0px; border: 0; border-top: 1px solid #e2e8f0;'/>"),
    tab_modos,
    container_progreso_fichas,
    widgets.HBox([btn_procesar, box_corte], layout={'justify_content': 'center', 'align_items': 'center', 'margin': '15px 0px 10px 0px'}),
    progress_bar,
    widgets.HTML("<b>Monitor de Procesamiento Consolidado:</b>"),
    lbl_progreso_ficha,
    output_consola,
    acordeon_detalle
], layout={'min_width': '620px'}).add_class('app-card')

# Renderizar panel
display(css_estilos, ui_titulo, card_proceso)

HTML(value="\n<style>\n    .app-container { font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; colo…

HTML(value="\n<div class='app-header'>\n    <h2>⚙\u2005 Panel de Control - Juicios Evaluativos SENA</h2>\n    …